In [ ]:
# Cell 1: Install & Import Libraries

# Step A: Install required packages (runs directly inside the notebook environment)
# %pip install sqlalchemy redshift-connector jinja2 python-dateutil --quiet
# %pip install python-dotenv --quiet
# %pip install sqlalchemy-redshift redshift-connector

# Step B: Load modules
import os
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import jinja2
from sqlalchemy import create_engine, text

print("All Redshift libraries installed and imported successfully!")

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
All Redshift libraries installed and imported successfully!


In [10]:
import os
import pandas as pd
import redshift_connector
from dotenv import load_dotenv

# Load credentials
load_dotenv("db.env")

# Establish connection
conn = redshift_connector.connect(
    host=os.getenv("REDSHIFT_HOST"),
    port=int(os.getenv("REDSHIFT_PORT", "5439")),
    database=os.getenv("REDSHIFT_DB"),
    user=os.getenv("REDSHIFT_USER"),
    password=os.getenv("REDSHIFT_PASSWORD")
)

# Test query
test_query = """
    SELECT *
    FROM credit_risk_playground.stg_mrt_neo_credit_risk_calculations
    LIMIT 10;
"""

# Fetch results into Pandas DataFrame
cursor = conn.cursor()
cursor.execute(test_query)
df = cursor.fetch_dataframe()

print("✅ Connected and retrieved sample data:")
display(df)

✅ Connected and retrieved sample data:


,pool_cut_off_date,borrower_identifier,loan_identifier,segment,primary_income,secondary_income,total_income,borrowers_employment_status,loan_origination_date,date_of_loan_maturity,...,adjusted_collateral_value,fitch_foreclosure_value,lgd_pit,lgd,expected_loss_1year,performance_trigger,province_code,pronvince_name,etl_updated,created_by
0,2025-10-31,5389046,2171143102,Non-NHG,28259.00,40746.00,69005.00,8,2023-12-27,2054-01-01,...,251205.681326,183380.147368,0.000000,5.000000,14.921250,2070.267947,0,ND,2025-11-05 14:01:37,alfred_teye
1,2025-10-31,5390731,2182846102,Non-NHG,78849.60,NaN,78849.60,3,2024-02-08,2054-03-01,...,75366.104614,55017.256369,0.000000,5.000000,1.300000,416.238843,0,ND,2025-11-05 14:01:37,alfred_teye
2,2025-10-31,5397038,2174436102,Non-NHG,64800.00,55252.37,120052.37,3,2024-03-14,2050-06-01,...,290177.421521,211829.517711,0.000000,5.000000,4.705093,2081.194217,0,ND,2025-11-05 14:01:37,alfred_teye
3,2025-10-31,5401768,2176471101,Non-NHG,0.00,55056.66,55056.66,4,2024-01-03,2054-02-01,...,547544.872898,399707.757215,0.000000,5.000000,9.500000,2081.194217,0,ND,2025-11-05 14:01:37,alfred_teye
4,2025-10-31,5418978,2183852101,NHG,45360.00,32756.36,78116.36,3,2024-02-15,2054-03-01,...,356638.422960,295793.540000,6.332147,6.332147,102.606000,7263.367817,0,ND,2025-11-05 14:01:37,alfred_teye
5,2025-10-31,5418994,2183859101,NHG,34099.00,NaN,34099.00,8,2024-02-01,2054-03-01,...,263642.856300,192459.285099,0.000000,5.000000,7.125000,1560.895663,0,ND,2025-11-05 14:01:37,alfred_teye
6,2025-10-31,5422281,2185246103,NHG,68726.35,NaN,68726.35,3,2024-02-06,2054-03-01,...,83127.813859,68508.950000,6.203462,6.203462,23.226000,1644.143431,0,ND,2025-11-05 14:01:37,alfred_teye
7,2025-10-31,5423489,2185773101,Non-NHG,99012.93,NaN,99012.93,3,2024-04-11,2054-05-01,...,670270.447858,489297.426936,0.000000,5.000000,54.641208,7970.973851,0,ND,2025-11-05 14:01:37,alfred_teye
8,2025-10-31,5429289,2188238102,Non-NHG,129600.00,0.00,129600.00,3,2024-02-06,2026-09-01,...,258815.441237,188935.272103,0.000000,0.000000,0.000000,673.350000,0,ND,2025-11-05 14:01:37,alfred_teye
9,2025-10-31,5430921,2188963102,Non-NHG,36686.00,NaN,36686.00,8,2024-03-28,2054-04-01,...,364449.400456,266048.062333,0.000000,5.000000,24.750000,3433.970458,0,ND,2025-11-05 14:01:37,alfred_teye


In [5]:
# Cell 3: Embedded SQL Pipeline Registry (Folder 1: MEV)

QUERIES = {
    "01_MEV_Macroeconomic_Scenarios_v3": """
        INSERT INTO credit_risk_playground.macroeconomic_scenarios_v3
        WITH report_date AS (
            SELECT '{{ end_date }}'::date AS scenario_date
        )
        SELECT 
            dbt_mev.* 
        FROM dbt.macro_economic_scenarios AS dbt_mev
        INNER JOIN report_date AS rep_dt
        USING(scenario_date)
        ORDER BY tnc_country, reference_date;
    """,

    "02_MEV_IFRS9_Input_Data": """
        INSERT INTO credit_risk_playground.stg_mrt_ifrs9_mev_input_data
        WITH report_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        raw_data AS (
            SELECT 
                a.*
            FROM credit_risk_playground.macroeconomic_scenarios_v3 AS a
            INNER JOIN report_date AS b
            ON a.scenario_date = b.reporting_date
            WHERE tnc_country = 'NLD' 
        ),
        temp_tbl AS (
            SELECT 
                *,
                CASE 
                    WHEN real_gdp_growth_rate_hist_q IS NOT NULL THEN real_gdp_growth_rate_hist_q
                    ELSE (SELECT real_gdp_growth_rate_hist_q FROM raw_data WHERE reference_date = (SELECT MAX(reference_date) FROM raw_data WHERE real_gdp_growth_rate_hist_q IS NOT NULL))
                END AS real_gdp_growth_rate_hist,
                
                CASE 
                    WHEN inflation_rate_hist_q IS NOT NULL THEN inflation_rate_hist_q
                    ELSE (SELECT inflation_rate_hist_q FROM raw_data WHERE reference_date = (SELECT MAX(reference_date) FROM raw_data WHERE inflation_rate_hist_q IS NOT NULL))
                END AS inflation_rate_hist,
                
                CASE 
                    WHEN unemployment_rate_hist_q IS NOT NULL THEN unemployment_rate_hist_q
                    ELSE (SELECT unemployment_rate_hist_q FROM raw_data WHERE reference_date = (SELECT MAX(reference_date) FROM raw_data WHERE unemployment_rate_hist_q IS NOT NULL))
                END AS unemployment_rate_hist,
                
                CASE 
                    WHEN nominal_rhpi_hist_q IS NOT NULL THEN nominal_rhpi_hist_q
                    ELSE (SELECT nominal_rhpi_hist_q FROM raw_data WHERE reference_date = (SELECT MAX(reference_date) FROM raw_data WHERE nominal_rhpi_hist_q IS NOT NULL))
                END AS nominal_rhpi_hist,
                
                CASE 
                    WHEN mortgage_interest_rate_hist_m IS NOT NULL THEN mortgage_interest_rate_hist_m
                    ELSE (SELECT mortgage_interest_rate_hist_m FROM raw_data WHERE reference_date = (SELECT MAX(reference_date) FROM raw_data WHERE mortgage_interest_rate_hist_m IS NOT NULL))
                END AS mortgage_interest_rate_hist,
                
                CASE 
                    WHEN real_gdp_growth_rate_fcst_q IS NOT NULL THEN real_gdp_growth_rate_fcst_q
                    ELSE (SELECT real_gdp_growth_rate_fcst_q FROM raw_data WHERE reference_date = (SELECT MAX(reference_date) FROM raw_data WHERE real_gdp_growth_rate_fcst_q IS NOT NULL))
                END AS real_gdp_growth_rate_fcst_q1,
                
                CASE 
                    WHEN real_gdp_growth_rate_fcst_q_scen_down IS NOT NULL THEN real_gdp_growth_rate_fcst_q_scen_down
                    ELSE (SELECT real_gdp_growth_rate_fcst_q_scen_down FROM raw_data WHERE reference_date = (SELECT MAX(reference_date) FROM raw_data WHERE real_gdp_growth_rate_fcst_q_scen_down IS NOT NULL))
                END AS real_gdp_growth_rate_fcst_q_scen_down1,
                
                CASE 
                    WHEN real_gdp_growth_rate_fcst_q_scen_neu IS NOT NULL THEN real_gdp_growth_rate_fcst_q_scen_neu
                    ELSE (SELECT real_gdp_growth_rate_fcst_q_scen_neu FROM raw_data WHERE reference_date = (SELECT MAX(reference_date) FROM raw_data WHERE real_gdp_growth_rate_fcst_q_scen_neu IS NOT NULL))
                END AS real_gdp_growth_rate_fcst_q_scen_neu1,
                
                CASE 
                    WHEN real_gdp_growth_rate_fcst_q_scen_up IS NOT NULL THEN real_gdp_growth_rate_fcst_q_scen_up
                    ELSE (SELECT real_gdp_growth_rate_fcst_q_scen_up FROM raw_data WHERE reference_date = (SELECT MAX(reference_date) FROM raw_data WHERE real_gdp_growth_rate_fcst_q_scen_up IS NOT NULL))
                END AS real_gdp_growth_rate_fcst_q_scen_up1
            FROM raw_data
            ORDER BY reference_date
        ),
        required_mevs AS (
            SELECT
                reference_date,
                unemployment_rate_hist - LAG(unemployment_rate_hist, 12) OVER(ORDER BY reference_date) AS ur_diff_12m,
                mortgage_interest_rate_hist - LAG(mortgage_interest_rate_hist, 12) OVER(ORDER BY reference_date) AS ir_diff_12m,
                ROUND(LN(LAG(nominal_rhpi_hist, 1) OVER(ORDER BY reference_date) / LAG(nominal_rhpi_hist, 13) OVER(ORDER BY reference_date)), 4) AS rhpi_log_diff_12m_lag_1,
                ROUND(LEAD(real_gdp_growth_rate_fcst_q1, 12) OVER(ORDER BY reference_date) - real_gdp_growth_rate_hist, 4) AS gdp_diff_12m, 
                
                unemployment_rate_hist - LAG(unemployment_rate_hist, 12) OVER(ORDER BY reference_date) AS ur_diff_12m_down_scen,
                unemployment_rate_hist - LAG(unemployment_rate_hist, 12) OVER(ORDER BY reference_date) AS ur_diff_12m_neu_scen,
                unemployment_rate_hist - LAG(unemployment_rate_hist, 12) OVER(ORDER BY reference_date) AS ur_diff_12m_up_scen,
                
                mortgage_interest_rate_hist - LAG(mortgage_interest_rate_hist, 12) OVER(ORDER BY reference_date) AS ir_diff_12m_down_scen,
                mortgage_interest_rate_hist - LAG(mortgage_interest_rate_hist, 12) OVER(ORDER BY reference_date) AS ir_diff_12m_neu_scen,
                mortgage_interest_rate_hist - LAG(mortgage_interest_rate_hist, 12) OVER(ORDER BY reference_date) AS ir_diff_12m_up_scen,
                
                ROUND(LEAD(real_gdp_growth_rate_fcst_q_scen_down1, 12) OVER(ORDER BY reference_date) - real_gdp_growth_rate_hist, 4) AS gdp_diff_12m_down_scen, 
                ROUND(LEAD(real_gdp_growth_rate_fcst_q_scen_neu1, 12) OVER(ORDER BY reference_date) - real_gdp_growth_rate_hist, 4) AS gdp_diff_12m_neu_scen, 
                ROUND(LEAD(real_gdp_growth_rate_fcst_q_scen_up1, 12) OVER(ORDER BY reference_date) - real_gdp_growth_rate_hist, 4) AS gdp_diff_12m_up_scen, 
                
                rhpi_log_diff_12m_lag_1 AS rhpi_log_diff_12m_lag_1_down_scen,
                rhpi_log_diff_12m_lag_1 AS rhpi_log_diff_12m_lag_1_neu_scen,
                rhpi_log_diff_12m_lag_1 AS rhpi_log_diff_12m_lag_1_up_scen
            FROM temp_tbl
        )
        SELECT 
            mevs.*,
            GETDATE() AS etl_updated,
            'alfred_teye' AS created_by
        FROM required_mevs AS mevs
        INNER JOIN report_date AS rdate
        ON mevs.reference_date::date = LAST_DAY(DATE_ADD('month', -1, rdate.reporting_date::date)::date);
    """,

    "03_MEV_Input_Data": """
        INSERT INTO credit_risk_playground.stg_mrt_mev_input_data
        WITH reporting_date AS (
           SELECT '{{ end_date }}'::date AS reporting_date
        )
        SELECT
            reference_date AS ref_date,
            rhpi_log_diff_12m_lag_1 AS log_diff_12m_rhpi_lag_1,
            ur_diff_12m AS diff_12m_ur,
            ir_diff_12m AS diff_12m_ir,
            gdp_diff_12m AS diff_12m_gdp,
            GETDATE() AS etl_updated,
            USER AS created_by
        FROM credit_risk_playground.stg_mrt_ifrs9_mev_input_data AS a
        INNER JOIN reporting_date AS b
        ON a.reference_date = LAST_DAY(DATE_ADD('month', -1, b.reporting_date)::date);
    """
}

print(f" Registered {len(QUERIES)} MEV queries in execution order:")
for step_num, name in enumerate(QUERIES.keys(), 1):
    print(f"  Step {step_num}: {name}")

 Registered 3 MEV queries in execution order:
  Step 1: 01_MEV_Macroeconomic_Scenarios_v3
  Step 2: 02_MEV_IFRS9_Input_Data
  Step 3: 03_MEV_Input_Data


In [6]:
# Cell 3: Embedded SQL Pipeline Registry (Folder 2: Rabobank)

QUERIES = {
    "04_Rabo_Raw": """
        INSERT INTO credit_risk_playground.stg_mrt_rabobank_raw
        WITH report_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        temp_tbl AS (
            SELECT 
                TO_DATE(TRIM(REPLACE(REPLACE(pool_cut_off_date, '.', '-'), '-', '/')), 'DD/MM/YYYY')::date AS pool_cut_off_date,   
                CASE WHEN pool_identifier IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE pool_identifier END,
                CASE WHEN loan_identifier IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE loan_identifier END,	
                CASE WHEN regulated_loan IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE regulated_loan END,
                CASE WHEN originator IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE originator END,	
                CASE WHEN servicer_identifier IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE servicer_identifier END,
                CASE WHEN borrower_identifier IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE borrower_identifier END,
                CASE WHEN property_identifier IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE property_identifier END,
                CASE WHEN borrower_type ~* '[A-Za-z]' THEN NULL ELSE borrower_type::smallint END,
                CASE WHEN foreign_national IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE foreign_national END,
                CASE WHEN borrower_credit_quality ~* '[A-Za-z]' THEN NULL ELSE borrower_credit_quality END,
                CASE WHEN borrower_year_of_birth ~* '[A-Za-z]' THEN NULL ELSE borrower_year_of_birth END,
                CASE WHEN number_of_debtors ~* '[A-Za-z]' THEN NULL ELSE number_of_debtors::smallint END, 
                CASE WHEN second_applicant_year_of_birth ~* '[A-Za-z]' THEN NULL ELSE second_applicant_year_of_birth END,
                CASE WHEN borrowers_employment_status ~* '[A-Za-z]' THEN NULL ELSE borrowers_employment_status::smallint END,
                CASE WHEN first_time_buyer IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE first_time_buyer END,
                CASE WHEN right_to_buy IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE right_to_buy END,
                CASE WHEN right_to_buy_price ~* '[A-Za-z]' THEN NULL ELSE right_to_buy_price::float END,
                CASE WHEN class_of_borrower IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE class_of_borrower END,
                CASE WHEN primary_income ~* '[A-Za-z]' THEN NULL ELSE primary_income::float END,
                CASE WHEN income_verification_for_primary_income IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE income_verification_for_primary_income END,
                CASE WHEN secondary_income ~* '[A-Za-z]' THEN NULL ELSE secondary_income::float END,
                CASE WHEN income_verification_for_secondary_income IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE income_verification_for_secondary_income END,
                CASE WHEN resident ~* '[A-Za-z]' THEN NULL ELSE resident::smallint END,
                CASE WHEN number_of_county_court_judgements_or_equivalent_satisfied ~* '[A-Za-z]' THEN NULL ELSE number_of_county_court_judgements_or_equivalent_satisfied::int END,
                CASE WHEN value_of_county_court_judgements_or_equivalent_satisfied ~* '[A-Za-z]' THEN NULL ELSE value_of_county_court_judgements_or_equivalent_satisfied::float END,
                CASE WHEN number_of_county_court_judgements_or_equivalent_unsatisfied ~* '[A-Za-z]' THEN NULL ELSE number_of_county_court_judgements_or_equivalent_unsatisfied::int END,
                CASE WHEN value_of_county_court_judgements_or_equivalent_unsatisfied ~* '[A-Za-z]' THEN NULL ELSE value_of_county_court_judgements_or_equivalent_unsatisfied::float END,
                CASE WHEN last_county_court_judgements_or_equivalent_year ~* '[A-Za-z]' THEN NULL ELSE last_county_court_judgements_or_equivalent_year END,
                CASE WHEN bankruptcy_or_individual_voluntary_arrangement_flag IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE bankruptcy_or_individual_voluntary_arrangement_flag END,
                CASE WHEN bureau_krediet_registratie_1_to_10_credit_type ~* '[A-Za-z]' THEN NULL ELSE bureau_krediet_registratie_1_to_10_credit_type::smallint END,
                CASE WHEN bureau_krediet_registratie_1_to_10_registration_date ~* '[A-Za-z]' THEN NULL ELSE bureau_krediet_registratie_1_to_10_registration_date END,
                CASE WHEN bureau_krediet_registratie_1_to_10_arrears_code ~* '[A-Za-z]' THEN NULL ELSE bureau_krediet_registratie_1_to_10_arrears_code::smallint END,
                CASE WHEN bureau_krediet_registratie_1_to_10_credit_amount ~* '[A-Za-z]' THEN NULL ELSE bureau_krediet_registratie_1_to_10_credit_amount::float END,
                CASE WHEN bureau_krediet_registratie_1_to_10_is_coding_cured IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE bureau_krediet_registratie_1_to_10_is_coding_cured END,
                CASE WHEN bureau_krediet_registratie_1_to_10_number_of_months_since_cured ~* '[A-Za-z]' THEN NULL ELSE bureau_krediet_registratie_1_to_10_number_of_months_since_cured::int END,
                CASE WHEN bureau_score_provider ~* '[A-Za-z]' THEN NULL ELSE bureau_score_provider::smallint END,
                CASE WHEN bureau_score_type ~* '[A-Za-z]' THEN NULL ELSE bureau_score_type::smallint END,
                CASE WHEN bureau_score_date ~* '[A-Za-z]' THEN NULL ELSE bureau_score_date END,
                CASE WHEN bureau_score_value IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE bureau_score_value END,
                CASE WHEN prior_repossessions IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE prior_repossessions END,
                CASE WHEN previous_mortgage_arrears_6_months ~* '[A-Za-z]' THEN NULL ELSE previous_mortgage_arrears_6_months::smallint END,
                CASE WHEN previous_mortgage_arrears_over_6_months ~* '[A-Za-z]' THEN NULL ELSE previous_mortgage_arrears_over_6_months::smallint END,
                CASE WHEN loan_origination_date ~* '[A-Za-z]' THEN NULL ELSE TO_DATE(CONCAT(REPLACE(loan_origination_date,'-','/'),'/01'),'YYYY/MM/DD')::date END AS loan_origination_date,
                CASE WHEN date_of_loan_maturity ~* '[A-Za-z]' THEN NULL ELSE TO_DATE(CONCAT(REPLACE(date_of_loan_maturity,'-','/'),'/01'),'YYYY/MM/DD')::date END AS date_of_loan_maturity,
                CASE WHEN account_status_date ~* '[A-Za-z]' THEN NULL ELSE TO_DATE(CONCAT(REPLACE(account_status_date,'-','/'),'/01'),'YYYY/MM/DD')::date END AS account_status_date,
                CASE WHEN origination_channel_arranging_cank_or_division ~* '[A-Za-z]' THEN NULL ELSE origination_channel_arranging_cank_or_division::smallint END,
                CASE WHEN purpose ~* '[A-Za-z]' THEN NULL ELSE purpose::smallint END,
                CASE WHEN shared_ownership ~* '[A-Za-z]' THEN NULL ELSE shared_ownership::smallint END,
                CASE WHEN loan_term ~* '[A-Za-z]' THEN NULL ELSE loan_term::int END,
                CASE WHEN principal_grace_period ~* '[A-Za-z]' THEN NULL ELSE principal_grace_period::int END,
                CASE WHEN amount_guaranteed ~* '[A-Za-z]' THEN NULL ELSE amount_guaranteed::float END,
                CASE WHEN subsidy IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE subsidy END,
                CASE WHEN loan_currency_denomination IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE loan_currency_denomination END,
                CASE WHEN original_balance ~* '[A-Za-z]' THEN 0 ELSE original_balance::float END AS original_balance,
                CASE WHEN current_balance ~* '[A-Za-z]' THEN 0 ELSE current_balance::float END AS current_balance, 
                CASE WHEN fractioned_subrogated_loans IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE fractioned_subrogated_loans END,
                CASE WHEN repayment_method ~* '[A-Za-z]' THEN NULL ELSE repayment_method::smallint END,
                CASE WHEN payment_frequency ~* '[A-Za-z]' THEN NULL ELSE payment_frequency::smallint END,
                CASE WHEN payment_due ~* '[A-Za-z]' THEN NULL ELSE payment_due::float END,
                CASE WHEN payment_type ~* '[A-Za-z]' THEN NULL ELSE payment_type::smallint END,
                CASE WHEN debt_to_income ~* '[A-Za-z]' THEN NULL ELSE debt_to_income::float END,
                CASE WHEN type_of_guarantee_provider ~* '[A-Za-z]' THEN NULL ELSE type_of_guarantee_provider::smallint END,
                CASE WHEN guarantee_provider ~* '[A-Za-z]' THEN NULL ELSE guarantee_provider::smallint END,
                CASE WHEN income_guarantor ~* '[A-Za-z]' THEN NULL ELSE income_guarantor::float END AS guarantor_income,
                CASE WHEN subsidy_received ~* '[A-Za-z]' THEN NULL ELSE subsidy_received::float END AS subsidy_received,
                CASE WHEN mortgage_indemnity_guarantee_provider IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE mortgage_indemnity_guarantee_provider END,	
                CASE WHEN mortgage_indemnity_guarantee_attachment_point ~* '[A-Za-z]' THEN NULL ELSE mortgage_indemnity_guarantee_attachment_point::float END,	
                CASE WHEN prior_balances ~* '[A-Za-z]' THEN NULL ELSE prior_balances::float END,	
                CASE WHEN other_prior_balances ~* '[A-Za-z]' THEN NULL ELSE other_prior_balances::float END,	
                CASE WHEN pari_passu_loans ~* '[A-Za-z]' THEN NULL ELSE pari_passu_loans::float END,
                CASE WHEN subordinated_claims ~* '[A-Za-z]' THEN NULL ELSE subordinated_claims::float END,	
                CASE WHEN lien ~* '[A-Za-z]' THEN NULL ELSE lien::smallint END,	
                CASE WHEN retained_amount ~* '[A-Za-z]' THEN NULL ELSE retained_amount::float END,	
                CASE WHEN retained_amount_date ~* '[A-Za-z]' THEN NULL ELSE TO_DATE(CONCAT(REPLACE(retained_amount_date,'-','/'),'/01'),'YYYY/MM/DD')::date END AS retained_amount_date,
                CASE WHEN maximum_balance ~* '[A-Za-z]' THEN NULL ELSE maximum_balance::float END,	
                CASE WHEN further_loan_advance ~* '[A-Za-z]' THEN NULL ELSE further_loan_advance::float END,	
                CASE WHEN further_loan_advance_date ~* '[A-Za-z]' THEN NULL ELSE further_loan_advance_date::date END,
                CASE WHEN flexible_loan_amount ~* '[A-Za-z]' THEN NULL ELSE flexible_loan_amount::float END,	
                CASE WHEN further_advances IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE further_advances END,
                CASE WHEN length_of_payment_holiday ~* '[A-Za-z]' THEN NULL ELSE length_of_payment_holiday::int END,
                CASE WHEN subsidy_period ~* '[A-Za-z]' THEN NULL ELSE subsidy_period::int END,
                CASE WHEN mortgage_inscription ~* '[A-Za-z]' THEN NULL ELSE mortgage_inscription::float END,
                CASE WHEN mortgage_mandate ~* '[A-Za-z]' THEN NULL ELSE mortgage_mandate::float END,
                CASE WHEN deed_of_postponement IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE deed_of_postponement END,
                CASE WHEN pre_payment_amount ~* '[A-Za-z]' THEN NULL ELSE pre_payment_amount::float END,
                CASE WHEN pre_payment_date ~* '[A-Za-z]' THEN NULL ELSE pre_payment_date::date END,
                CASE WHEN pre_payment_penalties ~* '[A-Za-z]' THEN NULL ELSE pre_payment_penalties::float END,
                CASE WHEN cumulative_pre_payments ~* '[A-Za-z]' THEN NULL ELSE cumulative_pre_payments::float END,
                CASE WHEN percentage_of_pre_payments_allowed_per_year ~* '[A-Za-z]' THEN NULL ELSE percentage_of_pre_payments_allowed_per_year::float END,
                CASE WHEN interest_rate_type ~* '[A-Za-z]' THEN NULL ELSE interest_rate_type::smallint END,
                CASE WHEN current_interest_rate_index ~* '[A-Za-z]' THEN NULL ELSE current_interest_rate_index::smallint END,
                CASE WHEN current_interest_rate ~* '[A-Za-z]' THEN NULL ELSE current_interest_rate::float END,
                CASE WHEN current_interest_rate_margin ~* '[A-Za-z]' THEN NULL ELSE current_interest_rate_margin::float END,
                CASE WHEN interest_rate_reset_interval ~* '[A-Za-z]' THEN NULL ELSE interest_rate_reset_interval::int END,
                CASE WHEN interest_cap_rate ~* '[A-Za-z]' THEN NULL ELSE interest_cap_rate::float END,
                CASE WHEN revision_margin_1 ~* '[A-Za-z]' THEN NULL ELSE revision_margin_1::float END,
                CASE WHEN interest_revision_date_1 ~* '[A-Za-z]' THEN NULL ELSE interest_revision_date_1 END,
                CASE WHEN revision_margin_2 ~* '[A-Za-z]' THEN NULL ELSE revision_margin_2::float END,
                CASE WHEN interest_revision_date_2 ~* '[A-Za-z]' THEN NULL ELSE interest_revision_date_2 END,
                CASE WHEN revision_margin_3 ~* '[A-Za-z]' THEN NULL ELSE revision_margin_3::float END,
                CASE WHEN interest_revision_date_3 ~* '[A-Za-z]' THEN NULL ELSE interest_revision_date_3 END,
                CASE WHEN revised_interest_rate_index ~* '[A-Za-z]' THEN NULL ELSE revised_interest_rate_index::smallint END,
                CASE WHEN final_margin ~* '[A-Za-z]' THEN NULL ELSE final_margin::float END,
                CASE WHEN final_step_date ~* '[A-Za-z]' THEN NULL ELSE final_step_date::date END,
                CASE WHEN restructuring_arrangement IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE restructuring_arrangement END,
                CASE WHEN geographic_region_list IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE geographic_region_list END,
                CASE WHEN property_postcode ~* '[A-Za-z]' THEN NULL ELSE property_postcode::int END,
                CASE WHEN occupancy_type ~* '[A-Za-z]' THEN NULL ELSE occupancy_type::smallint END,
                CASE WHEN property_type ~* '[A-Za-z]' THEN NULL ELSE property_type::smallint END,
                CASE WHEN new_property ~* '[A-Za-z]' THEN NULL ELSE new_property::smallint END,
                CASE WHEN construction_year ~* '[A-Za-z]' THEN NULL ELSE construction_year END,
                CASE WHEN property_rating IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE property_rating END,
                CASE WHEN original_loan_to_value ~* '[A-Za-z]' THEN NULL ELSE original_loan_to_value::float END,
                CASE WHEN valuation_amount ~* '[A-Za-z]' THEN NULL ELSE valuation_amount::float END,
                CASE WHEN original_valuation_type ~* '[A-Za-z]' THEN NULL ELSE original_valuation_type::int END,
                CASE WHEN valuation_date ~* '[A-Za-z]' THEN NULL ELSE TO_DATE(CONCAT(REPLACE(valuation_date,'-','/'),'/01'),'YYYY/MM/DD')::date END AS valuation_date,
                CASE WHEN confidence_interval_for_original_automated_valuation_model_valuation ~* '[A-Za-z]' THEN NULL ELSE confidence_interval_for_original_automated_valuation_model_valuation::numeric END,
                CASE WHEN provider_of_original_automated_valuation_model_valuation IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE provider_of_original_automated_valuation_model_valuation END,
                CASE WHEN current_loan_to_value ~* '[A-Za-z]' THEN NULL ELSE current_loan_to_value::float END,
                CASE WHEN purchase_price_lower_limit ~* '[A-Za-z]' THEN NULL ELSE purchase_price_lower_limit::float END,
                CASE WHEN current_valuation_amount ~* '[A-Za-z]' THEN NULL ELSE current_valuation_amount::float END,
                CASE WHEN current_valuation_type ~* '[A-Za-z]' THEN NULL ELSE current_valuation_type::smallint END,
                CASE WHEN current_valuation_date ~* '[A-Za-z]' THEN NULL ELSE TO_DATE(CONCAT(REPLACE(current_valuation_date,'-','/'),'/01'),'YYYY/MM/DD')::date END AS current_valuation_date,
                CASE WHEN confidence_interval_for_current_automated_valuation_model_valuation ~* '[A-Za-z]' THEN NULL ELSE confidence_interval_for_current_automated_valuation_model_valuation::numeric END,
                CASE WHEN provider_of_current_automated_valuation_model_valuation IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE provider_of_current_automated_valuation_model_valuation END,
                CASE WHEN property_value_at_time_of_latest_loan_advance ~* '[A-Za-z]' THEN NULL ELSE property_value_at_time_of_latest_loan_advance::float END,
                CASE WHEN indexed_foreclosure_value ~* '[A-Za-z]' THEN NULL ELSE indexed_foreclosure_value::float END,
                CASE WHEN ipoteca ~* '[A-Za-z]' THEN NULL ELSE ipoteca::float END,
                CASE WHEN date_of_sale ~* '[A-Za-z]' THEN NULL ELSE date_of_sale::date END,
                CASE WHEN additional_collateral ~* '[A-Za-z]' THEN NULL ELSE additional_collateral::smallint END,	
                CASE WHEN additional_collateral_provider IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE additional_collateral_provider END,
                CASE WHEN gross_annual_rental_income ~* '[A-Za-z]' THEN NULL ELSE gross_annual_rental_income::float END,	
                CASE WHEN number_of_buy_to_let_properties ~* '[A-Za-z]' THEN NULL ELSE number_of_buy_to_let_properties::int END,
                CASE WHEN debt_service_coverage_ratio ~* '[A-Za-z]' THEN NULL ELSE debt_service_coverage_ratio::float END,
                CASE WHEN additional_collateral_value ~* '[A-Za-z]' THEN NULL ELSE additional_collateral_value::float END,
                CASE WHEN real_estate_owned IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE real_estate_owned END,
                CASE WHEN is_property_transferability_limited IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE is_property_transferability_limited END,
                CASE WHEN time_until_declassification ~* '[A-Za-z]' THEN NULL ELSE time_until_declassification::numeric END,
                CASE WHEN account_status ~* '[A-Za-z]' THEN NULL ELSE account_status::smallint END,
                CASE WHEN date_last_current ~* '[A-Za-z]' THEN NULL ELSE LAST_DAY(TO_DATE(CONCAT(REPLACE(date_last_current,'-','/'),'/01'),'YYYY/MM/DD'))::date END AS date_last_current,
                CASE WHEN date_last_in_arrears ~* '[A-Za-z]' THEN NULL ELSE date_last_in_arrears END,
                CASE WHEN arrears_balance ~* '[A-Za-z]' THEN NULL ELSE arrears_balance::float END,
                CASE WHEN number_months_in_arrears ~* '[A-Za-z]' OR arrears_balance <= 5 THEN NULL ELSE number_months_in_arrears::int END,
                CASE WHEN arrears_1_month_ago ~* '[A-Za-z]' THEN NULL ELSE arrears_1_month_ago::float END,
                CASE WHEN arrears_2_months_ago ~* '[A-Za-z]' THEN NULL ELSE arrears_2_months_ago::float END,
                CASE WHEN performance_arrangement ~* '[A-Za-z]' THEN NULL ELSE performance_arrangement END,
                CASE WHEN litigation IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE litigation END,
                CASE WHEN redemption_date ~* '[A-Za-z]' THEN NULL ELSE LAST_DAY(TO_DATE(CONCAT(REPLACE(redemption_date,'-','/'),'/01'),'YYYY/MM/DD'))::date END AS redemption_date,
                CASE WHEN months_in_arrears_prior ~* '[A-Za-z]' THEN NULL ELSE months_in_arrears_prior::float END,
                CASE WHEN default_or_foreclosure ~* '[A-Za-z]' THEN NULL ELSE default_or_foreclosure::float END,
                CASE WHEN date_of_default_or_foreclosure ~* '[A-Za-z]' THEN NULL ELSE date_of_default_or_foreclosure END,
                CASE WHEN sale_price_lower_limit ~* '[A-Za-z]' THEN NULL ELSE sale_price_lower_limit::float END,
                CASE WHEN loss_on_sale ~* '[A-Za-z]' THEN NULL ELSE loss_on_sale::float END,
                CASE WHEN cumulative_recoveries ~* '[A-Za-z]' THEN NULL ELSE cumulative_recoveries::float END,
                CASE WHEN professional_negligence_recoveries ~* '[A-Za-z]' THEN NULL ELSE professional_negligence_recoveries::float END,
                CASE WHEN loan_flagged_as_contencioso IN ('ND', 'ND,1', 'ND,2', 'ND,3', 'ND,4', 'ND,5', 'ND,6') THEN NULL ELSE loan_flagged_as_contencioso END
            FROM etl_reporting.rabobank_mortgages AS a
            WHERE current_balance > 0 OR retained_amount > 0
        )
        SELECT 
            *,
            GETDATE() AS etl_updated,
            USER AS created_by                      
        FROM temp_tbl;
    """,

    "05_Rabo_Raw_Temp": """
        INSERT INTO credit_risk_playground.stg_mrt_rabobank_raw_temp
        WITH report_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        rabo_raw AS (
            SELECT 
                a.*,
                current_balance - (CASE WHEN retained_amount IS NULL THEN 0 ELSE retained_amount::float END) AS current_balance_adj
            FROM credit_risk_playground.stg_mrt_rabobank_raw AS a
            INNER JOIN report_date AS b
            ON a.pool_cut_off_date = b.reporting_date::date
        )
        SELECT * FROM rabo_raw
        WHERE account_status <> 4 AND current_balance <> 0;
    """,

    "06_Rabo_Arrears_List": """
        INSERT INTO credit_risk_playground.stg_mrt_rabobank_arrears_list
        WITH reporting_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        rabo_customer_level_expo AS (
            SELECT 
                pool_cut_off_date,
                borrower_identifier,
                COUNT(DISTINCT loan_identifier) AS number_of_loans,
                SUM(original_balance) AS customer_level_original_balance,
                SUM(current_balance) AS customer_level_current_balance,
                SUM(current_valuation_amount) AS total_collateral_value
            FROM credit_risk_playground.stg_mrt_rabobank_raw a
            INNER JOIN reporting_date b
            ON a.pool_cut_off_date = b.reporting_date
            WHERE current_balance > 0 
            GROUP BY pool_cut_off_date, borrower_identifier
        ),
        previous_arrears_list AS (
            SELECT a.* 
            FROM credit_risk_playground.stg_mrt_rabobank_arrears_list a
            INNER JOIN reporting_date AS b
            ON a.pool_cut_off_date::date = LAST_DAY(DATE_ADD('month', -1, b.reporting_date::date)::date)
        ),
        temp_tbl AS (
            SELECT 
                x.pool_cut_off_date,
                x.borrower_identifier,
                x.loan_identifier,
                CASE WHEN type_of_guarantee_provider = 7 THEN 'NHG' ELSE 'Non-NHG' END AS segment,
                account_status,
                number_months_in_arrears AS number_of_months_arrears,
                current_balance,
                customer_level_current_balance,
                arrears_balance AS arrears_balance_total,
                CASE WHEN number_of_loans IN (NULL, 0) THEN 0 ELSE (arrears_balance / number_of_loans) END AS arrears_balance,
                CASE WHEN customer_level_current_balance IN (NULL, 0) THEN NULL ELSE ROUND(100 * (arrears_balance / customer_level_current_balance), 1) END AS arrears_balance_ratio
            FROM credit_risk_playground.stg_mrt_rabobank_raw AS x
            INNER JOIN rabo_customer_level_expo AS y
            ON x.pool_cut_off_date = y.pool_cut_off_date
            AND x.borrower_identifier = y.borrower_identifier
            AND x.current_balance > 0
        ),
        temp_tbl1 AS (
            SELECT 
                t1.*,
                t2.account_status AS prev_account_status,
                t2.months_in_arrears AS prev_months_in_arrears,
                t2.consecutive_months_in_arrears AS prev_consecutive_months_in_arrears,
                t2.first_date_in_arrears AS prev_first_date_in_arrears,
                t2.last_arrears_info_date AS prev_last_arrears_info_date
            FROM temp_tbl AS t1
            LEFT JOIN previous_arrears_list AS t2
            USING(borrower_identifier, loan_identifier)
        ),
        months_in_arrears AS (
            SELECT
                tbl1.*,
                CASE WHEN prev_first_date_in_arrears IS NULL THEN pool_cut_off_date ELSE prev_first_date_in_arrears END AS first_date_in_arrears,
                CASE WHEN prev_months_in_arrears = 'probation' AND arrears_balance_total = 0 THEN prev_last_arrears_info_date ELSE pool_cut_off_date END AS last_arrears_info_date,
                CASE 
                    WHEN prev_consecutive_months_in_arrears IS NULL AND arrears_balance_total > 5 THEN 1
                    WHEN prev_months_in_arrears = 'probation' AND arrears_balance_total = 0 THEN 0
                    ELSE prev_consecutive_months_in_arrears + 1
                END AS consecutive_months_in_arrears
            FROM temp_tbl1 AS tbl1
        ),
        arrears_aggregation AS (
            SELECT 
                arrs.*,
                CASE 
                    WHEN account_status IS NULL OR consecutive_months_in_arrears IS NULL OR arrears_balance < 5 OR arrears_balance IS NULL THEN 'performing'
                    WHEN account_status IN (2,3) AND consecutive_months_in_arrears = 3 AND ROUND(arrears_balance_ratio, 1) < 1 THEN consecutive_months_in_arrears::text
                    WHEN account_status IN (2,3) AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 'default'
                    WHEN account_status IN (2,3) AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) < 1 THEN '>3'
                    WHEN prev_months_in_arrears = 'default' AND arrears_balance_total > 5 THEN 'default'
                    WHEN prev_months_in_arrears = 'default' AND arrears_balance_total = 0 THEN 'probabtion'
                    WHEN prev_months_in_arrears = 'probation' AND DATEDIFF(month, last_arrears_info_date, pool_cut_off_date) > 12 THEN 'performing'
                    ELSE consecutive_months_in_arrears::text
                END months_in_arrears,
                ROW_NUMBER() OVER(PARTITION BY borrower_identifier, loan_identifier, segment, current_balance ORDER BY borrower_identifier, loan_identifier, segment, current_balance) AS duplicates
            FROM months_in_arrears AS arrs
        )
        SELECT 
            pool_cut_off_date,
            borrower_identifier,
            loan_identifier,
            segment,
            account_status,
            number_of_months_arrears, 
            consecutive_months_in_arrears,
            months_in_arrears::text,
            arrears_balance_ratio,
            first_date_in_arrears,
            last_arrears_info_date,
            ROUND(arrears_balance_total, 2) AS arrears_balance_customer_level,
            ROUND(arrears_balance, 2) AS arrears_balance_loan_level,
            current_balance,
            ROUND(customer_level_current_balance, 2) AS customer_level_current_balance
        FROM arrears_aggregation
        WHERE current_balance > 0 AND months_in_arrears <> 'performing' AND duplicates = 1
        ORDER BY pool_cut_off_date, consecutive_months_in_arrears, borrower_identifier, loan_identifier;
    """,

    "07_Rabo_Credit_Risk_Calculations": """
        INSERT INTO credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations
        WITH reporting_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        rabo_exposures AS (
            SELECT 
                pool_cut_off_date,
                borrower_identifier,
                loan_identifier,
                CASE WHEN type_of_guarantee_provider = 7 THEN 'NHG' ELSE 'Non-NHG' END AS segment,
                primary_income,
                secondary_income,
                CASE WHEN secondary_income IS NULL THEN primary_income ELSE primary_income + secondary_income END AS total_income,
                CASE WHEN borrowers_employment_status IS NULL THEN 10 ELSE borrowers_employment_status END AS borrowers_employment_status,			
                loan_origination_date,
                date_of_loan_maturity,
                TO_DATE(CONCAT(REPLACE(interest_revision_date_1, '-', '/'), '/01'), 'YYYY/MM/DD') AS interest_revision_date,
                loan_term,
                CASE WHEN loan_term <= 24 THEN 'Brigding' ELSE 'Standard' END AS purpose,
                original_balance,
                current_balance_adj AS current_balance,
                CASE WHEN retained_amount IS NULL THEN 0 ELSE retained_amount::float END AS undrawn_loanpart,
                payment_type, 
                payment_due,
                debt_to_income,
                current_interest_rate,
                interest_rate_reset_interval,
                geographic_region_list,
                original_loan_to_value,
                valuation_amount,
                current_loan_to_value,
                current_valuation_amount,
                property_value_at_time_of_latest_loan_advance,
                indexed_foreclosure_value,
                CASE WHEN type_of_guarantee_provider = 7 THEN GREATEST(0, current_balance - (original_balance * 0.1)) ELSE 0 END AS guaranteed_foreclosure_value,
                debt_service_coverage_ratio,
                CASE WHEN arrears_balance = 0 THEN 1 ELSE account_status END AS account_status,
                CASE WHEN number_months_in_arrears IS NULL THEN 0 WHEN arrears_balance = 0 THEN 0 ELSE number_months_in_arrears END AS number_months_in_arrears,
                CASE WHEN arrears_balance = 0 THEN NULL ELSE LAST_DAY(DATEADD(Day, 15, date_last_current))::date END AS first_date_in_arrears,
                CASE 
                    WHEN date_last_in_arrears IS NULL AND date_last_current IS NULL THEN NULL
                    WHEN date_last_in_arrears IS NULL AND date_last_current IS NOT NULL THEN pool_cut_off_date::date
                    WHEN date_last_in_arrears IS NOT NULL THEN date_last_in_arrears::date
                END AS last_arrears_info_date,
                arrears_balance,
                property_type,
                CASE 
                    WHEN property_type IN (1, 3, 5, 6, 7) THEN 'house'
                    WHEN property_type = 2 THEN 'apartment'
                    WHEN property_type = 4 THEN 'terraced'
                    WHEN property_type IN (8,9) THEN 'commercial'
                    WHEN property_type = 10 THEN 'land'
                    ELSE 'other'
                END AS property_class,
                construction_year,
                CASE 
                    WHEN construction_year <= 1945 OR construction_year IS NULL THEN 1
                    WHEN construction_year <= 1964 THEN 2
                    WHEN construction_year <= 1974 THEN 3
                    WHEN construction_year <= 1982 THEN 4
                    WHEN construction_year <= 1987 THEN 5
                    WHEN construction_year <= 1991 THEN 6
                    WHEN construction_year <= 1999 THEN 7
                    WHEN construction_year <= 2005 THEN 8
                    ELSE 9
                END building_year,
                CASE 
                    WHEN building_year = 1 AND property_class IN ('house','apartment') THEN 'G'
                    WHEN building_year = 1 AND property_class IN ('terraced') THEN 'F'
                    WHEN building_year = 2 AND property_class IN ('house') THEN 'F'
                    WHEN building_year = 2 AND property_class IN ('apartment', 'terraced') THEN 'E'
                    WHEN building_year = 3 AND property_class IN ('house') THEN 'D'
                    WHEN building_year = 3 AND property_class IN ('apartment') THEN 'F'
                    WHEN building_year = 3 AND property_class IN ('terraced') THEN 'C'
                    WHEN building_year IN (4,5,6) THEN 'C'
                    WHEN building_year = 7 THEN 'B'
                    WHEN building_year = 8 AND property_class IN ('house','apartment') THEN 'B'
                    ELSE 'A'
                END AS energy_label
            FROM credit_risk_playground.stg_mrt_rabobank_raw_temp AS rabo
            INNER JOIN reporting_date rdate
            ON rabo.pool_cut_off_date = rdate.reporting_date
        ),
        rabo_customer_level_expo AS (
            SELECT 
                pool_cut_off_date,
                borrower_identifier,
                SUM(original_balance) AS customer_level_original_balance,
                SUM(current_balance) AS customer_level_current_balance,
                SUM(current_valuation_amount) AS total_collateral_value
            FROM rabo_exposures
            GROUP BY pool_cut_off_date, borrower_identifier
        ),
        bridge_exposures AS (
            SELECT 
                pool_cut_off_date,
                borrower_identifier,
                SUM(CASE WHEN current_balance IS NULL THEN 0 ELSE current_balance END) AS current_bridge_exposure
            FROM rabo_exposures
            WHERE loan_term <= 24
            GROUP BY pool_cut_off_date, borrower_identifier
        ),
        mev_plus_parameter_inputs AS (
            SELECT 
                temp_mev.*,
                EXP(pd_factor_constant + (pd_factor_ur_coeff * diff_12m_ur) + (pd_factor_ir_coeff * diff_12m_ir)) AS pd_factor,
                EXP(indexed_vlaue_scalar_const + (indexed_vlaue_scalar_rhpi_coeff * log_diff_12m_rhpi_lag_1) + (indexed_vlaue_scalar_gdp_coeff * diff_12m_gdp)) AS indexed_vlaue_scalar
            FROM (
                SELECT 
                    a.ref_date,
                    a.log_diff_12m_rhpi_lag_1,
                    a.diff_12m_ur, 
                    a.diff_12m_ir, 
                    a.diff_12m_gdp,
                    b.value AS cure_rate,
                    c.coefficient AS pd_factor_constant,
                    d.coefficient AS pd_factor_ur_coeff,
                    e.coefficient AS pd_factor_ir_coeff,
                    f.coefficient AS indexed_vlaue_scalar_const,
                    g.coefficient AS indexed_vlaue_scalar_rhpi_coeff,
                    h.coefficient AS indexed_vlaue_scalar_gdp_coeff,
                    i.surcharge AS forcl_discount,
                    j.surcharge AS forcl_variable_cost
                FROM credit_risk_playground.stg_mrt_mev_input_data AS a, 
                     credit_risk_playground.stg_mrt_portfolio_parameters AS b,
                     credit_risk_playground.stg_mrt_risk_model_parameters AS c,
                     credit_risk_playground.stg_mrt_risk_model_parameters AS d,
                     credit_risk_playground.stg_mrt_risk_model_parameters AS e,
                     credit_risk_playground.stg_mrt_risk_model_parameters AS f,
                     credit_risk_playground.stg_mrt_risk_model_parameters AS g,
                     credit_risk_playground.stg_mrt_risk_model_parameters AS h,
                     credit_risk_playground.stg_mrt_fitch_foreclosure_discounts AS i,
                     credit_risk_playground.stg_mrt_fitch_foreclosure_discounts AS j
                WHERE b.parameter_name = 'cure_rate'
                AND c.main_model = 'pd' AND c.sub_model = 'pd_factor' AND c.variable_name = 'constant' 
                AND d.main_model = 'pd' AND d.sub_model = 'pd_factor' AND d.variable_name = 'diff_12m_ur' 
                AND e.main_model = 'pd' AND e.sub_model = 'pd_factor' AND e.variable_name = 'diff_12m_ir' 
                AND f.main_model = 'lgd' AND f.sub_model = 'indexed_value_scaler' AND f.variable_name = 'constant'
                AND g.main_model = 'lgd' AND g.sub_model = 'indexed_value_scaler' AND g.variable_name = 'log_diff_12m_rhpi_lag_1'
                AND h.main_model = 'lgd' AND h.sub_model = 'indexed_value_scaler' AND h.variable_name = 'diff_12m_gdp'
                AND i.discount = 'Foreclosed_Sale_Adjustment'
                AND j.discount = 'Variable_Foreclosure_Cost'
            ) temp_mev
        ),
        temp_aggregation AS (
            SELECT 
                rabo_expo.*,
                CASE 
                    WHEN arrears_balance = 0 THEN 0 
                    ELSE COALESCE(DATEDIFF('months', rabo_expo.first_date_in_arrears, rabo_expo.last_arrears_info_date) + 1, 0)
                END AS consecutive_months_in_arrears,
                cust_level_expo.customer_level_original_balance,
                cust_level_expo.customer_level_current_balance,	
                cust_level_expo.total_collateral_value,
                CASE WHEN bridge.current_bridge_exposure IS NULL THEN 0 ELSE bridge.current_bridge_exposure END AS current_bridge_exposure 
            FROM rabo_exposures AS rabo_expo
            LEFT JOIN rabo_customer_level_expo AS cust_level_expo
            ON rabo_expo.pool_cut_off_date = cust_level_expo.pool_cut_off_date
            AND rabo_expo.borrower_identifier = cust_level_expo.borrower_identifier
            LEFT JOIN bridge_exposures AS bridge
            ON cust_level_expo.pool_cut_off_date = bridge.pool_cut_off_date
            AND cust_level_expo.borrower_identifier = bridge.borrower_identifier
        ),
        temp_aggregation1 AS (
            SELECT 
                tmp_tbl.*,
                100 * (arrears_balance / customer_level_current_balance) AS arrears_balance_ratio,
                customer_level_current_balance / total_income AS current_debt_to_income_ratio,
                (customer_level_current_balance - current_bridge_exposure) / total_income AS current_debt_to_income_ratio_excl_bridge,
                CASE 
                    WHEN segment = 'NHG' THEN 0.75 * 0.1 * current_balance
                    ELSE 0.35 * current_balance + 0.4 * GREATEST(0, current_balance - 0.8 * current_valuation_amount)
                END AS rwa,
                DATEDIFF('month', pool_cut_off_date, LAST_DAY(interest_revision_date)) AS months_to_revision_date,
                DATEDIFF('year', pool_cut_off_date, date_of_loan_maturity) AS remaining_term,
                (current_balance - undrawn_loanpart) * ((current_interest_rate / 100) / 12) AS next_month_interest_payment
            FROM temp_aggregation AS tmp_tbl
        ),
        temp_aggregation1a AS ( 
            SELECT 
                *,
                GREATEST(0, payment_due - next_month_interest_payment) AS one_month_paydown
            FROM temp_aggregation1	
        ),
        temp_aggregation2 AS (
            SELECT 
                tmp_tbl1.*,
                (current_balance - undrawn_loanpart + one_month_paydown) * ((current_interest_rate / 100) / 12) AS prev_month_interest_payment,
                CASE WHEN loan_term <= 24 THEN 0 ELSE ROUND(original_balance / loan_term, 2) END AS monthly_redemption
            FROM temp_aggregation1a AS tmp_tbl1
        ),
        temp_aggregation3 AS (
            SELECT 
                tmp_tbl3.*,
                total_income / 12 AS total_monthly_income,
                cust_level_prev_month_interest_payment,
                cust_level_monthly_redemption,
                CASE WHEN loan_term <= 24 THEN 0 ELSE cust_level_monthly_redemption + cust_level_prev_month_interest_payment END AS total_monthly_debt_service
            FROM temp_aggregation2 AS tmp_tbl3
            LEFT JOIN (
                SELECT 
                    pool_cut_off_date,
                    borrower_identifier,
                    SUM(prev_month_interest_payment) AS cust_level_prev_month_interest_payment,
                    SUM(monthly_redemption) AS cust_level_monthly_redemption
                FROM temp_aggregation2
                GROUP BY 1, 2
            ) tmp_tbl4
            ON tmp_tbl3.pool_cut_off_date = tmp_tbl4.pool_cut_off_date
            AND tmp_tbl3.borrower_identifier = tmp_tbl4.borrower_identifier
        ),
        temp_aggregation3a AS (
            SELECT
                *,
                CASE WHEN loan_term <= 24 OR current_balance = 0 THEN 0 ELSE 100 * (total_monthly_debt_service / total_monthly_income) END AS fitch_dti	
            FROM temp_aggregation3
        ),
        temp_aggregation4 AS (
            SELECT 
                *,
                CASE
                    WHEN current_loan_to_value IS NULL THEN NULL
                    WHEN current_loan_to_value < 65 THEN 1
                    WHEN current_loan_to_value < 85 THEN 2
                    WHEN current_loan_to_value < 95 THEN 3
                    ELSE 4
                END AS fitch_ltv_class,
                CASE 
                    WHEN months_to_revision_date IS NULL THEN NULL
                    WHEN months_to_revision_date / 12 < 8 THEN 1
                    WHEN months_to_revision_date / 12 < 12 THEN 2
                    WHEN months_to_revision_date / 12 < 17 THEN 3
                    WHEN months_to_revision_date / 12 < 25 THEN 4
                    ELSE 5
                END AS fitch_term_class,
                CASE 
                    WHEN loan_term IS NULL THEN NULL
                    WHEN loan_term <= 24 THEN 0
                    WHEN fitch_dti < 20 THEN 1
                    WHEN fitch_dti < 30 THEN 2
                    WHEN fitch_dti < 40 THEN 3
                    WHEN fitch_dti < 50 THEN 4
                    ELSE 5
                END AS fitch_dti_class,
                CASE 
                    WHEN original_loan_to_value IS NULL THEN NULL
                    WHEN original_loan_to_value < 40 THEN '<40'
                    WHEN original_loan_to_value < 50 THEN '40-50'
                    WHEN original_loan_to_value < 60 THEN '50-60'
                    WHEN original_loan_to_value < 65 THEN '60-65'
                    WHEN original_loan_to_value < 70 THEN '65-70'
                    WHEN original_loan_to_value < 75 THEN '70-75'
                    WHEN original_loan_to_value < 80 THEN '75-80'
                    WHEN original_loan_to_value < 85 THEN '80-85'
                    WHEN original_loan_to_value < 90 THEN '85-90'
                    WHEN original_loan_to_value < 95 THEN '90-95'
                    WHEN original_loan_to_value < 100 THEN '95-100'
                    WHEN original_loan_to_value < 105 THEN '100-105'
                    WHEN original_loan_to_value < 110 THEN '105-110'
                    ELSE '>=110'
                END AS fitch_oltv_class,
                CASE
                    WHEN current_loan_to_value IS NULL THEN NULL
                    WHEN current_loan_to_value < 40 THEN '<40'
                    WHEN current_loan_to_value < 50 THEN '40-50'
                    WHEN current_loan_to_value < 60 THEN '50-60'
                    WHEN current_loan_to_value < 65 THEN '60-65'
                    WHEN current_loan_to_value < 70 THEN '65-70'
                    WHEN current_loan_to_value < 75 THEN '70-75'
                    WHEN current_loan_to_value < 80 THEN '75-80'
                    WHEN current_loan_to_value < 85 THEN '80-85'
                    WHEN current_loan_to_value < 90 THEN '85-90'
                    WHEN current_loan_to_value < 95 THEN '90-95'
                    WHEN current_loan_to_value < 100 THEN '95-100'
                    WHEN current_loan_to_value < 105 THEN '100-105'
                    WHEN current_loan_to_value < 110 THEN '105-110'
                    ELSE '>=110'	
                END AS fitch_cltv_class,
                CASE 
                    WHEN remaining_term IS NULL THEN NULL
                    WHEN remaining_term < 5 THEN '<5' 
                    WHEN remaining_term < 10 THEN '5-10' 
                    WHEN remaining_term < 15 THEN '10-15' 
                    WHEN remaining_term < 20 THEN '15-20'
                    WHEN remaining_term < 25 THEN '20-25' 
                    ELSE '>=25'
                END AS fitch_remaining_term_class,
                CASE 
                    WHEN original_loan_to_value IS NULL THEN NULL
                    WHEN original_loan_to_value < 50 THEN '<50'
                    WHEN original_loan_to_value < 60 THEN '50-60'
                    WHEN original_loan_to_value < 70 THEN '60-70'
                    WHEN original_loan_to_value < 80 THEN '70-80'
                    WHEN original_loan_to_value < 90 THEN '80-90'
                    ELSE '>=90' 	
                END AS fitch_oltv_term_class,
                CASE 
                    WHEN current_loan_to_value IS NULL THEN NULL
                    WHEN current_loan_to_value < 50 THEN '<50'
                    WHEN current_loan_to_value < 60 THEN '50-60'
                    WHEN current_loan_to_value < 70 THEN '60-70'
                    WHEN current_loan_to_value < 80 THEN '70-80'
                    WHEN current_loan_to_value < 90 THEN '80-90'
                    ELSE '>=90' 	
                END AS fitch_cltv_term_class
            FROM temp_aggregation3a
        ),
        temp_aggregation5 AS (
            SELECT
                temp_tbl6.*,
                base_ff.baseline_ff 
            FROM temp_aggregation4 AS temp_tbl6
            LEFT JOIN credit_risk_playground.stg_mrt_fitch_ff_baseline AS base_ff
            ON temp_tbl6.fitch_cltv_class = base_ff.ltv_class 
            AND temp_tbl6.fitch_dti_class = base_ff.dti_class 
        ),
        temp_aggregation6 AS (
            SELECT
                temp_tbl7.*,
                io_adjustment.io_adjustment
            FROM temp_aggregation5 AS temp_tbl7
            LEFT JOIN credit_risk_playground.stg_mrt_fitch_ff_io_adjustment AS io_adjustment
            ON temp_tbl7.fitch_oltv_term_class = io_adjustment.ltv_class 
            AND temp_tbl7.fitch_remaining_term_class = io_adjustment.remaining_term_class 
        ),
        temp_aggregation7 AS (
            SELECT
                temp_tbl8.*,
                income_adj.income_adjustment 
            FROM temp_aggregation6 AS temp_tbl8
            LEFT JOIN credit_risk_playground.stg_mrt_fitch_income_adjustment AS income_adj
            ON temp_tbl8.borrowers_employment_status = income_adj.income_source
        ),
        temp_aggregation8 AS (
            SELECT
                temp_tbl9.*,
                arrers_floor.ff_floor  
            FROM temp_aggregation7 AS temp_tbl9
            LEFT JOIN credit_risk_playground.stg_mrt_fitch_arrears_floor AS arrers_floor
            ON temp_tbl9.consecutive_months_in_arrears = arrers_floor.months_in_arrears 
        ),
        temp_aggregation9 AS (
            SELECT 
                *,
                CASE WHEN loan_term <= 24 THEN 0 ELSE baseline_ff END AS final_baseline_ff,
                CASE WHEN fitch_dti_class = 0 OR payment_type <> 6 THEN 1 ELSE io_adjustment END AS final_io_adjustment, 
                CASE 
                    WHEN account_status IN (2,3) AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 1
                    WHEN consecutive_months_in_arrears > 4 AND arrears_balance > 5 THEN 0.55
                    WHEN consecutive_months_in_arrears > 4 AND arrears_balance < 5 THEN 0
                    ELSE ff_floor 
                END AS final_ff_floor 
            FROM temp_aggregation8
        ),
        temp_aggregation10 AS (
            SELECT 
                tmptbl.*,
                CASE 
                    WHEN tmptbl.account_status IN (2,3) AND arrears_balance_ratio > 1 THEN 100
                    WHEN tmptbl.account_status = 4 THEN NULL
                    ELSE 100 * (1 - POWER((1 - tmptbl.adjusted_fitch_base_ff / 100), 0.1428571))
                END AS annualised_ff
            FROM (
                SELECT
                    *,
                    100 * GREATEST(final_baseline_ff * final_io_adjustment * income_adjustment, final_ff_floor) + 
                    (CASE WHEN energy_label IN ('A', 'B') THEN 0 ELSE 0.34 END) AS adjusted_fitch_base_ff
                FROM temp_aggregation9
            ) AS tmptbl
        ),
        temp_aggregation11 AS ( 
            SELECT  
                temp10.*,
                forcl_discount,
                forcl_variable_cost,
                cure_rate,
                pd_factor,
                indexed_vlaue_scalar
            FROM temp_aggregation10 AS temp10
            LEFT JOIN mev_plus_parameter_inputs AS mevs_input
            ON temp10.pool_cut_off_date = mevs_input.ref_date
        ),
        pd_pit_calculation AS (
            SELECT 
                temp11.*,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) END AS pd_ttc,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) * pd_factor END AS pd_pit
            FROM temp_aggregation11 AS temp11
        ),
        pd_pit_to_master_scale AS (
            SELECT 
                temp_hold2.*,
                CASE 
                    WHEN pd_pit < 0.01 THEN 1
                    WHEN pd_pit <= 0.04 THEN 2
                    WHEN pd_pit <= 0.09 THEN 3
                    WHEN pd_pit <= 0.15 THEN 4
                    WHEN pd_pit <= 0.23 THEN 5
                    WHEN pd_pit <= 0.38 THEN 6
                    WHEN pd_pit <= 0.73 THEN 7
                    WHEN pd_pit <= 1.42 THEN 8
                    WHEN pd_pit <= 2.25 THEN 9
                    WHEN pd_pit <= 3.21 THEN 10
                    WHEN pd_pit <= 4.29 THEN 11
                    WHEN pd_pit <= 5.61 THEN 12
                    WHEN pd_pit <= 11.18 THEN 13
                    WHEN pd_pit <= 27.89 THEN 14
                    WHEN pd_pit <= 46.00 THEN 15
                    WHEN pd_pit <= 60.86 THEN 16
                    WHEN pd_pit > 60.86 AND pd_pit <= 99.99 THEN 17
                    WHEN account_status = 3 THEN 18
                    WHEN annualised_ff = 100 THEN 18
                END AS pd_pit_class,
                (current_balance / customer_level_current_balance) * total_collateral_value AS collateral_value
            FROM (
                SELECT 
                    temp_hold1.*,
                    CASE 
                        WHEN pd_pit < 0.01 THEN 0.00
                        WHEN pd_pit <= 0.04 THEN 0.03
                        WHEN pd_pit <= 0.09 THEN 0.07
                        WHEN pd_pit <= 0.15 THEN 0.13
                        WHEN pd_pit <= 0.23 THEN 0.19
                        WHEN pd_pit <= 0.38 THEN 0.30
                        WHEN pd_pit <= 0.73 THEN 0.49
                        WHEN pd_pit <= 1.42 THEN 1.10
                        WHEN pd_pit <= 2.25 THEN 1.85
                        WHEN pd_pit <= 3.21 THEN 2.75
                        WHEN pd_pit <= 4.29 THEN 3.75
                        WHEN pd_pit <= 5.61 THEN 4.91
                        WHEN pd_pit <= 11.18 THEN 6.43
                        WHEN pd_pit <= 27.89 THEN 19.45
                        WHEN pd_pit <= 46.00 THEN 40
                        WHEN pd_pit <= 60.86 THEN 52.92
                        WHEN pd_pit > 60.86 AND pd_pit <= 99.99 THEN 70
                        WHEN account_status = 3 THEN 100
                        WHEN annualised_ff = 100 THEN 100
                    END AS pd_pit_mastered
                FROM pd_pit_calculation temp_hold1
            ) temp_hold2
        ),
        collateral_adjustement AS (
            SELECT  
                temp13.*,
                indexed_vlaue_scalar * collateral_value * (CASE WHEN energy_label IN ('A', 'B', 'C') THEN 1 ELSE (1 - 0.037) END) AS adjusted_collateral_value
            FROM pd_pit_to_master_scale AS temp13
        ),
        foreclosure_value_calculation AS (
            SELECT 
                temp14.*,
                GREATEST(guaranteed_foreclosure_value, adjusted_collateral_value * (1 - forcl_discount - forcl_variable_cost)) AS fitch_foreclosure_value
            FROM collateral_adjustement AS temp14
        ),
        lgd_calculation AS (
            SELECT 
                temp16.*,
                CASE WHEN loan_term <= 24 THEN 0 ELSE GREATEST(5, lgd_pit) END AS lgd
            FROM (
                SELECT 
                    temp15.*,
                    CASE 
                        WHEN loan_term <= 24 OR current_balance = 0 THEN 0
                        ELSE 100 * (1 - cure_rate) * GREATEST(0, 1 - (fitch_foreclosure_value / current_balance))
                    END AS lgd_pit
                FROM foreclosure_value_calculation AS temp15
            ) temp16
        ),
        ecl_calculation AS (
            SELECT 
                temp15.*,
                pd_pit_mastered / 100 * (lgd / 100) * current_balance AS expected_loss_1Year,
                (1 - POWER((1 - 0.003), LEAST(7, GREATEST(0, remaining_term)))) * original_balance AS performance_trigger, 
                province_code,
                pronvince_name
            FROM lgd_calculation temp15
            LEFT JOIN credit_risk_playground.stg_mrt_fitch_location_codes AS geo_code
            ON temp15.geographic_region_list = geo_code.geographical_code 
        )
        SELECT 
            pool_cut_off_date::DATE, borrower_identifier::VARCHAR, loan_identifier::VARCHAR, segment::VARCHAR, primary_income::float, secondary_income::float, total_income::float, borrowers_employment_status::INTEGER, loan_origination_date::DATE, date_of_loan_maturity::DATE, interest_revision_date::DATE, loan_term::INTEGER, purpose::VARCHAR, original_balance::float, current_balance::float, undrawn_loanpart::float, payment_type::SMALLINT, payment_due::float, debt_to_income::float, current_interest_rate::float, interest_rate_reset_interval::INTEGER, geographic_region_list::VARCHAR, original_loan_to_value::float, valuation_amount::float, current_loan_to_value::float, current_valuation_amount::float, property_value_at_time_of_latest_loan_advance::float, indexed_foreclosure_value::float, guaranteed_foreclosure_value::float, debt_service_coverage_ratio::float, account_status::SMALLINT, number_months_in_arrears::INTEGER, first_date_in_arrears::DATE, last_arrears_info_date::DATE, arrears_balance::float, consecutive_months_in_arrears::BIGINT, customer_level_original_balance::float, customer_level_current_balance::float, total_collateral_value::float, current_bridge_exposure::float, arrears_balance_ratio::float, current_debt_to_income_ratio::float, current_debt_to_income_ratio_excl_bridge::float, rwa::float, months_to_revision_date::BIGINT, remaining_term::BIGINT, next_month_interest_payment::float, one_month_paydown::float, prev_month_interest_payment::float, monthly_redemption::float, total_monthly_income::float, cust_level_prev_month_interest_payment::float, cust_level_monthly_redemption::float, total_monthly_debt_service::float, fitch_dti::float, fitch_ltv_class::INTEGER, fitch_term_class::INTEGER, fitch_dti_class::INTEGER, fitch_oltv_class::VARCHAR, fitch_cltv_class::VARCHAR, fitch_remaining_term_class::VARCHAR, fitch_oltv_term_class::VARCHAR, fitch_cltv_term_class::VARCHAR, baseline_ff::float, io_adjustment::float, income_adjustment::float, ff_floor::float, final_baseline_ff::float, final_io_adjustment::float, final_ff_floor::float, adjusted_fitch_base_ff::float, annualised_ff::float, forcl_discount::float, forcl_variable_cost::float, cure_rate::float, pd_factor::float, indexed_vlaue_scalar::float, pd_ttc::float, pd_pit::float, pd_pit_mastered::float, pd_pit_class::INTEGER, collateral_value::float, adjusted_collateral_value::float, fitch_foreclosure_value::float, lgd_pit::float, lgd::float, expected_loss_1year::float, performance_trigger::float, province_code::SMALLINT, pronvince_name::VARCHAR,
            GETDATE() AS etl_updated, 'alfred_teye' AS created_by
        FROM ecl_calculation 
        ORDER BY pool_cut_off_date, borrower_identifier, loan_identifier;
    """,

    "08_Rabo_Credit_Risk_Calculations_Scenarios": """
        INSERT INTO credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations_with_scenarios 
        WITH reporting_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        rabo_exposures AS (
            SELECT 
                pool_cut_off_date, borrower_identifier, loan_identifier,
                CASE WHEN type_of_guarantee_provider = 7 THEN 'NHG' ELSE 'Non-NHG' END AS segment,
                primary_income, secondary_income,
                CASE WHEN secondary_income IS NULL THEN primary_income ELSE primary_income + secondary_income END AS total_income,
                CASE WHEN borrowers_employment_status IS NULL THEN 10 ELSE borrowers_employment_status END AS borrowers_employment_status,			
                loan_origination_date, date_of_loan_maturity,
                TO_DATE(CONCAT(REPLACE(interest_revision_date_1, '-', '/'), '/01'), 'YYYY/MM/DD') AS interest_revision_date,
                loan_term, CASE WHEN loan_term <= 24 THEN 'Brigding' ELSE 'Standard' END AS purpose,
                original_balance, current_balance_adj AS current_balance,
                CASE WHEN retained_amount IS NULL THEN 0 ELSE retained_amount::float END AS undrawn_loanpart,
                payment_type, payment_due, debt_to_income, current_interest_rate, interest_rate_reset_interval, geographic_region_list, original_loan_to_value, valuation_amount, current_loan_to_value, current_valuation_amount, property_value_at_time_of_latest_loan_advance, indexed_foreclosure_value,
                CASE WHEN type_of_guarantee_provider = 7 THEN GREATEST(0, current_balance - (original_balance * 0.1)) ELSE 0 END AS guaranteed_foreclosure_value,
                debt_service_coverage_ratio,
                CASE WHEN arrears_balance = 0 THEN 1 ELSE account_status END AS account_status,
                CASE WHEN number_months_in_arrears IS NULL THEN 0 WHEN arrears_balance = 0 THEN 0 ELSE number_months_in_arrears END AS number_months_in_arrears,
                CASE WHEN arrears_balance = 0 THEN NULL ELSE LAST_DAY(DATEADD(Day, 15, date_last_current))::date END AS first_date_in_arrears,
                CASE 
                    WHEN date_last_in_arrears IS NULL AND date_last_current IS NULL THEN NULL
                    WHEN date_last_in_arrears IS NULL AND date_last_current IS NOT NULL THEN pool_cut_off_date::date
                    WHEN date_last_in_arrears IS NOT NULL THEN date_last_in_arrears::date
                END AS last_arrears_info_date,
                arrears_balance, property_type,
                CASE 
                    WHEN property_type IN (1, 3, 5, 6, 7) THEN 'house'
                    WHEN property_type = 2 THEN 'apartment'
                    WHEN property_type = 4 THEN 'terraced'
                    WHEN property_type IN (8,9) THEN 'commercial'
                    WHEN property_type = 10 THEN 'land'
                    ELSE 'other'
                END AS property_class,
                construction_year,
                CASE 
                    WHEN construction_year <= 1945 OR construction_year IS NULL THEN 1
                    WHEN construction_year <= 1964 THEN 2
                    WHEN construction_year <= 1974 THEN 3
                    WHEN construction_year <= 1982 THEN 4
                    WHEN construction_year <= 1987 THEN 5
                    WHEN construction_year <= 1991 THEN 6
                    WHEN construction_year <= 1999 THEN 7
                    WHEN construction_year <= 2005 THEN 8
                    ELSE 9
                END building_year,
                CASE 
                    WHEN building_year = 1 AND property_class IN ('house','apartment') THEN 'G'
                    WHEN building_year = 1 AND property_class IN ('terraced') THEN 'F'
                    WHEN building_year = 2 AND property_class IN ('house') THEN 'F'
                    WHEN building_year = 2 AND property_class IN ('apartment', 'terraced') THEN 'E'
                    WHEN building_year = 3 AND property_class IN ('house') THEN 'D'
                    WHEN building_year = 3 AND property_class IN ('apartment') THEN 'F'
                    WHEN building_year = 3 AND property_class IN ('terraced') THEN 'C'
                    WHEN building_year IN (4,5,6) THEN 'C'
                    WHEN building_year = 7 THEN 'B'
                    WHEN building_year = 8 AND property_class IN ('house','apartment') THEN 'B'
                    ELSE 'A'
                END AS energy_label
            FROM credit_risk_playground.stg_mrt_rabobank_raw_temp AS rabo
            INNER JOIN reporting_date rdate ON rabo.pool_cut_off_date = rdate.reporting_date
        ),
        rabo_customer_level_expo AS (
            SELECT pool_cut_off_date, borrower_identifier, SUM(original_balance) AS customer_level_original_balance, SUM(current_balance) AS customer_level_current_balance, SUM(current_valuation_amount) AS total_collateral_value
            FROM rabo_exposures GROUP BY pool_cut_off_date, borrower_identifier
        ),
        bridge_exposures AS (
            SELECT pool_cut_off_date, borrower_identifier, SUM(CASE WHEN current_balance IS NULL THEN 0 ELSE current_balance END) AS current_bridge_exposure
            FROM rabo_exposures WHERE loan_term <= 24 GROUP BY pool_cut_off_date, borrower_identifier
        ),
        mev_plus_parameter_inputs AS (
            SELECT temp_mev.*,
                EXP(pd_factor_constant + (pd_factor_ur_coeff * ur_diff_12m_down_scen) + (pd_factor_ir_coeff * ir_diff_12m_down_scen)) AS pd_factor_down_scen,
                EXP(pd_factor_constant + (pd_factor_ur_coeff * ur_diff_12m_neu_scen) + (pd_factor_ir_coeff * ir_diff_12m_neu_scen)) AS pd_factor_neu_scen,
                EXP(pd_factor_constant + (pd_factor_ur_coeff * ur_diff_12m_up_scen) + (pd_factor_ir_coeff * ir_diff_12m_up_scen)) AS pd_factor_up_scen,
                EXP(indexed_vlaue_scalar_const + (indexed_vlaue_scalar_rhpi_coeff * rhpi_log_diff_12m_lag_1_down_scen) + (indexed_vlaue_scalar_gdp_coeff * gdp_diff_12m_down_scen)) AS indexed_vlaue_scalar_down_scen,
                EXP(indexed_vlaue_scalar_const + (indexed_vlaue_scalar_rhpi_coeff * rhpi_log_diff_12m_lag_1_neu_scen) + (indexed_vlaue_scalar_gdp_coeff * gdp_diff_12m_neu_scen)) AS indexed_vlaue_scalar_neu_scen,
                EXP(indexed_vlaue_scalar_const + (indexed_vlaue_scalar_rhpi_coeff * rhpi_log_diff_12m_lag_1_up_scen) + (indexed_vlaue_scalar_gdp_coeff * gdp_diff_12m_up_scen)) AS indexed_vlaue_scalar_up_scen
            FROM (
                SELECT 
                    a.reference_date AS ref_date, a.ur_diff_12m_down_scen, a.ur_diff_12m_neu_scen, a.ur_diff_12m_up_scen, a.ir_diff_12m_down_scen, a.ir_diff_12m_neu_scen, a.ir_diff_12m_up_scen, a.gdp_diff_12m_down_scen, a.gdp_diff_12m_neu_scen, a.gdp_diff_12m_up_scen, a.rhpi_log_diff_12m_lag_1_down_scen, a.rhpi_log_diff_12m_lag_1_neu_scen, a.rhpi_log_diff_12m_lag_1_up_scen,
                    b.value AS cure_rate, c.coefficient AS pd_factor_constant, d.coefficient AS pd_factor_ur_coeff, e.coefficient AS pd_factor_ir_coeff, f.coefficient AS indexed_vlaue_scalar_const, g.coefficient AS indexed_vlaue_scalar_rhpi_coeff, h.coefficient AS indexed_vlaue_scalar_gdp_coeff, i.surcharge AS forcl_discount, j.surcharge AS forcl_variable_cost
                FROM credit_risk_playground.stg_mrt_ifrs9_mev_input_data AS a, credit_risk_playground.stg_mrt_portfolio_parameters AS b, credit_risk_playground.stg_mrt_risk_model_parameters AS c, credit_risk_playground.stg_mrt_risk_model_parameters AS d, credit_risk_playground.stg_mrt_risk_model_parameters AS e, credit_risk_playground.stg_mrt_risk_model_parameters AS f, credit_risk_playground.stg_mrt_risk_model_parameters AS g, credit_risk_playground.stg_mrt_risk_model_parameters AS h, credit_risk_playground.stg_mrt_fitch_foreclosure_discounts AS i, credit_risk_playground.stg_mrt_fitch_foreclosure_discounts AS j
                WHERE b.parameter_name = 'cure_rate'
                AND c.main_model = 'pd' AND c.sub_model = 'pd_factor' AND c.variable_name = 'constant'
                AND d.main_model = 'pd' AND d.sub_model = 'pd_factor' AND d.variable_name = 'diff_12m_ur'
                AND e.main_model = 'pd' AND e.sub_model = 'pd_factor' AND e.variable_name = 'diff_12m_ir'
                AND f.main_model = 'lgd' AND f.sub_model = 'indexed_value_scaler' AND f.variable_name = 'constant'
                AND g.main_model = 'lgd' AND g.sub_model = 'indexed_value_scaler' AND g.variable_name = 'log_diff_12m_rhpi_lag_1'
                AND h.main_model = 'lgd' AND h.sub_model = 'indexed_value_scaler' AND h.variable_name = 'diff_12m_gdp'
                AND i.discount = 'Foreclosed_Sale_Adjustment' AND j.discount = 'Variable_Foreclosure_Cost'
            ) temp_mev
        ),
        temp_aggregation AS (
            SELECT rabo_expo.*,
                CASE WHEN arrears_balance = 0 THEN 0 ELSE COALESCE(DATEDIFF('months', rabo_expo.first_date_in_arrears, rabo_expo.last_arrears_info_date) + 1, 0) END AS consecutive_months_in_arrears,
                cust_level_expo.customer_level_original_balance, cust_level_expo.customer_level_current_balance, cust_level_expo.total_collateral_value,
                CASE WHEN bridge.current_bridge_exposure IS NULL THEN 0 ELSE bridge.current_bridge_exposure END AS current_bridge_exposure 
            FROM rabo_exposures AS rabo_expo
            LEFT JOIN rabo_customer_level_expo AS cust_level_expo ON rabo_expo.pool_cut_off_date = cust_level_expo.pool_cut_off_date AND rabo_expo.borrower_identifier = cust_level_expo.borrower_identifier
            LEFT JOIN bridge_exposures AS bridge ON cust_level_expo.pool_cut_off_date = bridge.pool_cut_off_date AND cust_level_expo.borrower_identifier = bridge.borrower_identifier
        ),
        temp_aggregation1 AS (
            SELECT tmp_tbl.*, 100 * (arrears_balance / customer_level_current_balance) AS arrears_balance_ratio, customer_level_current_balance / total_income AS current_debt_to_income_ratio, (customer_level_current_balance - current_bridge_exposure) / total_income AS current_debt_to_income_ratio_excl_bridge,
                CASE WHEN segment = 'NHG' THEN 0.75 * 0.1 * current_balance ELSE 0.35 * current_balance + 0.4 * GREATEST(0, current_balance - 0.8 * current_valuation_amount) END AS rwa,
                DATEDIFF('month', pool_cut_off_date, LAST_DAY(interest_revision_date)) AS months_to_revision_date, DATEDIFF('year', pool_cut_off_date, date_of_loan_maturity) AS remaining_term, (current_balance - undrawn_loanpart) * ((current_interest_rate / 100) / 12) AS next_month_interest_payment
            FROM temp_aggregation AS tmp_tbl
        ),
        temp_aggregation1a AS (SELECT *, GREATEST(0, payment_due - next_month_interest_payment) AS one_month_paydown FROM temp_aggregation1),
        temp_aggregation2 AS (SELECT tmp_tbl1.*, (current_balance - undrawn_loanpart + one_month_paydown) * ((current_interest_rate / 100) / 12) AS prev_month_interest_payment, CASE WHEN loan_term <= 24 THEN 0 ELSE ROUND(original_balance / loan_term, 2) END AS monthly_redemption FROM temp_aggregation1a AS tmp_tbl1),
        temp_aggregation3 AS (
            SELECT tmp_tbl3.*, total_income / 12 AS total_monthly_income, cust_level_prev_month_interest_payment, cust_level_monthly_redemption, CASE WHEN loan_term <= 24 THEN 0 ELSE cust_level_monthly_redemption + cust_level_prev_month_interest_payment END AS total_monthly_debt_service
            FROM temp_aggregation2 AS tmp_tbl3 LEFT JOIN (SELECT pool_cut_off_date, borrower_identifier, SUM(prev_month_interest_payment) AS cust_level_prev_month_interest_payment, SUM(monthly_redemption) AS cust_level_monthly_redemption FROM temp_aggregation2 GROUP BY 1, 2) tmp_tbl4 ON tmp_tbl3.pool_cut_off_date = tmp_tbl4.pool_cut_off_date AND tmp_tbl3.borrower_identifier = tmp_tbl4.borrower_identifier
        ),
        temp_aggregation3a AS (SELECT *, CASE WHEN loan_term <= 24 OR current_balance = 0 THEN 0 ELSE 100 * (total_monthly_debt_service / total_monthly_income) END AS fitch_dti FROM temp_aggregation3),
        temp_aggregation4 AS (
            SELECT *,
                CASE WHEN current_loan_to_value IS NULL THEN NULL WHEN current_loan_to_value < 65 THEN 1 WHEN current_loan_to_value < 85 THEN 2 WHEN current_loan_to_value < 95 THEN 3 ELSE 4 END AS fitch_ltv_class,
                CASE WHEN months_to_revision_date IS NULL THEN NULL WHEN months_to_revision_date / 12 < 8 THEN 1 WHEN months_to_revision_date / 12 < 12 THEN 2 WHEN months_to_revision_date / 12 < 17 THEN 3 WHEN months_to_revision_date / 12 < 25 THEN 4 ELSE 5 END AS fitch_term_class,
                CASE WHEN loan_term IS NULL THEN NULL WHEN loan_term <= 24 THEN 0 WHEN fitch_dti < 20 THEN 1 WHEN fitch_dti < 30 THEN 2 WHEN fitch_dti < 40 THEN 3 WHEN fitch_dti < 50 THEN 4 ELSE 5 END AS fitch_dti_class,
                CASE WHEN original_loan_to_value IS NULL THEN NULL WHEN original_loan_to_value < 40 THEN '<40' WHEN original_loan_to_value < 50 THEN '40-50' WHEN original_loan_to_value < 60 THEN '50-60' WHEN original_loan_to_value < 65 THEN '60-65' WHEN original_loan_to_value < 70 THEN '65-70' WHEN original_loan_to_value < 75 THEN '70-75' WHEN original_loan_to_value < 80 THEN '75-80' WHEN original_loan_to_value < 85 THEN '80-85' WHEN original_loan_to_value < 90 THEN '85-90' WHEN original_loan_to_value < 95 THEN '90-95' WHEN original_loan_to_value < 100 THEN '95-100' WHEN original_loan_to_value < 105 THEN '100-105' WHEN original_loan_to_value < 110 THEN '105-110' ELSE '>=110' END AS fitch_oltv_class,
                CASE WHEN current_loan_to_value IS NULL THEN NULL WHEN current_loan_to_value < 40 THEN '<40' WHEN current_loan_to_value < 50 THEN '40-50' WHEN current_loan_to_value < 60 THEN '50-60' WHEN current_loan_to_value < 65 THEN '60-65' WHEN current_loan_to_value < 70 THEN '65-70' WHEN current_loan_to_value < 75 THEN '70-75' WHEN current_loan_to_value < 80 THEN '75-80' WHEN current_loan_to_value < 85 THEN '80-85' WHEN current_loan_to_value < 90 THEN '85-90' WHEN current_loan_to_value < 95 THEN '90-95' WHEN current_loan_to_value < 100 THEN '95-100' WHEN current_loan_to_value < 105 THEN '100-105' WHEN current_loan_to_value < 110 THEN '105-110' ELSE '>=110' END AS fitch_cltv_class,
                CASE WHEN remaining_term IS NULL THEN NULL WHEN remaining_term < 5 THEN '<5' WHEN remaining_term < 10 THEN '5-10' WHEN remaining_term < 15 THEN '10-15' WHEN remaining_term < 20 THEN '15-20' WHEN remaining_term < 25 THEN '20-25' ELSE '>=25' END AS fitch_remaining_term_class,
                CASE WHEN original_loan_to_value IS NULL THEN NULL WHEN original_loan_to_value < 50 THEN '<50' WHEN original_loan_to_value < 60 THEN '50-60' WHEN original_loan_to_value < 70 THEN '60-70' WHEN original_loan_to_value < 80 THEN '70-80' WHEN original_loan_to_value < 90 THEN '80-90' ELSE '>=90' END AS fitch_oltv_term_class,
                CASE WHEN current_loan_to_value IS NULL THEN NULL WHEN current_loan_to_value < 50 THEN '<50' WHEN current_loan_to_value < 60 THEN '50-60' WHEN current_loan_to_value < 70 THEN '60-70' WHEN current_loan_to_value < 80 THEN '70-80' WHEN current_loan_to_value < 90 THEN '80-90' ELSE '>=90' END AS fitch_cltv_term_class
            FROM temp_aggregation3a
        ),
        temp_aggregation5 AS (SELECT temp_tbl6.*, base_ff.baseline_ff FROM temp_aggregation4 AS temp_tbl6 LEFT JOIN credit_risk_playground.stg_mrt_fitch_ff_baseline AS base_ff ON temp_tbl6.fitch_cltv_class = base_ff.ltv_class AND temp_tbl6.fitch_dti_class = base_ff.dti_class),
        temp_aggregation6 AS (SELECT temp_tbl7.*, io_adjustment.io_adjustment FROM temp_aggregation5 AS temp_tbl7 LEFT JOIN credit_risk_playground.stg_mrt_fitch_ff_io_adjustment AS io_adjustment ON temp_tbl7.fitch_oltv_term_class = io_adjustment.ltv_class AND temp_tbl7.fitch_remaining_term_class = io_adjustment.remaining_term_class),
        temp_aggregation7 AS (SELECT temp_tbl8.*, income_adj.income_adjustment FROM temp_aggregation6 AS temp_tbl8 LEFT JOIN credit_risk_playground.stg_mrt_fitch_income_adjustment AS income_adj ON temp_tbl8.borrowers_employment_status = income_adj.income_source),
        temp_aggregation8 AS (SELECT temp_tbl9.*, arrers_floor.ff_floor FROM temp_aggregation7 AS temp_tbl9 LEFT JOIN credit_risk_playground.stg_mrt_fitch_arrears_floor AS arrers_floor ON temp_tbl9.consecutive_months_in_arrears = arrers_floor.months_in_arrears),
        temp_aggregation9 AS (
            SELECT *,
                CASE WHEN loan_term <= 24 THEN 0 ELSE baseline_ff END AS final_baseline_ff,
                CASE WHEN fitch_dti_class = 0 OR payment_type <> 6 THEN 1 ELSE io_adjustment END AS final_io_adjustment,
                CASE WHEN account_status IN (2,3) AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 1 WHEN consecutive_months_in_arrears > 4 AND arrears_balance > 5 THEN 0.55 WHEN consecutive_months_in_arrears > 4 AND arrears_balance < 5 THEN 0 ELSE ff_floor END AS final_ff_floor
            FROM temp_aggregation8
        ),
        temp_aggregation10 AS (
            SELECT tmptbl.*,
                CASE WHEN tmptbl.account_status IN (2,3) AND arrears_balance_ratio > 1 THEN 100 WHEN tmptbl.account_status = 4 THEN NULL ELSE 100 * (1 - POWER((1 - CASE WHEN tmptbl.adjusted_fitch_base_ff IS NULL THEN 100 * final_baseline_ff ELSE tmptbl.adjusted_fitch_base_ff END / 100), 0.1428571)) END AS annualised_ff
            FROM (SELECT *, 100 * GREATEST(final_baseline_ff * final_io_adjustment * income_adjustment, final_ff_floor) + (CASE WHEN energy_label IN ('A', 'B') THEN 0 ELSE 0.34 END) AS adjusted_fitch_base_ff FROM temp_aggregation9) AS tmptbl
        ),
        temp_aggregation11 AS (
            SELECT temp10.*, forcl_discount, forcl_variable_cost, cure_rate, pd_factor_down_scen, pd_factor_neu_scen, pd_factor_up_scen, indexed_vlaue_scalar_down_scen, indexed_vlaue_scalar_neu_scen, indexed_vlaue_scalar_up_scen
            FROM temp_aggregation10 AS temp10 LEFT JOIN mev_plus_parameter_inputs AS mevs_input ON temp10.pool_cut_off_date = mevs_input.ref_date
        ),
        pd_pit_scenarios AS (
            SELECT temp11.*,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) END AS pd_ttc,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) * pd_factor_down_scen END AS pd_pit_down_scen,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) * pd_factor_neu_scen END AS pd_pit_neu_scen,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) * pd_factor_up_scen END AS pd_pit_up_scen
            FROM temp_aggregation11 AS temp11
        ),
        pd_pit_scenario_weighted AS (
            SELECT *, pd_pit_down_scen * 0.0225 + pd_pit_neu_scen * 0.955 + pd_pit_up_scen * 0.0225 AS pd_pit FROM pd_pit_scenarios
        ),
        pd_pit_to_master_scale AS (
            SELECT temp_hold2.*,
                CASE WHEN pd_pit < 0.01 THEN 1 WHEN pd_pit <= 0.04 THEN 2 WHEN pd_pit <= 0.09 THEN 3 WHEN pd_pit <= 0.15 THEN 4 WHEN pd_pit <= 0.23 THEN 5 WHEN pd_pit <= 0.38 THEN 6 WHEN pd_pit <= 0.73 THEN 7 WHEN pd_pit <= 1.42 THEN 8 WHEN pd_pit <= 2.25 THEN 9 WHEN pd_pit <= 3.21 THEN 10 WHEN pd_pit <= 4.29 THEN 11 WHEN pd_pit <= 5.61 THEN 12 WHEN pd_pit <= 11.18 THEN 13 WHEN pd_pit <= 27.89 THEN 14 WHEN pd_pit <= 46.00 THEN 15 WHEN pd_pit <= 60.86 THEN 16 WHEN pd_pit > 60.86 AND pd_pit <= 99.99 THEN 17 WHEN account_status = 3 THEN 18 WHEN annualised_ff = 100 THEN 18 END AS pd_pit_class,
                (current_balance / customer_level_current_balance) * total_collateral_value * (CASE WHEN energy_label IN ('A', 'B', 'C') THEN 1 ELSE (1 - 0.037) END) AS collateral_value
            FROM (
                SELECT temp_hold1.*,
                    CASE WHEN pd_pit < 0.01 THEN 0.00 WHEN pd_pit <= 0.04 THEN 0.03 WHEN pd_pit <= 0.09 THEN 0.07 WHEN pd_pit <= 0.15 THEN 0.13 WHEN pd_pit <= 0.23 THEN 0.19 WHEN pd_pit <= 0.38 THEN 0.30 WHEN pd_pit <= 0.73 THEN 0.49 WHEN pd_pit <= 1.42 THEN 1.10 WHEN pd_pit <= 2.25 THEN 1.85 WHEN pd_pit <= 3.21 THEN 2.75 WHEN pd_pit <= 4.29 THEN 3.75 WHEN pd_pit <= 5.61 THEN 4.91 WHEN pd_pit <= 11.18 THEN 6.43 WHEN pd_pit <= 27.89 THEN 19.45 WHEN pd_pit <= 46.00 THEN 40 WHEN pd_pit <= 60.86 THEN 52.92 WHEN pd_pit > 60.86 AND pd_pit <= 99.99 THEN 70 WHEN account_status = 3 THEN 100 WHEN annualised_ff = 100 THEN 100 END AS pd_pit_mastered,
                    CASE WHEN pd_pit_neu_scen < 0.01 THEN 0.00 WHEN pd_pit_neu_scen <= 0.04 THEN 0.03 WHEN pd_pit_neu_scen <= 0.09 THEN 0.07 WHEN pd_pit_neu_scen <= 0.15 THEN 0.13 WHEN pd_pit_neu_scen <= 0.23 THEN 0.19 WHEN pd_pit_neu_scen <= 0.38 THEN 0.30 WHEN pd_pit_neu_scen <= 0.73 THEN 0.49 WHEN pd_pit_neu_scen <= 1.42 THEN 1.10 WHEN pd_pit_neu_scen <= 2.25 THEN 1.85 WHEN pd_pit_neu_scen <= 3.21 THEN 2.75 WHEN pd_pit_neu_scen <= 4.29 THEN 3.75 WHEN pd_pit_neu_scen <= 5.61 THEN 4.91 WHEN pd_pit_neu_scen <= 11.18 THEN 6.43 WHEN pd_pit_neu_scen <= 27.89 THEN 19.45 WHEN pd_pit_neu_scen <= 46.00 THEN 40 WHEN pd_pit_neu_scen <= 60.86 THEN 52.92 WHEN pd_pit_neu_scen > 60.86 AND pd_pit <= 99.99 THEN 70 WHEN account_status = 3 THEN 100 WHEN annualised_ff = 100 THEN 100 END AS pd_pit_mastered_neu_scen,
                    CASE WHEN pd_pit_up_scen < 0.01 THEN 0.00 WHEN pd_pit_up_scen <= 0.04 THEN 0.03 WHEN pd_pit_up_scen <= 0.09 THEN 0.07 WHEN pd_pit_up_scen <= 0.15 THEN 0.13 WHEN pd_pit_up_scen <= 0.23 THEN 0.19 WHEN pd_pit_up_scen <= 0.38 THEN 0.30 WHEN pd_pit_up_scen <= 0.73 THEN 0.49 WHEN pd_pit_up_scen <= 1.42 THEN 1.10 WHEN pd_pit_up_scen <= 2.25 THEN 1.85 WHEN pd_pit_up_scen <= 3.21 THEN 2.75 WHEN pd_pit_up_scen <= 4.29 THEN 3.75 WHEN pd_pit_up_scen <= 5.61 THEN 4.91 WHEN pd_pit_up_scen <= 11.18 THEN 6.43 WHEN pd_pit_up_scen <= 27.89 THEN 19.45 WHEN pd_pit_up_scen <= 46.00 THEN 40 WHEN pd_pit_up_scen <= 60.86 THEN 52.92 WHEN pd_pit_up_scen > 60.86 AND pd_pit <= 99.99 THEN 70 WHEN account_status = 3 THEN 100 WHEN annualised_ff = 100 THEN 100 END AS pd_pit_mastered_up_scen,
                    CASE WHEN pd_pit_down_scen < 0.01 THEN 0.00 WHEN pd_pit_down_scen <= 0.04 THEN 0.03 WHEN pd_pit_down_scen <= 0.09 THEN 0.07 WHEN pd_pit_down_scen <= 0.15 THEN 0.13 WHEN pd_pit_down_scen <= 0.23 THEN 0.19 WHEN pd_pit_down_scen <= 0.38 THEN 0.30 WHEN pd_pit_down_scen <= 0.73 THEN 0.49 WHEN pd_pit_down_scen <= 1.42 THEN 1.10 WHEN pd_pit_down_scen <= 2.25 THEN 1.85 WHEN pd_pit_down_scen <= 3.21 THEN 2.75 WHEN pd_pit_down_scen <= 4.29 THEN 3.75 WHEN pd_pit_down_scen <= 5.61 THEN 4.91 WHEN pd_pit_down_scen <= 11.18 THEN 6.43 WHEN pd_pit_down_scen <= 27.89 THEN 19.45 WHEN pd_pit_down_scen <= 46.00 THEN 40 WHEN pd_pit_down_scen <= 60.86 THEN 52.92 WHEN pd_pit_down_scen > 60.86 AND pd_pit <= 99.99 THEN 70 WHEN account_status = 3 THEN 100 WHEN annualised_ff = 100 THEN 100 END AS pd_pit_mastered_down_scen
                FROM pd_pit_scenario_weighted AS temp_hold1
            ) AS temp_hold2
        ),
        collateral_adjustement AS (
            SELECT temp13.*, indexed_vlaue_scalar_down_scen * collateral_value AS adjusted_collateral_value_down_scen, indexed_vlaue_scalar_neu_scen * collateral_value AS adjusted_collateral_value_neu_scen, indexed_vlaue_scalar_up_scen * collateral_value AS adjusted_collateral_value_up_scen
            FROM pd_pit_to_master_scale AS temp13
        ),
        foreclosure_value_calculation AS (
            SELECT temp14.*, GREATEST(guaranteed_foreclosure_value, adjusted_collateral_value_down_scen * (1 - forcl_discount - forcl_variable_cost)) AS fitch_foreclosure_value_down_scen, GREATEST(guaranteed_foreclosure_value, adjusted_collateral_value_neu_scen * (1 - forcl_discount - forcl_variable_cost)) AS fitch_foreclosure_value_neu_scen, GREATEST(guaranteed_foreclosure_value, adjusted_collateral_value_up_scen * (1 - forcl_discount - forcl_variable_cost)) AS fitch_foreclosure_value_up_scen
            FROM collateral_adjustement AS temp14
        ),
        lgd_calculation AS (
            SELECT temp16.*, CASE WHEN loan_term <= 24 THEN 0 ELSE GREATEST(5, lgd_pit_down_scen) END AS lgd_down_scen, CASE WHEN loan_term <= 24 THEN 0 ELSE GREATEST(5, lgd_pit_neu_scen) END AS lgd_neu_scen, CASE WHEN loan_term <= 24 THEN 0 ELSE GREATEST(5, lgd_pit_up_scen) END AS lgd_up_scen	 
            FROM (
                SELECT temp15.*, CASE WHEN loan_term <= 24 OR current_balance = 0 THEN 0 ELSE 100 * (1 - cure_rate) * GREATEST(0, 1 - (fitch_foreclosure_value_down_scen / current_balance)) END AS lgd_pit_down_scen, CASE WHEN loan_term <= 24 OR current_balance = 0 THEN 0 ELSE 100 * (1 - cure_rate) * GREATEST(0, 1 - (fitch_foreclosure_value_neu_scen / current_balance)) END AS lgd_pit_neu_scen, CASE WHEN loan_term <= 24 OR current_balance = 0 THEN 0 ELSE 100 * (1 - cure_rate) * GREATEST(0, 1 - (fitch_foreclosure_value_up_scen / current_balance)) END AS lgd_pit_up_scen
                FROM foreclosure_value_calculation AS temp15
            ) temp16
        ),
        ecl_calculation AS (
            SELECT *, lgd_down_scen * 0.0225 + lgd_neu_scen * 0.955 + lgd_up_scen * 0.0225 AS lgd_weighted, pd_pit_mastered_down_scen / 100 * (lgd_down_scen / 100) * current_balance AS ecl_down_scen, pd_pit_mastered_neu_scen / 100 * (lgd_neu_scen / 100) * current_balance AS ecl_neu_scen, pd_pit_mastered_up_scen / 100 * (lgd_up_scen / 100) * current_balance AS ecl_up_scen
            FROM lgd_calculation
        ),
        final_tbl AS (
            SELECT temp17.*, ecl_down_scen * 0.0225 + ecl_neu_scen * 0.955 + ecl_up_scen * 0.0225 AS expected_loss_1year, (1 - POWER((1 - 0.003), LEAST(7, GREATEST(0, remaining_term)))) * original_balance AS performance_trigger, province_code, pronvince_name, GETDATE() AS etl_updated, 'alfred_teye'::text AS created_by
            FROM ecl_calculation AS temp17 LEFT JOIN credit_risk_playground.stg_mrt_fitch_location_codes AS geo_code ON temp17.geographic_region_list = geo_code.geographical_code 
        )
        SELECT pool_cut_off_date::DATE, borrower_identifier::VARCHAR, loan_identifier::VARCHAR, segment::VARCHAR, primary_income::FLOAT, secondary_income::FLOAT, total_income::FLOAT, borrowers_employment_status::INTEGER, loan_origination_date::DATE, date_of_loan_maturity::DATE, interest_revision_date::DATE, loan_term::INTEGER, purpose::VARCHAR, original_balance::FLOAT, current_balance::FLOAT, undrawn_loanpart::FLOAT, payment_type::SMALLINT, payment_due::FLOAT, debt_to_income::FLOAT, current_interest_rate::FLOAT, interest_rate_reset_interval::INTEGER, geographic_region_list::VARCHAR, original_loan_to_value::FLOAT, valuation_amount::FLOAT, current_loan_to_value::FLOAT, current_valuation_amount::FLOAT, property_value_at_time_of_latest_loan_advance::FLOAT, indexed_foreclosure_value::FLOAT, guaranteed_foreclosure_value::FLOAT, debt_service_coverage_ratio::FLOAT, account_status::SMALLINT, number_months_in_arrears::INTEGER, first_date_in_arrears::DATE, last_arrears_info_date::DATE, arrears_balance::FLOAT, consecutive_months_in_arrears::INT, customer_level_original_balance::FLOAT, customer_level_current_balance::FLOAT, total_collateral_value::FLOAT, current_bridge_exposure::FLOAT, arrears_balance_ratio::FLOAT, current_debt_to_income_ratio::FLOAT, current_debt_to_income_ratio_excl_bridge::FLOAT, rwa::FLOAT, months_to_revision_date::INT, remaining_term::INT, next_month_interest_payment::FLOAT, one_month_paydown::FLOAT, prev_month_interest_payment::FLOAT, monthly_redemption::FLOAT, total_monthly_income::FLOAT, cust_level_prev_month_interest_payment::FLOAT, cust_level_monthly_redemption::FLOAT, total_monthly_debt_service::FLOAT, fitch_dti::FLOAT, fitch_ltv_class::INTEGER, fitch_term_class::INTEGER, fitch_dti_class::INTEGER, fitch_oltv_class::VARCHAR, fitch_cltv_class::VARCHAR, fitch_remaining_term_class::VARCHAR, fitch_oltv_term_class::VARCHAR, fitch_cltv_term_class::VARCHAR, baseline_ff::FLOAT, io_adjustment::FLOAT, income_adjustment::FLOAT, ff_floor::FLOAT, final_baseline_ff::FLOAT, final_io_adjustment::FLOAT, final_ff_floor::FLOAT, adjusted_fitch_base_ff::FLOAT, annualised_ff::FLOAT, forcl_discount::FLOAT, forcl_variable_cost::FLOAT, cure_rate::FLOAT, pd_factor_down_scen::FLOAT, pd_factor_neu_scen::FLOAT, pd_factor_up_scen::FLOAT, indexed_vlaue_scalar_down_scen::FLOAT, indexed_vlaue_scalar_neu_scen::FLOAT, indexed_vlaue_scalar_up_scen::FLOAT, pd_ttc::FLOAT, pd_pit_down_scen::FLOAT, pd_pit_neu_scen::FLOAT, pd_pit_up_scen::FLOAT, pd_pit::FLOAT, pd_pit_mastered::FLOAT, pd_pit_mastered_neu_scen::FLOAT, pd_pit_mastered_up_scen::FLOAT, pd_pit_mastered_down_scen::FLOAT, pd_pit_class::INTEGER, collateral_value::FLOAT, adjusted_collateral_value_down_scen::FLOAT, adjusted_collateral_value_neu_scen::FLOAT, adjusted_collateral_value_up_scen::FLOAT, fitch_foreclosure_value_down_scen::FLOAT, fitch_foreclosure_value_neu_scen::FLOAT, fitch_foreclosure_value_up_scen::FLOAT, lgd_pit_down_scen::FLOAT, lgd_pit_neu_scen::FLOAT, lgd_pit_up_scen::FLOAT, lgd_down_scen::FLOAT, lgd_neu_scen::FLOAT, lgd_up_scen::FLOAT, lgd_weighted::FLOAT, ecl_down_scen::FLOAT, ecl_neu_scen::FLOAT, ecl_up_scen::FLOAT, expected_loss_1year::FLOAT, performance_trigger::FLOAT, province_code::INT, pronvince_name::VARCHAR, GETDATE() AS etl_updated, 'alfred_teye'::text AS created_by
        FROM final_tbl;
    """,

    "09_Rabo_Portfolio_Stats": """
        INSERT INTO credit_risk_playground.stg_mrt_rabobank_portfolio_stats
        WITH reporting_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        credit_risk_calculations AS (
            SELECT risk_cal.* 
            FROM credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations risk_cal
            INNER JOIN reporting_date rep_date
            ON risk_cal.pool_cut_off_date = rep_date.reporting_date
            AND current_balance > 0
        ),
        exposure_overview AS (
            SELECT 
                pool_cut_off_date,
                COUNT(DISTINCT borrower_identifier) AS customers,
                ROUND(SUM(current_balance), 0) AS total_principal,
                ROUND(SUM(undrawn_loanpart), 0) AS undrawn_amounts,
                ROUND(SUM(CASE WHEN purpose = 'Brigding' THEN current_balance END)) AS bridge_loans,
                ROUND(MAX(customer_level_current_balance)) AS largest_exposure,
                ROUND(SUM(DATEDIFF('year', pool_cut_off_date, LAST_DAY(interest_revision_date)) * current_balance) / SUM(current_balance), 2) AS wa_reset_period_years,
                ROUND(SUM(rwa), 0) AS rwa,
                ROUND(100 * SUM(rwa) / SUM(current_balance), 2) rwa_pct,
                ROUND(SUM(current_balance * current_loan_to_value) / SUM(current_balance), 2) AS wa_cltv_pct,
                ROUND(SUM(debt_service_coverage_ratio * current_balance) / SUM(current_balance), 2) AS wa_dscr,
                ROUND(SUM(current_interest_rate * current_balance) / SUM(current_balance) - 100 * value, 2) AS wa_coupon_net_servicing_cost_pct,
                ROUND(SUM(current_interest_rate * current_balance) / SUM(current_balance), 2) AS gross_coupon_pct,
                ROUND(SUM(expected_loss_1year), 0) AS ecl_one_year,
                ROUND(100 * ROUND(SUM(expected_loss_1year), 2) / SUM(current_balance), 2) AS ecl_one_year_pct,
                ROUND(SUM(lgd * current_balance) / SUM(current_balance), 2) AS wa_lgd_pct
            FROM credit_risk_calculations, credit_risk_playground.stg_mrt_portfolio_parameters 
            WHERE parameter_name = 'servicing_cost'
            GROUP BY pool_cut_off_date, value
            ORDER BY pool_cut_off_date
        ),
        sub_portfolio_overview AS (
            SELECT 
                pool_cut_off_date, segment,
                COUNT(DISTINCT borrower_identifier) AS customers,
                ROUND(SUM(current_balance)) AS total_principal,
                ROUND(SUM(undrawn_loanpart)) AS undrawn_amounts,
                ROUND(SUM(CASE WHEN purpose = 'Brigding' THEN current_balance END)) AS bridge_loans,
                ROUND(MAX(customer_level_current_balance)) AS largest_exposure,
                ROUND(SUM(DATEDIFF('year', pool_cut_off_date, LAST_DAY(interest_revision_date)) * current_balance) / SUM(current_balance), 2) AS wa_reset_period_years,
                ROUND(SUM(rwa), 0) AS rwa,
                ROUND(100 * SUM(rwa) / SUM(current_balance), 2) rwa_pct,
                ROUND(SUM(current_balance * current_loan_to_value) / SUM(current_balance), 2) AS wa_cltv_pct,
                ROUND(SUM(debt_service_coverage_ratio * current_balance) / SUM(current_balance), 2) AS wa_dscr,
                ROUND(SUM(current_interest_rate * current_balance) / SUM(current_balance) - 100 * value, 2) AS wa_coupon_net_servicing_cost_pct,
                ROUND(SUM(current_interest_rate * current_balance) / SUM(current_balance), 2) AS gross_coupon_pct,
                ROUND(SUM(expected_loss_1year), 0) AS ecl_one_year,
                ROUND(100 * ROUND(SUM(expected_loss_1year), 2) / SUM(current_balance), 2) AS ecl_one_year_pct,
                ROUND(SUM(lgd * current_balance) / SUM(current_balance), 2) AS wa_lgd_pct
            FROM credit_risk_calculations, credit_risk_playground.stg_mrt_portfolio_parameters 
            WHERE parameter_name = 'servicing_cost'
            GROUP BY pool_cut_off_date, segment, value
            ORDER BY pool_cut_off_date
        ),
        pd_overview AS (
            SELECT pool_cut_off_date, ROUND(SUM(pd_pit_mastered * current_balance) / SUM(current_balance), 2) wa_pd_pct
            FROM credit_risk_calculations WHERE current_balance > 0 AND pd_pit_mastered < 100
            GROUP BY pool_cut_off_date ORDER BY pool_cut_off_date
        ),
        pd_sub_portfolio_overview AS (
            SELECT pool_cut_off_date, segment, ROUND(SUM(pd_pit_mastered * current_balance) / SUM(current_balance), 2) wa_pd_pct
            FROM credit_risk_calculations WHERE current_balance > 0 AND pd_pit_mastered < 100
            GROUP BY pool_cut_off_date, segment ORDER BY pool_cut_off_date
        ),
        aggregate_stats AS (
            (SELECT a.pool_cut_off_date, 'total' AS segment, a.customers, a.total_principal, a.undrawn_amounts, a.bridge_loans, a.largest_exposure, a.wa_reset_period_years, a.rwa, a.rwa_pct, a.wa_cltv_pct, a.wa_dscr, a.wa_coupon_net_servicing_cost_pct, gross_coupon_pct, a.ecl_one_year, a.ecl_one_year_pct, a.wa_lgd_pct, b.wa_pd_pct
             FROM exposure_overview a INNER JOIN pd_overview b ON a.pool_cut_off_date = b.pool_cut_off_date ORDER BY pool_cut_off_date)
            UNION 
            (SELECT c.*, d.wa_pd_pct FROM sub_portfolio_overview c INNER JOIN pd_sub_portfolio_overview d ON c.pool_cut_off_date = d.pool_cut_off_date AND c.segment = d.segment ORDER BY pool_cut_off_date)
            ORDER BY pool_cut_off_date, segment
        ),
        capitalisation AS (
            SELECT stat.*, ROUND(rwa * value) AS required_capital, ROUND(rwa_pct * value, 2) AS required_capital_pct
            FROM aggregate_stats AS stat, credit_risk_playground.stg_mrt_portfolio_parameters 
            WHERE parameter_name = 'required_equity'
        )
        SELECT cap.*, ROUND(100 * (gross_coupon_pct - 100 * value) / required_capital_pct, 2) AS roe_pct, GETDATE() AS etl_updated, 'alfred_teye'::text AS created_by
        FROM capitalisation AS cap, credit_risk_playground.stg_mrt_portfolio_parameters 
        WHERE parameter_name = 'fixed_cost'
        ORDER BY pool_cut_off_date ASC, segment DESC;
    """,

    "10_Rabo_IFRS9_ECL_Reporting": """
        INSERT INTO credit_risk_playground.stg_mrt_rabobank_ifrs9_ecl_reporting
        WITH RECURSIVE horizon(next_period) AS (
            SELECT 0
            UNION ALL
            SELECT next_period + 1 FROM horizon WHERE next_period < 36
        ),
        reporting_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        base_calculations AS (
            SELECT 
                risk_cal.pool_cut_off_date AS reporting_date, risk_cal.borrower_identifier, risk_cal.loan_identifier, risk_cal.segment, risk_cal.purpose, risk_cal.current_balance, risk_cal.customer_level_current_balance, risk_cal.loan_origination_date, risk_cal.date_of_loan_maturity, risk_cal.loan_term, risk_cal.remaining_term, COALESCE(risk_cal.current_interest_rate, non_null_rate) AS current_interest_rate, risk_cal.payment_type, risk_cal.payment_due, risk_cal.arrears_balance, risk_cal.number_months_in_arrears, risk_cal.account_status, risk_cal.consecutive_months_in_arrears AS mpd, risk_cal.pd_pit_mastered AS one_year_pd, risk_cal.pd_pit_class AS rating_class, lgd, expected_loss_1Year
            FROM credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations AS risk_cal
            LEFT JOIN (
                SELECT borrower_identifier, loan_identifier, current_interest_rate AS non_null_rate, ROW_NUMBER() OVER(PARTITION BY borrower_identifier, loan_identifier ORDER BY pool_cut_off_date) AS rankings
                FROM credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations WHERE current_interest_rate IS NOT NULL
            ) AS init_rates
            ON risk_cal.borrower_identifier = init_rates.borrower_identifier AND risk_cal.loan_identifier = init_rates.loan_identifier
            WHERE current_balance > 0 AND rankings = 1
        ),
        initial_recognition_values AS (
            SELECT a.*, b.initial_pd
            FROM (SELECT MIN(reporting_date) AS first_date_in_portfolio, borrower_identifier, loan_identifier FROM base_calculations GROUP BY borrower_identifier, loan_identifier) a
            INNER JOIN (SELECT reporting_date, borrower_identifier, loan_identifier, one_year_pd AS initial_pd FROM base_calculations) b 
            ON a.first_date_in_portfolio = b.reporting_date AND a.borrower_identifier = b.borrower_identifier AND a.loan_identifier = b.loan_identifier
        ),
        merged_data AS (
            SELECT DISTINCT base.*, init.first_date_in_portfolio, init.initial_pd, ROW_NUMBER() OVER(PARTITION BY base.borrower_identifier, base.loan_identifier, base.reporting_date ORDER BY base.borrower_identifier, base.loan_identifier, base.reporting_date) AS duplicates
            FROM base_calculations AS base LEFT JOIN initial_recognition_values AS init ON base.borrower_identifier = init.borrower_identifier AND base.loan_identifier = init.loan_identifier
        ),
        main_data AS (
            SELECT md.reporting_date, borrower_identifier, loan_identifier, segment, purpose, current_balance, customer_level_current_balance, loan_origination_date, first_date_in_portfolio, date_of_loan_maturity, loan_term, payment_type, payment_due, remaining_term, current_interest_rate, arrears_balance, number_months_in_arrears, account_status, mpd, initial_pd, one_year_pd, rating_class, lgd, expected_loss_1Year,
                CASE WHEN initial_pd = 0 THEN 0 ELSE 100 * (1 - POWER(1 - initial_pd / 100, remaining_term)) END AS initial_ltpd,
                CASE WHEN one_year_pd = 0 THEN 0 ELSE 100 * (1 - POWER(1 - one_year_pd / 100, remaining_term)) END AS current_ltpd
            FROM merged_data AS md INNER JOIN reporting_date AS r_date ON md.reporting_date = r_date.reporting_date WHERE duplicates = 1
        ),
        ifrs_staging1 AS ( 
            SELECT *,
                CASE WHEN one_year_pd = 100 THEN 3 WHEN mpd >= 2 AND one_year_pd < 100 AND arrears_balance > 0 THEN 2 WHEN current_ltpd > 3 * initial_ltpd THEN 2 WHEN rating_class IN ('15','16','17') THEN 2 ELSE 1 END AS stage,
                current_balance AS ead
            FROM main_data
        ),
        ifrs_staging AS (SELECT *, MAX(stage) OVER(PARTITION BY reporting_date, borrower_identifier) AS ifrs_stage FROM ifrs_staging1),
        ecl_calculation AS (
            SELECT *,
                CASE WHEN ifrs_stage = 1 THEN (one_year_pd / 100) * (lgd / 100) * ead WHEN ifrs_stage = 3 AND current_ltpd = 100 THEN (lgd / 100) * ead END AS ecl
            FROM ifrs_staging 
        ),
        cashflow_projection1 AS (
            SELECT reporting_date, borrower_identifier, loan_identifier, remaining_term, payment_type, payment_due, current_interest_rate, ifrs_stage, one_year_pd, lgd, ead, next_period AS future_period
            FROM ecl_calculation CROSS JOIN horizon WHERE ifrs_stage = 2
        ),
        cashflow_projection2 AS (
            SELECT *,
                100 * (POWER(1 - one_year_pd / 100, future_period) - POWER(1 - one_year_pd / 100, future_period + 1)) AS pd_future_period,
                CASE 
                    WHEN payment_type = 1 THEN (CASE WHEN future_period = 0 THEN ead ELSE ROUND(ead * (POWER(1 + current_interest_rate / 1200, remaining_term) - POWER(1 + current_interest_rate / 1200, LEAST(future_period, remaining_term))) / (POWER(1 + current_interest_rate / 1200, remaining_term) - 1), 2) END)
                    WHEN payment_type = 2 THEN ROUND(GREATEST(ead - (future_period * ead / remaining_term), 0), 2)
                    WHEN payment_type = 6 THEN (CASE WHEN future_period <= remaining_term THEN ead ELSE 0 END)
                END ead_future_period,
                POWER(1 + current_interest_rate / 100, -future_period) AS discount_factor
            FROM cashflow_projection1
        ),
        temp_lifetime_ecl AS (
            SELECT *, (pd_future_period / 100) * (lgd / 100) * ead_future_period AS ecl_future_period, (pd_future_period / 100) * (lgd / 100) * ead_future_period * discount_factor AS ecl_future_period_discounted
            FROM cashflow_projection2
        ),
        lifetime_ecl AS (
            SELECT reporting_date, borrower_identifier, loan_identifier, ROUND(SUM(ecl_future_period_discounted), 2) AS lifetime_ecl
            FROM temp_lifetime_ecl GROUP BY reporting_date, borrower_identifier, loan_identifier, ifrs_stage
        ),
        final_ecl_calculation AS (
            SELECT a.*, ROUND(COALESCE(a.ecl, b.lifetime_ecl), 2) AS expected_credit_loss
            FROM ecl_calculation a LEFT JOIN lifetime_ecl b ON a.reporting_date = b.reporting_date AND a.borrower_identifier = b.borrower_identifier AND a.loan_identifier = b.loan_identifier
        )
        SELECT *, GETDATE() AS etl_updated, 'alfred_teye'::text AS created_by
        FROM final_ecl_calculation;
    """,

    "11_Rabo_IFRS9_ECL_Reporting_Scenarios": """
        INSERT INTO credit_risk_playground.stg_mrt_rabobank_ifrs9_ecl_reporting_with_scenarios
        WITH RECURSIVE horizon(next_period) AS (
            SELECT 0
            UNION ALL
            SELECT next_period + 1 FROM horizon WHERE next_period < 36
        ),
        report_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        base_calculations AS (
            SELECT 
                risk_cal.pool_cut_off_date AS reporting_date, risk_cal.borrower_identifier, risk_cal.loan_identifier, risk_cal.segment, risk_cal.purpose, risk_cal.current_balance, risk_cal.customer_level_current_balance, risk_cal.undrawn_loanpart, risk_cal.loan_origination_date, risk_cal.date_of_loan_maturity, risk_cal.loan_term, risk_cal.remaining_term, COALESCE(risk_cal.current_interest_rate, non_null_rate) AS current_interest_rate, risk_cal.payment_type, risk_cal.payment_due, risk_cal.arrears_balance, risk_cal.number_months_in_arrears, risk_cal.account_status, risk_cal.consecutive_months_in_arrears AS mpd, risk_cal.pd_pit_mastered AS one_year_pd, risk_cal.pd_pit_class AS rating_class, risk_cal.pd_pit_mastered_down_scen, risk_cal.pd_pit_mastered_neu_scen, risk_cal.pd_pit_mastered_up_scen, risk_cal.lgd_down_scen, risk_cal.lgd_neu_scen, risk_cal.lgd_up_scen, risk_cal.lgd_weighted, risk_cal.expected_loss_1year
            FROM credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations_with_scenarios AS risk_cal
            LEFT JOIN (
                SELECT borrower_identifier, loan_identifier, current_interest_rate AS non_null_rate, ROW_NUMBER() OVER(PARTITION BY borrower_identifier, loan_identifier ORDER BY pool_cut_off_date) AS rankings
                FROM credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations_with_scenarios WHERE current_interest_rate IS NOT NULL
            ) AS init_rates
            ON risk_cal.borrower_identifier = init_rates.borrower_identifier AND risk_cal.loan_identifier = init_rates.loan_identifier
            WHERE rankings = 1
        ),
        initial_recognition_values AS (
            SELECT a.*, b.initial_pd
            FROM (SELECT MIN(pool_cut_off_date) AS first_date_in_portfolio, borrower_identifier, loan_identifier FROM credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations GROUP BY borrower_identifier, loan_identifier) a
            INNER JOIN (SELECT pool_cut_off_date AS reporting_date, borrower_identifier, loan_identifier, pd_pit_mastered AS initial_pd FROM credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations) b 
            ON a.first_date_in_portfolio = b.reporting_date AND a.borrower_identifier = b.borrower_identifier AND a.loan_identifier = b.loan_identifier
        ),
        merged_data AS (
            SELECT init_values.first_date_in_portfolio, base.*,
                CASE WHEN base.reporting_date::date > init_values.first_date_in_portfolio::date THEN init_values.initial_pd ELSE one_year_pd END initial_pd,
                ROW_NUMBER() OVER(PARTITION BY base.borrower_identifier, base.loan_identifier, base.reporting_date ORDER BY base.borrower_identifier, base.loan_identifier, base.reporting_date) AS duplicates
            FROM base_calculations AS base LEFT JOIN initial_recognition_values AS init_values USING(borrower_identifier, loan_identifier)
        ),
        main_data AS (
            SELECT md.reporting_date, borrower_identifier, loan_identifier, segment, purpose, current_balance, undrawn_loanpart, customer_level_current_balance, loan_origination_date, first_date_in_portfolio, date_of_loan_maturity, loan_term, payment_type, payment_due, remaining_term, current_interest_rate, arrears_balance, number_months_in_arrears, account_status, mpd, initial_pd, one_year_pd, rating_class, pd_pit_mastered_down_scen, pd_pit_mastered_neu_scen, pd_pit_mastered_up_scen, lgd_down_scen, lgd_neu_scen, lgd_up_scen, lgd_weighted, expected_loss_1year,
                CASE WHEN initial_pd = 0 THEN 0 ELSE 100 * (1 - POWER(1 - initial_pd / 100, remaining_term)) END AS initial_ltpd,
                CASE WHEN one_year_pd = 0 THEN 0 ELSE 100 * (1 - POWER(1 - one_year_pd / 100, remaining_term)) END AS current_ltpd
            FROM merged_data AS md INNER JOIN report_date AS r_date USING(reporting_date) WHERE duplicates = 1
        ),
        ifrs_staging1 AS ( 
            SELECT *,
                CASE WHEN one_year_pd = 100 THEN 3 WHEN mpd >= 2 AND one_year_pd < 100 AND arrears_balance > 0 THEN 2 WHEN current_ltpd > 3 * initial_ltpd THEN 2 WHEN rating_class IN ('15','16','17') THEN 2 ELSE 1 END AS stage,
                current_balance AS ead
            FROM main_data
        ),
        ifrs_staging AS (SELECT *, MAX(stage) OVER(PARTITION BY reporting_date, borrower_identifier) AS ifrs_stage FROM ifrs_staging1),
        ecl_calculation AS (
            SELECT *,
                CASE WHEN ifrs_stage = 1 THEN (one_year_pd / 100) * (lgd_down_scen / 100) * ead WHEN ifrs_stage = 3 AND current_ltpd = 100 THEN (lgd_down_scen / 100) * ead END AS ecl_down_scen,
                CASE WHEN ifrs_stage = 1 THEN (one_year_pd / 100) * (lgd_neu_scen / 100) * ead WHEN ifrs_stage = 3 AND current_ltpd = 100 THEN (lgd_neu_scen / 100) * ead END AS ecl_neu_scen,
                CASE WHEN ifrs_stage = 1 THEN (one_year_pd / 100) * (lgd_up_scen / 100) * ead WHEN ifrs_stage = 3 AND current_ltpd = 100 THEN (lgd_up_scen / 100) * ead END AS ecl_up_scen
            FROM ifrs_staging 
        ),
        cashflow_projection1 AS (
            SELECT reporting_date, borrower_identifier, loan_identifier, remaining_term, payment_type, payment_due, current_interest_rate, ifrs_stage, one_year_pd, lgd_down_scen, lgd_neu_scen, lgd_up_scen, ead, next_period AS future_period
            FROM ecl_calculation CROSS JOIN horizon WHERE ifrs_stage = 2
        ),
        cashflow_projection2 AS (
            SELECT *,
                100 * (POWER(1 - one_year_pd / 100, future_period) - POWER(1 - one_year_pd / 100, future_period + 1)) AS pd_future_period,
                CASE 
                    WHEN payment_type = 1 THEN (CASE WHEN future_period = 0 THEN ead ELSE ROUND(ead * (POWER(1 + current_interest_rate / 1200, remaining_term) - POWER(1 + current_interest_rate / 1200, LEAST(future_period, remaining_term))) / (POWER(1 + current_interest_rate / 1200, remaining_term) - 1), 2) END)
                    WHEN payment_type = 2 THEN ROUND(GREATEST(ead - (future_period * ead / remaining_term), 0), 2)
                    WHEN payment_type = 6 THEN (CASE WHEN future_period <= remaining_term THEN ead ELSE 0 END)
                END ead_future_period,
                POWER(1 + current_interest_rate / 100, -future_period) AS discount_factor
            FROM cashflow_projection1
        ),
        temp_lifetime_ecl AS (
            SELECT *,
                (pd_future_period / 100) * (lgd_neu_scen / 100) * ead_future_period AS ecl_future_period_neu_scen,
                (pd_future_period / 100) * (lgd_down_scen / 100) * ead_future_period AS ecl_future_period_down_scen,
                (pd_future_period / 100) * (lgd_up_scen / 100) * ead_future_period AS ecl_future_period_up_scen,
                (pd_future_period / 100) * (lgd_neu_scen / 100) * ead_future_period * discount_factor AS ecl_future_period_discounted_neu_scen,
                (pd_future_period / 100) * (lgd_down_scen / 100) * ead_future_period * discount_factor AS ecl_future_period_discounted_down_scen,
                (pd_future_period / 100) * (lgd_up_scen / 100) * ead_future_period * discount_factor AS ecl_future_period_discounted_up_scen
            FROM cashflow_projection2
        ),
        lifetime_ecl AS (
            SELECT reporting_date, borrower_identifier, loan_identifier, ROUND(SUM(ecl_future_period_discounted_neu_scen * 0.955 + ecl_future_period_discounted_down_scen * 0.0225 + ecl_future_period_discounted_up_scen * 0.0225), 2) AS lifetime_ecl
            FROM temp_lifetime_ecl GROUP BY reporting_date, borrower_identifier, loan_identifier, ifrs_stage
        ),
        final_ecl_calculation AS (
            SELECT a.*, ROUND(COALESCE(a.ecl_down_scen * 0.0225 + a.ecl_neu_scen * 0.955 + a.ecl_up_scen * 0.0225, b.lifetime_ecl), 2) AS expected_credit_loss
            FROM ecl_calculation a LEFT JOIN lifetime_ecl b USING(reporting_date, borrower_identifier, loan_identifier)
        )
        SELECT *, GETDATE() AS etl_updated, 'alfred_teye'::text AS created_by
        FROM final_ecl_calculation
        ORDER BY borrower_identifier, loan_identifier, ifrs_stage;
    """
}

print(f" Registered {len(QUERIES)} Rabobank queries in execution order:")
for step_num, name in enumerate(QUERIES.keys(), 1):
    print(f"  Step {step_num}: {name}")

 Registered 8 Rabobank queries in execution order:
  Step 1: 04_Rabo_Raw
  Step 2: 05_Rabo_Raw_Temp
  Step 3: 06_Rabo_Arrears_List
  Step 4: 07_Rabo_Credit_Risk_Calculations
  Step 5: 08_Rabo_Credit_Risk_Calculations_Scenarios
  Step 6: 09_Rabo_Portfolio_Stats
  Step 7: 10_Rabo_IFRS9_ECL_Reporting
  Step 8: 11_Rabo_IFRS9_ECL_Reporting_Scenarios


In [7]:
# Cell 3: Embedded SQL Pipeline Registry (Folder 3: Neo)

QUERIES = {
    "12_Neo_ESME_Raw": """
        INSERT INTO credit_risk_playground.stg_mrt_neo_esme_raw
        WITH report_date AS (
            SELECT TO_CHAR('{{ end_date }}'::date, 'YYYY_MM')::text AS rep_dt
        ),
        esme_loan_data AS (
            SELECT 
                loan_identifier_number, loan_identifier_part, loan_identifier_old, loan_identifier, borrower_identifier_old, borrower_identifier, pool_additional_date, date_of_repurchase, redemption_date, resident, geo_region_borrower, geo_region_classification, borrowers_employment_status, credit_impaired_borrower, borrower_type, primary_income, primary_income_currency, primary_income_type, income_verification_for_primary_income, secondary_income, secondary_income_currency, income_verification_for_secondary_income, subsidy_received, loan_origination_date, date_of_loan_maturity, loan_term, origination_channel_arranging_cank_or_division, purpose, original_balance, original_balance_currency, current_balance, current_balance_currency, prior_balances, prior_balances_currency, pari_passu_loans, pari_passu_loans_currency, maximum_balance, maximum_balance_currency, purchase_price, payment_type, principal_grace_period, principal_payment_frequency, interest_payment_frequency, payment_due, payment_due_currency, debt_to_income, balloon_amount, balloon_amount_currency, interest_rate_type, current_interest_rate, current_interest_rate_index, current_interest_rate_index_tenor, current_interest_rate_margin, interest_rate_reset_interval, interest_cap_rate, interest_rate_floor, revision_margin_1, interest_revision_date_1, revision_margin_2, interest_revision_date_2, revision_margin_3, interest_revision_date_3, revised_interest_rate_index, revised_interest_rate_index_tenor, num_of_payments_before_securitisation, percentage_of_pre_payments_allowed_per_year, prepayment_lock_out_end_date, pre_payment_penalties, pre_payment_penalties_currency, pre_payment_fee_end_date, pre_payment_date, cumulative_pre_payments, cumulative_pre_payments_currency, date_of_restructuring, date_last_in_arrears, arrears_balance, arrears_balance_currency, number_months_in_arrears, account_status, reason_for_default_or_foreclosure, default_or_foreclosure, default_or_foreclosure_currency, date_of_default_or_foreclosure, loss_on_sale, loss_on_sale_currency, cumulative_recoveries, cumulative_recoveries_currency, litigation, recourse, deposit_amount, deposit_amount_currency, additional_collateral_provider, original_lender_name, original_lender_legal_entity_identifier, original_ledner_establishment_country, originator_name, originator_legal_entity_identifier, originator_establishment_country, batch_id, etl_source_file, etl_updated
            FROM residential_real_estate_loan_nl AS loans
            INNER JOIN report_date AS dt
            ON SUBSTRING(loans.etl_source_file, 42, 7)::text = dt.rep_dt
        ),
        esme_customer_data AS (
            SELECT  
                loan_identifier_number, loan_identifier_part, loan_identifier, property_identifier_old, property_identifier, collateral_type, geographic_region_list, occupancy_type, lien, property_type, energy_performance_certificate_value, energy_performance_certificate_provider_name, current_loan_to_value, current_valuation_amount, current_valuation_amount_currency, current_valuation_type, current_valuation_date, original_loan_to_value, valuation_amount, valuation_amount_currency, original_valuation_type, valuation_date, date_of_sale, sale_price_lower_limit, sale_price_lower_limit_currency, type_of_guarantee_provider, etl_source_file AS etl_source_file1
            FROM residential_real_estate_customer_nl AS customers
            INNER JOIN report_date AS dt 
            ON SUBSTRING(customers.etl_source_file, 42, 7)::text = dt.rep_dt
        ),
        temp_tbl AS (
            SELECT 
                LAST_DAY(CONCAT(REPLACE(SUBSTRING(rel.etl_source_file, 42, 7), '_', '-'), '-28')::date) AS pool_cut_off_date,
                SUBSTRING(rel.etl_source_file, 38, 3) AS pool_identifier,
                rel.*,
                rec.property_identifier_old, rec.property_identifier, rec.collateral_type, rec.geographic_region_list, rec.occupancy_type, rec.lien, rec.property_type, rec.energy_performance_certificate_value, rec.energy_performance_certificate_provider_name, rec.current_loan_to_value, rec.current_valuation_amount, rec.current_valuation_amount_currency, rec.current_valuation_type, rec.current_valuation_date, rec.original_loan_to_value, rec.valuation_amount, rec.valuation_amount_currency, rec.original_valuation_type, rec.valuation_date, rec.date_of_sale, rec.sale_price_lower_limit, rec.sale_price_lower_limit_currency, rec.type_of_guarantee_provider
            FROM esme_loan_data AS rel
            INNER JOIN esme_customer_data AS rec
            ON rel.loan_identifier = rec.loan_identifier
            AND SUBSTRING(rel.etl_source_file, 33, 11) = SUBSTRING(rec.etl_source_file1, 33, 11)
        )
        SELECT * FROM temp_tbl;
    """,

    "13_Neo_ESMA_To_ECB_Raw": """
        INSERT INTO credit_risk_playground.stg_mrt_neo_esma_to_ecb_raw
        WITH reporting_date AS (
            SELECT '{{ end_date }}'::date AS pool_cut_off_date
        ),
        esma_source_data AS (
            SELECT
                pool_cut_off_date, pool_identifier, loan_identifier, NULL AS regulated_loan, original_lender_name AS originator, 'N26 Hypotheken B.V.' AS servicer_identifier, borrower_identifier, property_identifier, NULL AS borrower_type, NULL AS foreign_national, NULL AS borrower_credit_quality, NULL AS borrower_year_of_birth, NULL AS number_of_debtors, NULL AS second_applicant_year_of_birth,
                CASE
                    WHEN borrowers_employment_status = 'EMRS' THEN '1' WHEN borrowers_employment_status = 'EMBL' THEN '2' WHEN borrowers_employment_status = 'EMUK' THEN '3' WHEN borrowers_employment_status = 'UNEM' THEN '4' WHEN borrowers_employment_status = 'SFEM' THEN '5' WHEN borrowers_employment_status = 'NOEM' THEN '6' WHEN borrowers_employment_status = 'STNT' THEN '7' WHEN borrowers_employment_status = 'PNNR' THEN '8' WHEN borrowers_employment_status = 'OTHR' THEN '9' WHEN borrowers_employment_status IS NULL OR borrowers_employment_status = 'ND5' THEN NULL
                END AS borrowers_employment_status,
                NULL AS first_time_buyer, NULL AS right_to_buy, NULL AS right_to_buy_price, NULL AS class_of_borrower, CONVERT(decimal(10,2), primary_income) AS primary_income,
                CASE
                    WHEN income_verification_for_primary_income = 'SCRT' THEN '1' WHEN income_verification_for_primary_income = 'SCNF' THEN '2' WHEN income_verification_for_primary_income = 'VRFD' THEN '3' WHEN income_verification_for_primary_income = 'NVRF' THEN '4' WHEN income_verification_for_primary_income = 'SCRG' THEN '6' WHEN income_verification_for_primary_income = 'OTHR' THEN '5' WHEN income_verification_for_primary_income IS NULL OR income_verification_for_primary_income = 'ND5' THEN NULL
                END AS income_verification_for_primary_income,
                CASE WHEN secondary_income != 'ND5' THEN CONVERT(decimal(10,2), secondary_income) ELSE NULL END AS secondary_income,
                CASE
                    WHEN income_verification_for_secondary_income = 'SCRT' THEN '1' WHEN income_verification_for_secondary_income = 'SCNF' THEN '2' WHEN income_verification_for_secondary_income = 'VRFD' THEN '3' WHEN income_verification_for_secondary_income = 'NVRF' THEN '4' WHEN income_verification_for_secondary_income = 'SCRG' THEN '6' WHEN income_verification_for_secondary_income = 'OTHR' THEN '5' WHEN income_verification_for_secondary_income IS NULL OR income_verification_for_secondary_income = 'ND5' THEN NULL
                END AS income_verification_for_secondary_income,
                NULL AS resident, NULL AS number_of_county_court_judgements_or_equivalent_satisfied, NULL AS value_of_county_court_judgements_or_equivalent_satisfied, NULL AS number_of_county_court_judgements_or_equivalent_unsatisfied, NULL AS value_of_county_court_judgements_or_equivalent_unsatisfied, NULL AS last_county_court_judgements_or_equivalent_year, NULL AS bankruptcy_or_individual_voluntary_arrangement_flag, NULL AS prior_repossessions, NULL AS previous_mortgage_arrears_6_months, NULL AS previous_mortgage_arrears_over_6_months,
                CASE SUBSTRING(loan_origination_date, 6, 2)
                    WHEN '01' THEN 'January' WHEN '02' THEN 'February' WHEN '03' THEN 'March' WHEN '04' THEN 'April' WHEN '05' THEN 'May' WHEN '06' THEN 'June' WHEN '07' THEN 'July' WHEN '08' THEN 'August' WHEN '09' THEN 'September' WHEN '10' THEN 'October' WHEN '11' THEN 'November' WHEN '12' THEN 'December'
                END || ' ' || CAST(CAST(SUBSTRING(loan_origination_date, 9, 2) AS INTEGER) AS TEXT) || ', ' || SUBSTRING(loan_origination_date, 1, 4) AS loan_origination_date,
                CASE SUBSTRING(date_of_loan_maturity, 6, 2)
                    WHEN '01' THEN 'January' WHEN '02' THEN 'February' WHEN '03' THEN 'March' WHEN '04' THEN 'April' WHEN '05' THEN 'May' WHEN '06' THEN 'June' WHEN '07' THEN 'July' WHEN '08' THEN 'August' WHEN '09' THEN 'September' WHEN '10' THEN 'October' WHEN '11' THEN 'November' WHEN '12' THEN 'December'
                END || ' ' || CAST(CAST(SUBSTRING(date_of_loan_maturity, 9, 2) AS INTEGER) AS TEXT) || ', ' || SUBSTRING(date_of_loan_maturity, 1, 4) AS date_of_loan_maturity,
                NULL AS account_status_date,
                CASE
                    WHEN origination_channel_arranging_cank_or_division = 'BRAN' THEN '1' WHEN origination_channel_arranging_cank_or_division = 'DRCT' THEN '2' WHEN origination_channel_arranging_cank_or_division = 'BROK' THEN '3' WHEN origination_channel_arranging_cank_or_division = 'WEBI' THEN '4' WHEN origination_channel_arranging_cank_or_division = 'TPAC' THEN '5' WHEN origination_channel_arranging_cank_or_division = 'TPTC' THEN '6' WHEN origination_channel_arranging_cank_or_division = 'OTHR' THEN '7' WHEN origination_channel_arranging_cank_or_division IS NULL OR origination_channel_arranging_cank_or_division = 'ND5' THEN NULL
                END AS origination_channel_arranging_cank_or_division,
                CASE
                    WHEN purpose = 'PURC' THEN '1' WHEN purpose = 'RMRT' THEN '2' WHEN purpose = 'RENV' THEN '3' WHEN purpose = 'EQRE' THEN '4' WHEN purpose = 'CNST' THEN '5' WHEN purpose = 'DCON' THEN '6' WHEN purpose = 'OTHR' THEN '7' WHEN purpose = 'RMEQ' THEN '8' WHEN purpose = 'CMRT' THEN '10' WHEN purpose = 'IMRT' THEN '11' WHEN purpose = 'RGBY' THEN '12' WHEN purpose = 'GSPL' THEN '13' WHEN purpose IS NULL OR purpose = 'ND5' THEN NULL
                END AS purpose,
                NULL AS shared_ownership, CAST(loan_term AS INTEGER) AS loan_term, NULL AS principal_grace_period, NULL AS amount_guaranteed, NULL AS subsidy, 'EUR' AS loan_currency_denomination, CONVERT(decimal(10, 2), original_balance) AS original_balance, CAST(current_balance AS NUMERIC(20, 2)) - CAST(deposit_amount AS NUMERIC(20, 2)) AS current_balance, NULL AS fractioned_subrogated_loans,
                CASE WHEN payment_type = 'BLLT' THEN '1' ELSE '2' END AS repayment_method,
                CASE
                    WHEN principal_payment_frequency = 'MNTH' THEN '1' WHEN principal_payment_frequency = 'QUTR' THEN '2' WHEN principal_payment_frequency = 'SEMI' THEN '3' WHEN principal_payment_frequency = 'YEAR' THEN '4' WHEN principal_payment_frequency = 'OTHR' THEN '6' WHEN principal_payment_frequency IS NULL OR principal_payment_frequency = 'ND5' THEN NULL
                END AS payment_frequency,
                CASE WHEN payment_due = 'ND5' THEN NULL ELSE CONVERT(decimal(10, 2), payment_due) END AS payment_due,
                CASE WHEN payment_type = 'FRXX' THEN '1' WHEN payment_type = 'FIXE' THEN '2' WHEN payment_type = 'BLLT' THEN '6' ELSE NULL END AS payment_type,
                debt_to_income,
                CASE
                    WHEN type_of_guarantee_provider = 'NGUA' THEN '1' WHEN type_of_guarantee_provider = 'FAML' THEN '2' WHEN type_of_guarantee_provider = 'IOTH' THEN '3' WHEN type_of_guarantee_provider = 'GOVE' THEN '4' WHEN type_of_guarantee_provider = 'BANK' THEN '5' WHEN type_of_guarantee_provider = 'INSU' THEN '6' WHEN type_of_guarantee_provider = 'NHGX' THEN '7' WHEN type_of_guarantee_provider = 'FGAS' THEN '8' WHEN type_of_guarantee_provider = 'CATN' THEN '9' WHEN type_of_guarantee_provider = 'OTHR' THEN '10' WHEN (type_of_guarantee_provider IS NULL OR type_of_guarantee_provider = 'ND5') THEN NULL
                END AS type_of_guarantee_provider,
                NULL AS guarantee_provider, NULL AS guarantor_income, CASE WHEN subsidy_received <> 'ND5' THEN subsidy_received ELSE NULL END AS subsidy_received, NULL AS mortgage_indemnity_guarantee_provider, NULL AS mortgage_indemnity_guarantee_attachment_point, NULL AS prior_balances, NULL AS other_prior_balances, NULL AS pari_passu_loans, NULL AS subordinated_claims, NULL AS lien, deposit_amount AS retained_amount, 'ND,5' AS retained_amount_date, NULL AS maximum_balance, NULL AS further_loan_advance, NULL AS flexible_loan_amount, NULL AS further_advances, NULL AS length_of_payment_holiday, NULL AS subsidy_period, NULL AS mortgage_inscription, NULL AS deed_of_postponement, NULL AS pre_payment_amount, NULL AS pre_payment_penalties, CONVERT(decimal(10, 2), cumulative_pre_payments) AS cumulative_pre_payments, percentage_of_pre_payments_allowed_per_year,
                CASE
                    WHEN interest_rate_type = 'FLIF' THEN '1' WHEN interest_rate_type = 'FINX' THEN '2' WHEN interest_rate_type = 'FXRL' THEN '3' WHEN interest_rate_type = 'FXPR' THEN '4' WHEN interest_rate_type = 'FLCF' THEN '5' WHEN interest_rate_type = 'CAPP' THEN '6' WHEN interest_rate_type = 'OTHR' THEN '8' WHEN interest_rate_type = 'FLFL' THEN '9' WHEN interest_rate_type = 'FLCA' THEN '10' WHEN interest_rate_type IS NULL OR interest_rate_type = 'ND5' THEN NULL
                END AS interest_rate_type,
                '12' AS current_interest_rate_index, CASE WHEN current_interest_rate <> 'ND5' THEN CONVERT(decimal(10, 2), current_interest_rate) ELSE NULL END AS current_interest_rate,
                CASE
                    WHEN interest_rate_type IN ('FXRL', 'FXPR', 'FLCF') AND current_interest_rate <> 'ND5' THEN current_interest_rate WHEN interest_rate_type IN ('FXRL', 'FXPR', 'FLCF') AND current_interest_rate = 'ND5' THEN '0' WHEN interest_rate_type IN ('FLIF', 'FINX', 'CAPP', 'FLFL', 'FLCA') THEN current_interest_rate_margin WHEN current_interest_rate_margin IS NULL OR current_interest_rate_margin = 'ND5' THEN NULL
                END AS current_interest_rate_margin,
                CAST(interest_rate_reset_interval AS INTEGER) AS interest_rate_reset_interval, NULL AS interest_cap_rate, NULL AS revision_margin_1, CASE WHEN interest_revision_date_1 <> 'ND5' THEN SUBSTRING(interest_revision_date_1,1,7) ELSE NULL END AS interest_revision_date_1, NULL AS revision_margin_2, NULL AS interest_revision_date_2, NULL AS revision_margin_3, NULL AS interest_revision_date_3, NULL AS revised_interest_rate_index, NULL AS final_margin, NULL AS restructuring_arrangement, 'NLZZZ' AS geographic_region_list, NULL AS property_postcode,
                CASE
                    WHEN occupancy_type = 'FOWN' THEN '1' WHEN occupancy_type = 'POWN' THEN '2' WHEN occupancy_type = 'TLET' THEN '3' WHEN occupancy_type = 'HOLD' THEN '4' WHEN occupancy_type = 'OTHR' THEN '5' WHEN occupancy_type = 'CAPP' THEN '6' WHEN occupancy_type IS NULL OR occupancy_type = 'ND5' THEN NULL
                END AS occupancy_type,
                CASE
                    WHEN property_type = 'RHOS' THEN '1' WHEN property_type = 'RFLT' THEN '2' WHEN property_type = 'RBGL' THEN '3' WHEN property_type = 'RTHS' THEN '4' WHEN property_type = 'MULF' THEN '5' WHEN property_type = 'PCMM' THEN '7' WHEN property_type = 'BIZZ' THEN '8' WHEN property_type = 'LAND' THEN '10' WHEN property_type = 'OTHR' THEN '11' WHEN property_type IS NULL OR property_type = 'ND5' THEN NULL
                END AS property_type,
                NULL AS new_property, NULL AS construction_year, NULL AS property_rating, CONVERT(decimal(10, 2), original_loan_to_value) AS original_loan_to_value, CONVERT(decimal(10, 2), valuation_amount) AS valuation_amount,
                CASE
                    WHEN original_valuation_type = 'FIEI' THEN '1' WHEN original_valuation_type = 'FOEI' THEN '2' WHEN original_valuation_type = 'DRVB' THEN '3' WHEN original_valuation_type = 'AUVM' THEN '4' WHEN original_valuation_type = 'IDXD' THEN '5' WHEN original_valuation_type = 'DKTP' THEN '6' WHEN original_valuation_type = 'MAEA' THEN '7' WHEN original_valuation_type = 'TXAT' THEN '8' WHEN original_valuation_type = 'OTHR' THEN '9' WHEN original_valuation_type IS NULL OR original_valuation_type = 'ND5' THEN NULL
                END AS original_valuation_type,
                CASE WHEN valuation_date <> 'ND5' THEN SUBSTRING(valuation_date,1,7) ELSE 'ND,5' END AS valuation_date, NULL AS confidence_interval_for_original_automated_valuation_model_valuation, NULL AS provider_of_original_automated_valuation_model_valuation, CONVERT(decimal(10, 2), current_loan_to_value) AS current_loan_to_value, NULL AS purchase_price_lower_limit, CONVERT(decimal(10, 2), current_valuation_amount) AS current_valuation_amount,
                CASE
                    WHEN current_valuation_type = 'FIEI' THEN '1' WHEN current_valuation_type = 'FOEI' THEN '2' WHEN current_valuation_type = 'DRVB' THEN '3' WHEN current_valuation_type = 'AUVM' THEN '4' WHEN current_valuation_type = 'IDXD' THEN '5' WHEN current_valuation_type = 'DKTP' THEN '6' WHEN current_valuation_type = 'MAEA' THEN '7' WHEN current_valuation_type = 'TXAT' THEN '8' WHEN current_valuation_type = 'OTHR' THEN '9' WHEN current_valuation_type IS NULL OR current_valuation_type = 'ND5' THEN NULL
                END AS current_valuation_type,
                CASE WHEN current_valuation_date <> 'ND5' THEN SUBSTRING(current_valuation_date,1,7) ELSE 'ND,5' END AS current_valuation_date, NULL AS confidence_interval_for_current_automated_valuation_model_valuation, NULL AS provider_of_current_automated_valuation_model_valuation, NULL AS property_value_at_time_of_latest_loan_advance, NULL AS indexed_foreclosure_value, NULL AS ipoteca, NULL AS additional_collateral, NULL AS additional_collateral_provider, NULL AS gross_annual_rental_income, NULL AS number_of_buy_to_let_properties, NULL AS debt_service_coverage_ratio, NULL AS additional_collateral_value, NULL AS real_estate_owned, NULL AS is_property_transferability_limited, NULL AS time_until_declassification,
                CASE
                    WHEN account_status IN ('PERF','RNAR') THEN '1' WHEN account_status IN ('ARRE','RARR') THEN '2' WHEN account_status IN ('DFLT','NDFT','DTCR','DADB') THEN '3' WHEN account_status = 'RDMD' THEN '4' WHEN account_status IN ('REBR','REDF', 'RERE', 'RESS', 'REOT') THEN '5' WHEN account_status = 'OTHR' THEN '6' WHEN account_status IS NULL OR account_status = 'ND5' THEN NULL
                END AS account_status,
                NULL AS date_last_current, CASE WHEN date_last_in_arrears <> 'ND5' THEN date_last_in_arrears ELSE NULL END AS date_last_in_arrears, CONVERT(decimal(10, 2), arrears_balance) AS arrears_balance, CAST(number_months_in_arrears AS INTEGER) AS number_months_in_arrears,
                LAG(arrears_balance, 1) OVER (PARTITION BY loan_identifier ORDER BY pool_cut_off_date, pool_identifier) AS arrears_1_month_ago,
                LAG(arrears_balance, 2) OVER (PARTITION BY loan_identifier ORDER BY pool_cut_off_date, pool_identifier) AS arrears_2_months_ago,
                NULL AS performance_arrangement, NULL AS litigation,
                CASE SUBSTRING(redemption_date, 6, 2)
                    WHEN '01' THEN 'January' WHEN '02' THEN 'February' WHEN '03' THEN 'March' WHEN '04' THEN 'April' WHEN '05' THEN 'May' WHEN '06' THEN 'June' WHEN '07' THEN 'July' WHEN '08' THEN 'August' WHEN '09' THEN 'September' WHEN '10' THEN 'October' WHEN '11' THEN 'November' WHEN '12' THEN 'December'
                END || ' ' || CAST(CAST(SUBSTRING(redemption_date, 9, 2) AS INTEGER) AS TEXT) || ', ' || SUBSTRING(redemption_date, 1, 4) AS redemption_date,
                NULL AS months_in_arrears_prior, NULL AS default_or_foreclosure, NULL AS sale_price_lower_limit, NULL AS loss_on_sale, NULL AS cumulative_recoveries, NULL AS professional_negligence_recoveries, NULL AS loan_flagged_as_contencioso
            FROM credit_risk_playground.stg_mrt_neo_esme_raw
            WHERE 1=1 AND pool_additional_date <> 'ND5'
        )
        SELECT 
            a.*, GETDATE() AS etl_updated, USER::text AS created_by   
        FROM esma_source_data AS a
        INNER JOIN reporting_date USING(pool_cut_off_date);
    """,

    "14_Neo_Arrears_List": """
        INSERT INTO credit_risk_playground.stg_mrt_neo_arrears_list
        WITH reporting_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        neo_customer_level_expo AS (
            SELECT 
                pool_cut_off_date, borrower_identifier, COUNT(DISTINCT loan_identifier) AS number_of_loans, SUM(original_balance) AS customer_level_original_balance, SUM(current_balance) AS customer_level_current_balance, SUM(current_valuation_amount) AS total_collateral_value, SUM(arrears_balance) AS arrears_balance_total
            FROM credit_risk_playground.stg_mrt_neo_esma_to_ecb_raw a 
            INNER JOIN reporting_date b ON a.pool_cut_off_date = b.reporting_date
            WHERE current_balance > 0 
            GROUP BY pool_cut_off_date, borrower_identifier
        ),
        previous_arrears_list AS (
            SELECT a.* 
            FROM credit_risk_playground.stg_mrt_neo_arrears_list a
            INNER JOIN reporting_date AS b
            ON a.pool_cut_off_date::date = LAST_DAY(DATE_ADD('month', -1, b.reporting_date::date)::date)
        ),
        temp_tbl AS (
            SELECT 
                x.pool_cut_off_date, x.borrower_identifier, x.loan_identifier,
                CASE WHEN type_of_guarantee_provider = 7 THEN 'NHG' ELSE 'Non-NHG' END AS segment,
                account_status, number_months_in_arrears AS number_of_months_arrears, current_balance, customer_level_current_balance, arrears_balance_total, arrears_balance,
                CASE WHEN customer_level_current_balance IN (NULL, 0) THEN NULL ELSE ROUND(100 * (arrears_balance / customer_level_current_balance), 1) END AS arrears_balance_ratio
            FROM credit_risk_playground.stg_mrt_neo_esma_to_ecb_raw AS x
            INNER JOIN neo_customer_level_expo AS y
            ON x.pool_cut_off_date = y.pool_cut_off_date AND x.borrower_identifier = y.borrower_identifier AND x.current_balance > 0
        ),
        temp_tbl1 AS (
            SELECT t1.*, t2.account_status AS prev_account_status, t2.months_in_arrears AS prev_months_in_arrears, t2.consecutive_months_in_arrears AS prev_consecutive_months_in_arrears, t2.first_date_in_arrears AS prev_first_date_in_arrears, t2.last_arrears_info_date AS prev_last_arrears_info_date
            FROM temp_tbl AS t1 LEFT JOIN previous_arrears_list AS t2 USING(loan_identifier)
        ),
        months_in_arrears AS (
            SELECT tbl1.*,
                CASE WHEN prev_first_date_in_arrears IS NULL AND arrears_balance_total > 5 THEN pool_cut_off_date ELSE prev_first_date_in_arrears END AS first_date_in_arrears,
                CASE WHEN prev_months_in_arrears = 'probation' AND arrears_balance_total = 0 THEN prev_last_arrears_info_date ELSE pool_cut_off_date END AS last_arrears_info_date,
                CASE 
                    WHEN prev_consecutive_months_in_arrears IS NULL THEN 1
                    WHEN prev_months_in_arrears = 'probation' AND arrears_balance_total = 0 THEN 0
                    ELSE prev_consecutive_months_in_arrears + 1
                END AS consecutive_months_in_arrears
            FROM temp_tbl1 AS tbl1
        ),
        arrears_aggregation AS (
            SELECT arrs.*,
                CASE 
                    WHEN account_status IS NULL OR consecutive_months_in_arrears IS NULL OR arrears_balance < 5 OR arrears_balance IS NULL THEN 'performing'
                    WHEN account_status IN (2,3) AND consecutive_months_in_arrears = 3 AND ROUND(arrears_balance_ratio, 1) < 1 THEN consecutive_months_in_arrears::text
                    WHEN account_status IN (2,3) AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 'default'
                    WHEN account_status IN (2,3) AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) < 1 THEN '>3'
                    WHEN prev_months_in_arrears = 'default' AND arrears_balance_total > 5 THEN 'default'
                    WHEN prev_months_in_arrears = 'default' AND arrears_balance_total = 0 THEN 'probabtion'
                    WHEN prev_months_in_arrears = 'probation' AND DATEDIFF(month, last_arrears_info_date, pool_cut_off_date) > 12 THEN 'performing'
                    ELSE consecutive_months_in_arrears::text
                END months_in_arrears,
                ROW_NUMBER() OVER(PARTITION BY borrower_identifier, loan_identifier, segment, current_balance ORDER BY borrower_identifier, loan_identifier, segment, current_balance) AS duplicates
            FROM months_in_arrears AS arrs
        )
        SELECT pool_cut_off_date, borrower_identifier, loan_identifier, segment, account_status::smallint, number_of_months_arrears, consecutive_months_in_arrears, months_in_arrears::text, arrears_balance_ratio, first_date_in_arrears, last_arrears_info_date, ROUND(arrears_balance_total, 2) AS arrears_balance_customer_level, ROUND(arrears_balance, 2) AS arrears_balance_loan_level, current_balance, ROUND(customer_level_current_balance, 2) AS customer_level_current_balance
        FROM arrears_aggregation
        WHERE current_balance > 0 AND months_in_arrears <> 'performing' AND duplicates = 1
        ORDER BY pool_cut_off_date, consecutive_months_in_arrears, borrower_identifier, loan_identifier;
    """,

    "15_Neo_Credit_Risk_Calculations": """
        INSERT INTO credit_risk_playground.stg_mrt_neo_credit_risk_calculations
        WITH reporting_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        collateral_details AS (
            SELECT DISTINCT loannumber, typeofrealestate,
                CASE WHEN LOWER(typeofrealestate) LIKE '%house%' THEN 'house' ELSE 'apartment' END AS property_class,
                constructionyear,
                CASE 
                    WHEN constructionyear <= 1945 OR constructionyear IS NULL THEN 1 WHEN constructionyear <= 1964 THEN 2 WHEN constructionyear <= 1974 THEN 3 WHEN constructionyear <= 1982 THEN 4 WHEN constructionyear <= 1987 THEN 5 WHEN constructionyear <= 1991 THEN 6 WHEN constructionyear <= 1999 THEN 7 WHEN constructionyear <= 2005 THEN 8 ELSE 9
                END building_year,
                energylabel,
                CASE 
                    WHEN LOWER(energylabel) LIKE '%gel%' THEN 'G' WHEN LOWER(energylabel) LIKE '%a%' THEN 'A' WHEN energylabel IS NULL AND building_year = 1 THEN 'G' WHEN energylabel IS NULL AND building_year IN (1,2) AND property_class IN ('house') THEN 'F' WHEN energylabel IS NULL AND building_year = 2 AND property_class IN ('apartment') THEN 'E' WHEN energylabel IS NULL AND building_year = 3 AND property_class IN ('house') THEN 'D' WHEN energylabel IS NULL AND building_year = 3 AND property_class IN ('apartment') THEN 'F' WHEN energylabel IS NULL AND building_year IN (4,5,6) THEN 'C' WHEN energylabel IS NULL AND building_year = 7 THEN 'B' WHEN energylabel IS NULL AND building_year = 8 THEN 'B' WHEN energylabel IS NULL AND building_year = 9 THEN 'A' ELSE energylabel
                END AS energy_label
            FROM mo_collateral_collateral 
            WHERE ismaincollateral = 'T' AND etl_updated = (SELECT MAX(etl_updated) FROM mo_collateral_collateral)
        ),
        arrears_info AS (
            SELECT borrower_identifier, first_date_in_arrears, last_arrears_info_date, consecutive_months_in_arrears
            FROM credit_risk_playground.stg_mrt_neo_arrears_list AS a
            INNER JOIN reporting_date AS b ON a.pool_cut_off_date = b.reporting_date
            GROUP BY borrower_identifier, first_date_in_arrears, last_arrears_info_date, consecutive_months_in_arrears
        ),
        temp_neo_exposures AS (
            SELECT 
                pool_cut_off_date::date AS pool_cut_off_date, neo.borrower_identifier, loan_identifier,
                CASE WHEN type_of_guarantee_provider = 7 THEN 'NHG' ELSE 'Non-NHG' END AS segment,
                primary_income, secondary_income,
                CASE WHEN secondary_income IS NULL THEN primary_income ELSE primary_income + secondary_income END AS total_income,
                CASE WHEN borrowers_employment_status IS NULL THEN 10 ELSE borrowers_employment_status::numeric END AS borrowers_employment_status,
                loan_origination_date::date, date_of_loan_maturity::date,
                TO_DATE(CONCAT(REPLACE(CASE WHEN interest_revision_date_1 LIKE 'N%' THEN NULL ELSE interest_revision_date_1 END, '-', '/'), '/01'), 'YYYY/MM/DD') AS interest_revision_date,
                loan_term,
                CASE WHEN loan_term::numeric <= 30 THEN 'Brigding' ELSE 'Standard' END AS purpose,
                original_balance, current_balance,
                CASE WHEN retained_amount IS NULL THEN 0 ELSE retained_amount::float END AS undrawn_loanpart,
                payment_type, payment_due, debt_to_income, current_interest_rate, interest_rate_reset_interval, geographic_region_list, original_loan_to_value, valuation_amount, current_loan_to_value, current_valuation_amount, property_value_at_time_of_latest_loan_advance, indexed_foreclosure_value,
                CASE WHEN type_of_guarantee_provider = 7 THEN GREATEST(0, current_balance - (original_balance * 0.1)) ELSE 0 END AS guaranteed_foreclosure_value,
                CASE WHEN arrears_balance = 0 THEN 1 ELSE account_status::SMALLINT END AS account_status,
                CASE WHEN number_months_in_arrears IS NULL THEN 0 WHEN arrears_balance::numeric = 0 THEN 0 ELSE number_months_in_arrears::numeric END AS number_months_in_arrears,
                ar_info.first_date_in_arrears,
                CASE WHEN date_last_in_arrears IS NULL THEN NULL ELSE LAST_DAY(date_last_in_arrears::date) END AS date_last_in_arrears_temp,
                CASE WHEN date_last_current LIKE 'N%' THEN NULL ELSE date_last_current::date END AS date_last_current_temp,
                ar_info.last_arrears_info_date, ar_info.consecutive_months_in_arrears AS consecutive_arrears, arrears_balance 
            FROM reporting_date rdate 
            INNER JOIN credit_risk_playground.stg_mrt_neo_esma_to_ecb_raw AS neo ON neo.pool_cut_off_date::date = rdate.reporting_date
            LEFT JOIN arrears_info AS ar_info USING(borrower_identifier)
            WHERE neo.current_balance <> 0
        ),
        temp_neo_exposures1 AS (
            SELECT a.*, b.loannumber, b.constructionyear, b.energylabel, b.energy_label, b.typeofrealestate, ROW_NUMBER() OVER(PARTITION BY loan_identifier ORDER BY loan_identifier) AS dups
            FROM temp_neo_exposures a LEFT JOIN collateral_details b ON SUBSTRING(a.loan_identifier, 1, 7)::numeric = b.loannumber::numeric
        ),
        neo_exposures AS (SELECT * FROM temp_neo_exposures1 WHERE dups = 1),
        neo_customer_level_expo AS (
            SELECT pool_cut_off_date, borrower_identifier, SUM(original_balance) AS customer_level_original_balance, SUM(current_balance) AS customer_level_current_balance, SUM(current_valuation_amount) AS total_collateral_value
            FROM neo_exposures GROUP BY pool_cut_off_date, borrower_identifier
        ),
        bridge_exposures AS (
            SELECT pool_cut_off_date, borrower_identifier, SUM(CASE WHEN current_balance IS NULL THEN 0 ELSE current_balance END) AS current_bridge_exposure
            FROM neo_exposures WHERE loan_term <= 30 GROUP BY pool_cut_off_date, borrower_identifier
        ),
        mev_plus_parameter_inputs AS (
            SELECT temp_mev.*,
                EXP(pd_factor_constant + (pd_factor_ur_coeff * diff_12m_ur) + (pd_factor_ir_coeff * diff_12m_ir)) AS pd_factor,
                EXP(indexed_vlaue_scalar_const + (indexed_vlaue_scalar_rhpi_coeff * log_diff_12m_rhpi_lag_1) + (indexed_vlaue_scalar_gdp_coeff * diff_12m_gdp)) AS indexed_vlaue_scalar
            FROM (
                SELECT a.ref_date, a.log_diff_12m_rhpi_lag_1, a.diff_12m_ur, a.diff_12m_ir, a.diff_12m_gdp, b.value AS cure_rate, c.coefficient AS pd_factor_constant, d.coefficient AS pd_factor_ur_coeff, e.coefficient AS pd_factor_ir_coeff, f.coefficient AS indexed_vlaue_scalar_const, g.coefficient AS indexed_vlaue_scalar_rhpi_coeff, h.coefficient AS indexed_vlaue_scalar_gdp_coeff, i.surcharge AS forcl_discount, j.surcharge AS forcl_variable_cost
                FROM credit_risk_playground.stg_mrt_mev_input_data AS a, credit_risk_playground.stg_mrt_portfolio_parameters AS b, credit_risk_playground.stg_mrt_risk_model_parameters AS c, credit_risk_playground.stg_mrt_risk_model_parameters AS d, credit_risk_playground.stg_mrt_risk_model_parameters AS e, credit_risk_playground.stg_mrt_risk_model_parameters AS f, credit_risk_playground.stg_mrt_risk_model_parameters AS g, credit_risk_playground.stg_mrt_risk_model_parameters AS h, credit_risk_playground.stg_mrt_fitch_foreclosure_discounts AS i, credit_risk_playground.stg_mrt_fitch_foreclosure_discounts AS j
                WHERE b.parameter_name = 'cure_rate' AND c.main_model = 'pd' AND c.sub_model = 'pd_factor' AND c.variable_name = 'constant' AND d.main_model = 'pd' AND d.sub_model = 'pd_factor' AND d.variable_name = 'diff_12m_ur' AND e.main_model = 'pd' AND e.sub_model = 'pd_factor' AND e.variable_name = 'diff_12m_ir' AND f.main_model = 'lgd' AND f.sub_model = 'indexed_value_scaler' AND f.variable_name = 'constant' AND g.main_model = 'lgd' AND g.sub_model = 'indexed_value_scaler' AND g.variable_name = 'log_diff_12m_rhpi_lag_1' AND h.main_model = 'lgd' AND h.sub_model = 'indexed_value_scaler' AND h.variable_name = 'diff_12m_gdp' AND i.discount = 'Foreclosed_Sale_Adjustment' AND j.discount = 'Variable_Foreclosure_Cost'
            ) temp_mev
        ),
        temp_aggregation AS (
            SELECT neo_expo.*, COALESCE(consecutive_arrears, 0) AS consecutive_months_in_arrears, cust_level_expo.customer_level_original_balance, cust_level_expo.customer_level_current_balance, cust_level_expo.total_collateral_value,
                CASE WHEN bridge.current_bridge_exposure IS NULL THEN 0 ELSE bridge.current_bridge_exposure END AS current_bridge_exposure 
            FROM neo_exposures AS neo_expo LEFT JOIN neo_customer_level_expo AS cust_level_expo ON neo_expo.pool_cut_off_date = cust_level_expo.pool_cut_off_date AND neo_expo.borrower_identifier = cust_level_expo.borrower_identifier LEFT JOIN bridge_exposures AS bridge ON cust_level_expo.pool_cut_off_date = bridge.pool_cut_off_date AND cust_level_expo.borrower_identifier = bridge.borrower_identifier
        ),
        temp_aggregation1 AS (
            SELECT tmp_tbl.*, 100 * (arrears_balance / customer_level_current_balance) AS arrears_balance_ratio, customer_level_current_balance / total_income AS current_debt_to_income_ratio, (customer_level_current_balance - current_bridge_exposure) / total_income AS current_debt_to_income_ratio_excl_bridge,
                CASE WHEN segment = 'NHG' THEN 0.75 * 0.1 * current_balance ELSE 0.35 * current_balance + 0.4 * GREATEST(0, current_balance - 0.8 * current_valuation_amount) END AS rwa,
                DATEDIFF('month', pool_cut_off_date::date, LAST_DAY(interest_revision_date)::date) AS months_to_revision_date, DATEDIFF('year', pool_cut_off_date::date, date_of_loan_maturity::date) AS remaining_term, (current_balance - undrawn_loanpart) * ((current_interest_rate / 100) / 12) AS next_month_interest_payment
            FROM temp_aggregation AS tmp_tbl
        ),
        temp_aggregation1a AS (SELECT *, GREATEST(0, payment_due - next_month_interest_payment) AS one_month_paydown FROM temp_aggregation1),
        temp_aggregation2 AS (SELECT tmp_tbl1.*, (current_balance - undrawn_loanpart + one_month_paydown) * ((current_interest_rate / 100) / 12) AS prev_month_interest_payment, CASE WHEN loan_term <= 30 THEN 0 ELSE ROUND(original_balance / loan_term, 2) END AS monthly_redemption FROM temp_aggregation1a AS tmp_tbl1),
        temp_aggregation3 AS (
            SELECT tmp_tbl3.*, total_income / 12 AS total_monthly_income, cust_level_prev_month_interest_payment, cust_level_monthly_redemption, cust_level_total_payment,
                CASE WHEN loan_term <= 30 THEN 0 ELSE cust_level_monthly_redemption + cust_level_prev_month_interest_payment END AS total_monthly_debt_service,
                CASE WHEN loan_term <= 30 THEN 0 ELSE total_monthly_income / cust_level_total_payment END AS debt_service_coverage_ratio
            FROM temp_aggregation2 AS tmp_tbl3 LEFT JOIN (SELECT pool_cut_off_date, borrower_identifier, SUM(prev_month_interest_payment) AS cust_level_prev_month_interest_payment, SUM(monthly_redemption) AS cust_level_monthly_redemption, SUM(payment_due) AS cust_level_total_payment FROM temp_aggregation2 GROUP BY 1, 2) tmp_tbl4 ON tmp_tbl3.pool_cut_off_date = tmp_tbl4.pool_cut_off_date AND tmp_tbl3.borrower_identifier = tmp_tbl4.borrower_identifier
        ),
        temp_aggregation3a AS (SELECT *, CASE WHEN loan_term <= 30 OR current_balance = 0 THEN 0 ELSE 100 * (total_monthly_debt_service / total_monthly_income) END AS fitch_dti FROM temp_aggregation3),
        temp_aggregation4 AS (
            SELECT *,
                CASE WHEN current_loan_to_value IS NULL THEN NULL WHEN current_loan_to_value < 65 THEN 1 WHEN current_loan_to_value < 85 THEN 2 WHEN current_loan_to_value < 95 THEN 3 ELSE 4 END AS fitch_ltv_class,
                CASE WHEN months_to_revision_date IS NULL THEN NULL WHEN months_to_revision_date / 12 < 8 THEN 1 WHEN months_to_revision_date / 12 < 12 THEN 2 WHEN months_to_revision_date / 12 < 17 THEN 3 WHEN months_to_revision_date / 12 < 25 THEN 4 ELSE 5 END AS fitch_term_class,
                CASE WHEN loan_term IS NULL THEN NULL WHEN loan_term <= 30 THEN 0 WHEN fitch_dti < 20 THEN 1 WHEN fitch_dti < 30 THEN 2 WHEN fitch_dti < 40 THEN 3 WHEN fitch_dti < 50 THEN 4 ELSE 5 END AS fitch_dti_class,
                CASE WHEN original_loan_to_value IS NULL THEN NULL WHEN original_loan_to_value < 40 THEN '<40' WHEN original_loan_to_value < 50 THEN '40-50' WHEN original_loan_to_value < 60 THEN '50-60' WHEN original_loan_to_value < 65 THEN '60-65' WHEN original_loan_to_value < 70 THEN '65-70' WHEN original_loan_to_value < 75 THEN '70-75' WHEN original_loan_to_value < 80 THEN '75-80' WHEN original_loan_to_value < 85 THEN '80-85' WHEN original_loan_to_value < 90 THEN '85-90' WHEN original_loan_to_value < 95 THEN '90-95' WHEN original_loan_to_value < 100 THEN '95-100' WHEN original_loan_to_value < 105 THEN '100-105' WHEN original_loan_to_value < 110 THEN '105-110' ELSE '>=110' END AS fitch_oltv_class,
                CASE WHEN current_loan_to_value IS NULL THEN NULL WHEN current_loan_to_value < 40 THEN '<40' WHEN current_loan_to_value < 50 THEN '40-50' WHEN current_loan_to_value < 60 THEN '50-60' WHEN current_loan_to_value < 65 THEN '60-65' WHEN current_loan_to_value < 70 THEN '65-70' WHEN current_loan_to_value < 75 THEN '70-75' WHEN current_loan_to_value < 80 THEN '75-80' WHEN current_loan_to_value < 85 THEN '80-85' WHEN current_loan_to_value < 90 THEN '85-90' WHEN current_loan_to_value < 95 THEN '90-95' WHEN current_loan_to_value < 100 THEN '95-100' WHEN current_loan_to_value < 105 THEN '100-105' WHEN current_loan_to_value < 110 THEN '105-110' ELSE '>=110' END AS fitch_cltv_class,
                CASE WHEN remaining_term IS NULL THEN NULL WHEN remaining_term < 5 THEN '<5' WHEN remaining_term < 10 THEN '5-10' WHEN remaining_term < 15 THEN '10-15' WHEN remaining_term < 20 THEN '15-20' WHEN remaining_term < 25 THEN '20-25' ELSE '>=25' END AS fitch_remaining_term_class,
                CASE WHEN original_loan_to_value IS NULL THEN NULL WHEN original_loan_to_value < 50 THEN '<50' WHEN original_loan_to_value < 60 THEN '50-60' WHEN original_loan_to_value < 70 THEN '60-70' WHEN original_loan_to_value < 80 THEN '70-80' WHEN original_loan_to_value < 90 THEN '80-90' ELSE '>=90' END AS fitch_oltv_term_class,
                CASE WHEN current_loan_to_value IS NULL THEN NULL WHEN current_loan_to_value < 50 THEN '<50' WHEN current_loan_to_value < 60 THEN '50-60' WHEN current_loan_to_value < 70 THEN '60-70' WHEN current_loan_to_value < 80 THEN '70-80' WHEN current_loan_to_value < 90 THEN '80-90' ELSE '>=90' END AS fitch_cltv_term_class
            FROM temp_aggregation3a
        ),
        temp_aggregation5 AS (SELECT temp_tbl6.*, base_ff.baseline_ff FROM temp_aggregation4 AS temp_tbl6 LEFT JOIN credit_risk_playground.stg_mrt_fitch_ff_baseline AS base_ff ON temp_tbl6.fitch_cltv_class = base_ff.ltv_class AND temp_tbl6.fitch_dti_class = base_ff.dti_class),
        temp_aggregation6 AS (SELECT temp_tbl7.*, io_adjustment.io_adjustment FROM temp_aggregation5 AS temp_tbl7 LEFT JOIN credit_risk_playground.stg_mrt_fitch_ff_io_adjustment AS io_adjustment ON temp_tbl7.fitch_oltv_term_class = io_adjustment.ltv_class AND temp_tbl7.fitch_remaining_term_class = io_adjustment.remaining_term_class),
        temp_aggregation7 AS (SELECT temp_tbl8.*, income_adj.income_adjustment FROM temp_aggregation6 AS temp_tbl8 LEFT JOIN credit_risk_playground.stg_mrt_fitch_income_adjustment AS income_adj ON temp_tbl8.borrowers_employment_status = income_adj.income_source),
        temp_aggregation8 AS (SELECT temp_tbl9.*, arrers_floor.ff_floor FROM temp_aggregation7 AS temp_tbl9 LEFT JOIN credit_risk_playground.stg_mrt_fitch_arrears_floor AS arrers_floor ON temp_tbl9.consecutive_months_in_arrears = arrers_floor.months_in_arrears),
        temp_aggregation9 AS (
            SELECT *,
                CASE WHEN loan_term <= 30 THEN 0 ELSE baseline_ff END AS final_baseline_ff,
                CASE WHEN fitch_dti_class = 0 OR payment_type <> 6 THEN 1 ELSE io_adjustment END AS final_io_adjustment,
                CASE WHEN account_status IN (2,3) AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 1 WHEN consecutive_months_in_arrears > 4 AND arrears_balance > 5 THEN 0.55 WHEN consecutive_months_in_arrears > 4 AND arrears_balance < 5 THEN 0 ELSE ff_floor END AS final_ff_floor
            FROM temp_aggregation8
        ),
        temp_aggregation10 AS (
            SELECT tmptbl.*,
                CASE WHEN borrower_identifier IN (5389716, 5638173, 5580361) THEN 100 ELSE 100 * (1 - POWER((1 - tmptbl.adjusted_fitch_base_ff / 100), 0.1428571)) END AS annualised_ff
            FROM (
                SELECT *, LEAST(100, 100 * GREATEST(final_baseline_ff * final_io_adjustment * (CASE WHEN income_adjustment IS NULL THEN 1 ELSE income_adjustment END), final_ff_floor) + (CASE WHEN energy_label IN ('A', 'B') THEN 0 ELSE 0.34 END)) AS adjusted_fitch_base_ff
                FROM temp_aggregation9
            ) AS tmptbl
        ),
        temp_aggregation11 AS (
            SELECT temp10.*, forcl_discount, forcl_variable_cost, cure_rate, pd_factor, indexed_vlaue_scalar
            FROM temp_aggregation10 AS temp10 LEFT JOIN mev_plus_parameter_inputs AS mevs_input ON LAST_DAY(DATE_ADD('month', -1, temp10.pool_cut_off_date))::date = mevs_input.ref_date
        ),
        pd_pit_calculation AS (
            SELECT temp11.*,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) END AS pd_ttc,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) * pd_factor END AS pd_pit
            FROM temp_aggregation11 AS temp11
        ),
        pd_pit_to_master_scale AS (
            SELECT temp_hold2.*,
                CASE 
                    WHEN pd_pit < 0.01 THEN 1 WHEN pd_pit <= 0.04 THEN 2 WHEN pd_pit <= 0.09 THEN 3 WHEN pd_pit <= 0.15 THEN 4 WHEN pd_pit <= 0.23 THEN 5 WHEN pd_pit <= 0.38 THEN 6 WHEN pd_pit <= 0.73 THEN 7 WHEN pd_pit <= 1.42 THEN 8 WHEN pd_pit <= 2.25 THEN 9 WHEN pd_pit <= 3.21 THEN 10 WHEN pd_pit <= 4.29 THEN 11 WHEN pd_pit <= 5.61 THEN 12 WHEN pd_pit <= 11.18 THEN 13 WHEN pd_pit <= 27.89 THEN 14 WHEN pd_pit <= 46.00 THEN 15 WHEN pd_pit <= 60.86 THEN 16 WHEN pd_pit > 60.86 AND pd_pit <= 99.99 THEN 17 WHEN account_status = 3 AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 18 WHEN annualised_ff = 100 THEN 18
                END AS pd_pit_class,
                (current_balance / customer_level_current_balance) * total_collateral_value AS collateral_value
            FROM (
                SELECT temp_hold1.*,
                    CASE 
                        WHEN pd_pit < 0.01 THEN 0.00 WHEN pd_pit <= 0.04 THEN 0.03 WHEN pd_pit <= 0.09 THEN 0.07 WHEN pd_pit <= 0.15 THEN 0.13 WHEN pd_pit <= 0.23 THEN 0.19 WHEN pd_pit <= 0.38 THEN 0.30 WHEN pd_pit <= 0.73 THEN 0.49 WHEN pd_pit <= 1.42 THEN 1.10 WHEN pd_pit <= 2.25 THEN 1.85 WHEN pd_pit <= 3.21 THEN 2.75 WHEN pd_pit <= 4.29 THEN 3.75 WHEN pd_pit <= 5.61 THEN 4.91 WHEN pd_pit <= 11.18 THEN 6.43 WHEN pd_pit <= 27.89 THEN 19.45 WHEN pd_pit <= 46.00 THEN 40 WHEN pd_pit <= 60.86 THEN 52.92 WHEN pd_pit > 60.86 AND pd_pit <= 99.99 THEN 70 WHEN account_status = 3 AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 100 WHEN annualised_ff = 100 THEN 100
                    END AS pd_pit_mastered
                FROM pd_pit_calculation temp_hold1
            ) temp_hold2
        ),
        collateral_adjustement AS (
            SELECT temp13.*, indexed_vlaue_scalar * collateral_value * (CASE WHEN energy_label IN ('A', 'B', 'C') THEN 1 ELSE (1 - 0.037) END) AS adjusted_collateral_value
            FROM pd_pit_to_master_scale AS temp13
        ),
        foreclosure_value_calculation AS (
            SELECT temp14.*, GREATEST(guaranteed_foreclosure_value, adjusted_collateral_value * (1 - forcl_discount - forcl_variable_cost)) AS fitch_foreclosure_value
            FROM collateral_adjustement AS temp14
        ),
        lgd_calculation AS (
            SELECT temp16.*, CASE WHEN loan_term <= 30 THEN 0 ELSE GREATEST(5, lgd_pit) END AS lgd
            FROM (
                SELECT temp15.*, CASE WHEN loan_term <= 30 OR current_balance = 0 THEN 0 ELSE 100 * (1 - cure_rate) * GREATEST(0, 1 - (fitch_foreclosure_value / current_balance)) END AS lgd_pit
                FROM foreclosure_value_calculation AS temp15
            ) temp16
        ),
        ecl_calculation AS (
            SELECT temp15.*, pd_pit_mastered / 100 * (lgd / 100) * current_balance AS expected_loss_1Year, (1 - POWER((1 - 0.003), LEAST(7, GREATEST(0, remaining_term)))) * original_balance AS performance_trigger, province_code, pronvince_name
            FROM lgd_calculation temp15 LEFT JOIN credit_risk_playground.stg_mrt_fitch_location_codes AS geo_code ON temp15.geographic_region_list = geo_code.geographical_code 
        )
        SELECT pool_cut_off_date::DATE, borrower_identifier::VARCHAR, loan_identifier::VARCHAR, segment::VARCHAR, primary_income::float, secondary_income::float, total_income::float, borrowers_employment_status::NUMERIC, loan_origination_date::DATE, date_of_loan_maturity::DATE, interest_revision_date::DATE, loan_term::INTEGER, purpose::VARCHAR, original_balance::float, current_balance::float, undrawn_loanpart::float, payment_type::SMALLINT, payment_due::float, debt_to_income::float, current_interest_rate::float, interest_rate_reset_interval::INTEGER, geographic_region_list::VARCHAR, original_loan_to_value::float, valuation_amount::float, current_loan_to_value::float, current_valuation_amount::float, property_value_at_time_of_latest_loan_advance::float, indexed_foreclosure_value::float, guaranteed_foreclosure_value::float, account_status::SMALLINT, number_months_in_arrears::NUMERIC, first_date_in_arrears::DATE, date_last_in_arrears_temp::DATE, date_last_current_temp::DATE, last_arrears_info_date::DATE, arrears_balance::float, consecutive_months_in_arrears::BIGINT, customer_level_original_balance::float, customer_level_current_balance::float, total_collateral_value::float, current_bridge_exposure::float, arrears_balance_ratio::float, current_debt_to_income_ratio::float, current_debt_to_income_ratio_excl_bridge::float, rwa::float, months_to_revision_date::BIGINT, remaining_term::BIGINT, next_month_interest_payment::float, one_month_paydown::float, prev_month_interest_payment::float, monthly_redemption::float, total_monthly_income::float, cust_level_prev_month_interest_payment::float, cust_level_monthly_redemption::float, cust_level_total_payment::float, total_monthly_debt_service::float, debt_service_coverage_ratio::float, fitch_dti::float, fitch_ltv_class::INTEGER, fitch_term_class::INTEGER, fitch_dti_class::INTEGER, fitch_oltv_class::VARCHAR, fitch_cltv_class::VARCHAR, fitch_remaining_term_class::VARCHAR, fitch_oltv_term_class::VARCHAR, fitch_cltv_term_class::VARCHAR, baseline_ff::float, io_adjustment::float, income_adjustment::float, ff_floor::float, final_baseline_ff::float, final_io_adjustment::float, final_ff_floor::float, adjusted_fitch_base_ff::float, annualised_ff::float, forcl_discount::float, forcl_variable_cost::float, cure_rate::float, pd_factor::float, indexed_vlaue_scalar::float, pd_ttc::float, pd_pit::float, pd_pit_mastered::FLOAT, pd_pit_class::INTEGER, collateral_value::float, adjusted_collateral_value::float, fitch_foreclosure_value::float, lgd_pit::float, lgd::float, expected_loss_1year::float, performance_trigger::float, province_code::SMALLINT, pronvince_name::VARCHAR, GETDATE() AS etl_updated, 'alfred_teye' AS created_by
        FROM ecl_calculation ORDER BY pool_cut_off_date, borrower_identifier, loan_identifier;
    """,

    "16_Neo_Credit_Risk_Calculations_Scenarios": """
        INSERT INTO credit_risk_playground.stg_mrt_neo_credit_risk_calculations_with_scenarios 
        WITH reporting_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        collateral_details AS (
            SELECT DISTINCT loannumber, typeofrealestate,
                CASE WHEN LOWER(typeofrealestate) LIKE '%house%' THEN 'house' ELSE 'apartment' END AS property_class,
                constructionyear,
                CASE 
                    WHEN constructionyear <= 1945 OR constructionyear IS NULL THEN 1 WHEN constructionyear <= 1964 THEN 2 WHEN constructionyear <= 1974 THEN 3 WHEN constructionyear <= 1982 THEN 4 WHEN constructionyear <= 1987 THEN 5 WHEN constructionyear <= 1991 THEN 6 WHEN constructionyear <= 1999 THEN 7 WHEN constructionyear <= 2005 THEN 8 ELSE 9
                END building_year,
                energylabel,
                CASE 
                    WHEN LOWER(energylabel) LIKE '%gel%' THEN 'G' WHEN LOWER(energylabel) LIKE '%a%' THEN 'A' WHEN energylabel IS NULL AND building_year = 1 THEN 'G' WHEN energylabel IS NULL AND building_year IN (1,2) AND property_class IN ('house') THEN 'F' WHEN energylabel IS NULL AND building_year = 2 AND property_class IN ('apartment') THEN 'E' WHEN energylabel IS NULL AND building_year = 3 AND property_class IN ('house') THEN 'D' WHEN energylabel IS NULL AND building_year = 3 AND property_class IN ('apartment') THEN 'F' WHEN energylabel IS NULL AND building_year IN (4,5,6) THEN 'C' WHEN energylabel IS NULL AND building_year = 7 THEN 'B' WHEN energylabel IS NULL AND building_year = 8 THEN 'B' WHEN energylabel IS NULL AND building_year = 9 THEN 'A' ELSE energylabel
                END AS energy_label
            FROM mo_collateral_collateral WHERE ismaincollateral = 'T' AND etl_updated = (SELECT MAX(etl_updated) FROM mo_collateral_collateral)
        ),
        temp_neo_exposures AS (
            SELECT 
                neo.pool_cut_off_date::date AS pool_cut_off_date, neo.borrower_identifier, neo.loan_identifier,
                CASE WHEN type_of_guarantee_provider = 7 THEN 'NHG' ELSE 'Non-NHG' END AS segment,
                primary_income, secondary_income,
                CASE WHEN secondary_income IS NULL THEN primary_income ELSE primary_income + secondary_income END AS total_income,
                CASE WHEN borrowers_employment_status IS NULL THEN 10 ELSE borrowers_employment_status::numeric END AS borrowers_employment_status,
                loan_origination_date, date_of_loan_maturity,
                TO_DATE(CONCAT(REPLACE(CASE WHEN interest_revision_date_1 LIKE 'N%' THEN NULL ELSE interest_revision_date_1 END, '-', '/'), '/01'), 'YYYY/MM/DD') AS interest_revision_date,
                loan_term,
                CASE WHEN loan_term::numeric <= 30 THEN 'Brigding' ELSE 'Standard' END AS purpose,
                original_balance, neo.current_balance,
                CASE WHEN retained_amount IS NULL THEN 0 ELSE retained_amount::float END AS undrawn_loanpart,
                payment_type, payment_due, debt_to_income, current_interest_rate, interest_rate_reset_interval, geographic_region_list, original_loan_to_value, valuation_amount, current_loan_to_value, current_valuation_amount, property_value_at_time_of_latest_loan_advance, indexed_foreclosure_value,
                CASE WHEN type_of_guarantee_provider = 7 THEN GREATEST(0, neo.current_balance - (original_balance * 0.1)) ELSE 0 END AS guaranteed_foreclosure_value,
                neo.account_status, arr.number_of_months_arrears AS number_months_in_arrears, arr.first_date_in_arrears, arr.last_arrears_info_date, arr.consecutive_months_in_arrears, arr.arrears_balance_ratio, neo.arrears_balance 
            FROM reporting_date rdate
            INNER JOIN credit_risk_playground.stg_mrt_neo_esma_to_ecb_raw AS neo ON neo.pool_cut_off_date::date = rdate.reporting_date
            LEFT JOIN credit_risk_playground.stg_mrt_neo_arrears_list AS arr ON neo.pool_cut_off_date = arr.pool_cut_off_date AND neo.borrower_identifier = arr.borrower_identifier AND neo.loan_identifier = arr.loan_identifier
            WHERE neo.current_balance <> 0
        ),
        temp_neo_exposures1 AS (
            SELECT a.*, loannumber, constructionyear, energylabel, energy_label, typeofrealestate, ROW_NUMBER() OVER(PARTITION BY loan_identifier ORDER BY loan_identifier) AS dups
            FROM temp_neo_exposures a LEFT JOIN collateral_details b ON SUBSTRING(a.loan_identifier, 1, 7)::numeric = b.loannumber::numeric
        ),
        neo_exposures AS (SELECT * FROM temp_neo_exposures1 WHERE dups = 1),
        neo_customer_level_expo AS (
            SELECT pool_cut_off_date, borrower_identifier, SUM(original_balance) AS customer_level_original_balance, SUM(current_balance) AS customer_level_current_balance, SUM(current_valuation_amount) AS total_collateral_value
            FROM neo_exposures WHERE dups = 1 GROUP BY pool_cut_off_date, borrower_identifier
        ),
        bridge_exposures AS (
            SELECT pool_cut_off_date, borrower_identifier, SUM(CASE WHEN current_balance IS NULL THEN 0 ELSE current_balance END) AS current_bridge_exposure
            FROM neo_exposures WHERE loan_term <= 30 GROUP BY pool_cut_off_date, borrower_identifier
        ),
        mev_plus_parameter_inputs AS (
            SELECT temp_mev.*,
                EXP(pd_factor_constant + (pd_factor_ur_coeff * ur_diff_12m_down_scen) + (pd_factor_ir_coeff * ir_diff_12m_down_scen)) AS pd_factor_down_scen,
                EXP(pd_factor_constant + (pd_factor_ur_coeff * ur_diff_12m_neu_scen) + (pd_factor_ir_coeff * ir_diff_12m_neu_scen)) AS pd_factor_neu_scen,
                EXP(pd_factor_constant + (pd_factor_ur_coeff * ur_diff_12m_up_scen) + (pd_factor_ir_coeff * ir_diff_12m_up_scen)) AS pd_factor_up_scen,
                EXP(indexed_vlaue_scalar_const + (indexed_vlaue_scalar_rhpi_coeff * rhpi_log_diff_12m_lag_1_down_scen) + (indexed_vlaue_scalar_gdp_coeff * gdp_diff_12m_down_scen)) AS indexed_vlaue_scalar_down_scen,
                EXP(indexed_vlaue_scalar_const + (indexed_vlaue_scalar_rhpi_coeff * rhpi_log_diff_12m_lag_1_neu_scen) + (indexed_vlaue_scalar_gdp_coeff * gdp_diff_12m_neu_scen)) AS indexed_vlaue_scalar_neu_scen,
                EXP(indexed_vlaue_scalar_const + (indexed_vlaue_scalar_rhpi_coeff * rhpi_log_diff_12m_lag_1_up_scen) + (indexed_vlaue_scalar_gdp_coeff * gdp_diff_12m_up_scen)) AS indexed_vlaue_scalar_up_scen
            FROM (
                SELECT a.reference_date AS ref_date, a.ur_diff_12m_down_scen, a.ur_diff_12m_neu_scen, a.ur_diff_12m_up_scen, a.ir_diff_12m_down_scen, a.ir_diff_12m_neu_scen, a.ir_diff_12m_up_scen, a.gdp_diff_12m_down_scen, a.gdp_diff_12m_neu_scen, a.gdp_diff_12m_up_scen, a.rhpi_log_diff_12m_lag_1_down_scen, a.rhpi_log_diff_12m_lag_1_neu_scen, a.rhpi_log_diff_12m_lag_1_up_scen, b.value AS cure_rate, c.coefficient AS pd_factor_constant, d.coefficient AS pd_factor_ur_coeff, e.coefficient AS pd_factor_ir_coeff, f.coefficient AS indexed_vlaue_scalar_const, g.coefficient AS indexed_vlaue_scalar_rhpi_coeff, h.coefficient AS indexed_vlaue_scalar_gdp_coeff, i.surcharge AS forcl_discount, j.surcharge AS forcl_variable_cost
                FROM credit_risk_playground.stg_mrt_ifrs9_mev_input_data AS a, credit_risk_playground.stg_mrt_portfolio_parameters AS b, credit_risk_playground.stg_mrt_risk_model_parameters AS c, credit_risk_playground.stg_mrt_risk_model_parameters AS d, credit_risk_playground.stg_mrt_risk_model_parameters AS e, credit_risk_playground.stg_mrt_risk_model_parameters AS f, credit_risk_playground.stg_mrt_risk_model_parameters AS g, credit_risk_playground.stg_mrt_risk_model_parameters AS h, credit_risk_playground.stg_mrt_fitch_foreclosure_discounts AS i, credit_risk_playground.stg_mrt_fitch_foreclosure_discounts AS j
                WHERE b.parameter_name = 'cure_rate' AND c.main_model = 'pd' AND c.sub_model = 'pd_factor' AND c.variable_name = 'constant' AND d.main_model = 'pd' AND d.sub_model = 'pd_factor' AND d.variable_name = 'diff_12m_ur' AND e.main_model = 'pd' AND e.sub_model = 'pd_factor' AND e.variable_name = 'diff_12m_ir' AND f.main_model = 'lgd' AND f.sub_model = 'indexed_value_scaler' AND f.variable_name = 'constant' AND g.main_model = 'lgd' AND g.sub_model = 'indexed_value_scaler' AND g.variable_name = 'log_diff_12m_rhpi_lag_1' AND h.main_model = 'lgd' AND h.sub_model = 'indexed_value_scaler' AND h.variable_name = 'diff_12m_gdp' AND i.discount = 'Foreclosed_Sale_Adjustment' AND j.discount = 'Variable_Foreclosure_Cost'
            ) temp_mev
        ),
        temp_aggregation AS (
            SELECT neo_expo.*, cust_level_expo.customer_level_original_balance, cust_level_expo.customer_level_current_balance, cust_level_expo.total_collateral_value,
                CASE WHEN bridge.current_bridge_exposure IS NULL THEN 0 ELSE bridge.current_bridge_exposure END AS current_bridge_exposure 
            FROM neo_exposures AS neo_expo LEFT JOIN neo_customer_level_expo AS cust_level_expo ON neo_expo.pool_cut_off_date = cust_level_expo.pool_cut_off_date AND neo_expo.borrower_identifier = cust_level_expo.borrower_identifier LEFT JOIN bridge_exposures AS bridge ON cust_level_expo.pool_cut_off_date = bridge.pool_cut_off_date AND cust_level_expo.borrower_identifier = bridge.borrower_identifier
        ),
        temp_aggregation1 AS (
            SELECT tmp_tbl.*, customer_level_current_balance / total_income AS current_debt_to_income_ratio, (customer_level_current_balance - current_bridge_exposure) / total_income AS current_debt_to_income_ratio_excl_bridge,
                CASE WHEN segment = 'NHG' THEN 0.75 * 0.1 * current_balance ELSE 0.35 * current_balance + 0.4 * GREATEST(0, current_balance - 0.8 * current_valuation_amount) END AS rwa,
                DATEDIFF('month', pool_cut_off_date::date, LAST_DAY(interest_revision_date)) AS months_to_revision_date, DATEDIFF('year', pool_cut_off_date::date, date_of_loan_maturity::date) AS remaining_term, (current_balance - undrawn_loanpart) * ((current_interest_rate / 100) / 12) AS next_month_interest_payment
            FROM temp_aggregation AS tmp_tbl
        ),
        temp_aggregation1a AS (SELECT *, GREATEST(0, payment_due - next_month_interest_payment) AS one_month_paydown FROM temp_aggregation1),
        temp_aggregation2 AS (SELECT tmp_tbl1.*, (current_balance - undrawn_loanpart + one_month_paydown) * ((current_interest_rate / 100) / 12) AS prev_month_interest_payment, CASE WHEN loan_term <= 24 THEN 0 ELSE ROUND(original_balance / loan_term, 2) END AS monthly_redemption FROM temp_aggregation1a AS tmp_tbl1),
        temp_aggregation3 AS (
            SELECT tmp_tbl3.*, total_income / 12 AS total_monthly_income, cust_level_prev_month_interest_payment, cust_level_monthly_redemption, CASE WHEN loan_term <= 24 THEN 0 ELSE cust_level_monthly_redemption + cust_level_prev_month_interest_payment END AS total_monthly_debt_service
            FROM temp_aggregation2 AS tmp_tbl3 LEFT JOIN (SELECT pool_cut_off_date, borrower_identifier, SUM(prev_month_interest_payment) AS cust_level_prev_month_interest_payment, SUM(monthly_redemption) AS cust_level_monthly_redemption FROM temp_aggregation2 GROUP BY 1, 2) tmp_tbl4 ON tmp_tbl3.pool_cut_off_date = tmp_tbl4.pool_cut_off_date AND tmp_tbl3.borrower_identifier = tmp_tbl4.borrower_identifier
        ),
        temp_aggregation3a AS (SELECT *, CASE WHEN loan_term <= 24 OR current_balance = 0 THEN 0 ELSE 100 * (total_monthly_debt_service / total_monthly_income) END AS fitch_dti FROM temp_aggregation3),
        temp_aggregation4 AS (
            SELECT *,
                CASE WHEN current_loan_to_value IS NULL THEN NULL WHEN current_loan_to_value < 65 THEN 1 WHEN current_loan_to_value < 85 THEN 2 WHEN current_loan_to_value < 95 THEN 3 ELSE 4 END AS fitch_ltv_class,
                CASE WHEN months_to_revision_date IS NULL THEN NULL WHEN months_to_revision_date / 12 < 8 THEN 1 WHEN months_to_revision_date / 12 < 12 THEN 2 WHEN months_to_revision_date / 12 < 17 THEN 3 WHEN months_to_revision_date / 12 < 25 THEN 4 ELSE 5 END AS fitch_term_class,
                CASE WHEN loan_term IS NULL THEN NULL WHEN loan_term <= 24 THEN 0 WHEN fitch_dti < 20 THEN 1 WHEN fitch_dti < 30 THEN 2 WHEN fitch_dti < 40 THEN 3 WHEN fitch_dti < 50 THEN 4 ELSE 5 END AS fitch_dti_class,
                CASE WHEN original_loan_to_value IS NULL THEN NULL WHEN original_loan_to_value < 40 THEN '<40' WHEN original_loan_to_value < 50 THEN '40-50' WHEN original_loan_to_value < 60 THEN '50-60' WHEN original_loan_to_value < 65 THEN '60-65' WHEN original_loan_to_value < 70 THEN '65-70' WHEN original_loan_to_value < 75 THEN '70-75' WHEN original_loan_to_value < 80 THEN '75-80' WHEN original_loan_to_value < 85 THEN '80-85' WHEN original_loan_to_value < 90 THEN '85-90' WHEN original_loan_to_value < 95 THEN '90-95' WHEN original_loan_to_value < 100 THEN '95-100' WHEN original_loan_to_value < 105 THEN '100-105' WHEN original_loan_to_value < 110 THEN '105-110' ELSE '>=110' END AS fitch_oltv_class,
                CASE WHEN current_loan_to_value IS NULL THEN NULL WHEN current_loan_to_value < 40 THEN '<40' WHEN current_loan_to_value < 50 THEN '40-50' WHEN current_loan_to_value < 60 THEN '50-60' WHEN current_loan_to_value < 65 THEN '60-65' WHEN current_loan_to_value < 70 THEN '65-70' WHEN current_loan_to_value < 75 THEN '70-75' WHEN current_loan_to_value < 80 THEN '75-80' WHEN current_loan_to_value < 85 THEN '80-85' WHEN current_loan_to_value < 90 THEN '85-90' WHEN current_loan_to_value < 95 THEN '90-95' WHEN current_loan_to_value < 100 THEN '95-100' WHEN current_loan_to_value < 105 THEN '100-105' WHEN current_loan_to_value < 110 THEN '105-110' ELSE '>=110' END AS fitch_cltv_class,
                CASE WHEN remaining_term IS NULL THEN NULL WHEN remaining_term < 5 THEN '<5' WHEN remaining_term < 10 THEN '5-10' WHEN remaining_term < 15 THEN '10-15' WHEN remaining_term < 20 THEN '15-20' WHEN remaining_term < 25 THEN '20-25' ELSE '>=25' END AS fitch_remaining_term_class,
                CASE WHEN original_loan_to_value IS NULL THEN NULL WHEN original_loan_to_value < 50 THEN '<50' WHEN original_loan_to_value < 60 THEN '50-60' WHEN original_loan_to_value < 70 THEN '60-70' WHEN original_loan_to_value < 80 THEN '70-80' WHEN original_loan_to_value < 90 THEN '80-90' ELSE '>=90' END AS fitch_oltv_term_class,
                CASE WHEN current_loan_to_value IS NULL THEN NULL WHEN current_loan_to_value < 50 THEN '<50' WHEN current_loan_to_value < 60 THEN '50-60' WHEN current_loan_to_value < 70 THEN '60-70' WHEN current_loan_to_value < 80 THEN '70-80' WHEN current_loan_to_value < 90 THEN '80-90' ELSE '>=90' END AS fitch_cltv_term_class
            FROM temp_aggregation3a
        ),
        temp_aggregation5 AS (SELECT temp_tbl6.*, base_ff.baseline_ff FROM temp_aggregation4 AS temp_tbl6 LEFT JOIN credit_risk_playground.stg_mrt_fitch_ff_baseline AS base_ff ON temp_tbl6.fitch_cltv_class = base_ff.ltv_class AND temp_tbl6.fitch_dti_class = base_ff.dti_class),
        temp_aggregation6 AS (SELECT temp_tbl7.*, io_adjustment.io_adjustment FROM temp_aggregation5 AS temp_tbl7 LEFT JOIN credit_risk_playground.stg_mrt_fitch_ff_io_adjustment AS io_adjustment ON temp_tbl7.fitch_oltv_term_class = io_adjustment.ltv_class AND temp_tbl7.fitch_remaining_term_class = io_adjustment.remaining_term_class),
        temp_aggregation7 AS (SELECT temp_tbl8.*, income_adj.income_adjustment FROM temp_aggregation6 AS temp_tbl8 LEFT JOIN credit_risk_playground.stg_mrt_fitch_income_adjustment AS income_adj ON temp_tbl8.borrowers_employment_status = income_adj.income_source),
        temp_aggregation8 AS (SELECT temp_tbl9.*, arrers_floor.ff_floor FROM temp_aggregation7 AS temp_tbl9 LEFT JOIN credit_risk_playground.stg_mrt_fitch_arrears_floor AS arrers_floor ON temp_tbl9.consecutive_months_in_arrears = arrers_floor.months_in_arrears),
        temp_aggregation9 AS (
            SELECT *,
                CASE WHEN loan_term <= 24 OR current_balance = 0 THEN 0 ELSE baseline_ff END AS final_baseline_ff,
                CASE WHEN fitch_dti_class = 0 OR payment_type <> 6 THEN 1 ELSE io_adjustment END AS final_io_adjustment,
                CASE WHEN account_status IN (2,3) AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 1 WHEN consecutive_months_in_arrears > 4 AND arrears_balance > 5 THEN 0.55 WHEN consecutive_months_in_arrears > 4 AND arrears_balance < 5 THEN 0 ELSE ff_floor END AS final_ff_floor
            FROM temp_aggregation8
        ),
        temp_aggregation10 AS (
            SELECT tmptbl.*,
                CASE WHEN borrower_identifier IN (5389716, 5638173, 5580361) THEN 100 ELSE 100 * (1 - POWER((1 - tmptbl.adjusted_fitch_base_ff / 100), 0.1428571)) END AS annualised_ff
            FROM (SELECT *, LEAST(100 * GREATEST(final_baseline_ff * final_io_adjustment * income_adjustment, final_ff_floor) + (CASE WHEN energy_label IN ('A', 'B') THEN 0 ELSE 0.34 END), 100) AS adjusted_fitch_base_ff FROM temp_aggregation9) AS tmptbl
        ),
        temp_aggregation11 AS (
            SELECT temp10.*, forcl_discount, forcl_variable_cost, cure_rate, pd_factor_down_scen, pd_factor_neu_scen, pd_factor_up_scen, indexed_vlaue_scalar_down_scen, indexed_vlaue_scalar_neu_scen, indexed_vlaue_scalar_up_scen
            FROM temp_aggregation10 AS temp10 LEFT JOIN mev_plus_parameter_inputs AS mevs_input ON LAST_DAY(DATE_ADD('month', -1, temp10.pool_cut_off_date))::date = mevs_input.ref_date
        ),
        pd_pit_scenarios AS (
            SELECT temp11.*,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) END AS pd_ttc,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) * pd_factor_down_scen END AS pd_pit_down_scen,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) * pd_factor_neu_scen END AS pd_pit_neu_scen,
                CASE WHEN annualised_ff = 100 THEN 100 ELSE annualised_ff * POWER((1 - cure_rate), -1) * pd_factor_up_scen END AS pd_pit_up_scen
            FROM temp_aggregation11 AS temp11
        ),
        pd_pit_scenario_weighted AS (
            SELECT *, pd_pit_down_scen * 0.0225 + pd_pit_neu_scen * 0.955 + pd_pit_up_scen * 0.0225 AS pd_pit FROM pd_pit_scenarios
        ),
        pd_pit_to_master_scale AS (
            SELECT temp_hold2.*,
                CASE WHEN pd_pit < 0.01 THEN 1 WHEN pd_pit <= 0.04 THEN 2 WHEN pd_pit <= 0.09 THEN 3 WHEN pd_pit <= 0.15 THEN 4 WHEN pd_pit <= 0.23 THEN 5 WHEN pd_pit <= 0.38 THEN 6 WHEN pd_pit <= 0.73 THEN 7 WHEN pd_pit <= 1.42 THEN 8 WHEN pd_pit <= 2.25 THEN 9 WHEN pd_pit <= 3.21 THEN 10 WHEN pd_pit <= 4.29 THEN 11 WHEN pd_pit <= 5.61 THEN 12 WHEN pd_pit <= 11.18 THEN 13 WHEN pd_pit <= 27.89 THEN 14 WHEN pd_pit <= 46.00 THEN 15 WHEN pd_pit <= 60.86 THEN 16 WHEN pd_pit > 60.86 AND pd_pit <= 99.99 THEN 17 WHEN account_status = 3 THEN 18 WHEN annualised_ff = 100 THEN 18 END AS pd_pit_class,
                (current_balance / customer_level_current_balance) * total_collateral_value * (CASE WHEN energy_label IN ('A', 'B', 'C') THEN 1 ELSE (1 - 0.037) END) AS collateral_value
            FROM (
                SELECT temp_hold1.*,
                    CASE WHEN pd_pit < 0.01 THEN 0.00 WHEN pd_pit <= 0.04 THEN 0.03 WHEN pd_pit <= 0.09 THEN 0.07 WHEN pd_pit <= 0.15 THEN 0.13 WHEN pd_pit <= 0.23 THEN 0.19 WHEN pd_pit <= 0.38 THEN 0.30 WHEN pd_pit <= 0.73 THEN 0.49 WHEN pd_pit <= 1.42 THEN 1.10 WHEN pd_pit <= 2.25 THEN 1.85 WHEN pd_pit <= 3.21 THEN 2.75 WHEN pd_pit <= 4.29 THEN 3.75 WHEN pd_pit <= 5.61 THEN 4.91 WHEN pd_pit <= 11.18 THEN 6.43 WHEN pd_pit <= 27.89 THEN 19.45 WHEN pd_pit <= 46.00 THEN 40 WHEN pd_pit <= 60.86 THEN 52.92 WHEN pd_pit > 60.86 AND pd_pit <= 99.99 THEN 70 WHEN account_status = 3 THEN 100 WHEN annualised_ff = 100 THEN 100 END AS pd_pit_mastered,
                    CASE WHEN pd_pit_neu_scen < 0.01 THEN 0.00 WHEN pd_pit_neu_scen <= 0.04 THEN 0.03 WHEN pd_pit_neu_scen <= 0.09 THEN 0.07 WHEN pd_pit_neu_scen <= 0.15 THEN 0.13 WHEN pd_pit_neu_scen <= 0.23 THEN 0.19 WHEN pd_pit_neu_scen <= 0.38 THEN 0.30 WHEN pd_pit_neu_scen <= 0.73 THEN 0.49 WHEN pd_pit_neu_scen <= 1.42 THEN 1.10 WHEN pd_pit_neu_scen <= 2.25 THEN 1.85 WHEN pd_pit_neu_scen <= 3.21 THEN 2.75 WHEN pd_pit_neu_scen <= 4.29 THEN 3.75 WHEN pd_pit_neu_scen <= 5.61 THEN 4.91 WHEN pd_pit_neu_scen <= 11.18 THEN 6.43 WHEN pd_pit_neu_scen <= 27.89 THEN 19.45 WHEN pd_pit_neu_scen <= 46.00 THEN 40 WHEN pd_pit_neu_scen <= 60.86 THEN 52.92 WHEN pd_pit_neu_scen > 60.86 AND pd_pit <= 99.99 THEN 70 WHEN account_status = 3 AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 100 WHEN annualised_ff = 100 THEN 100 END AS pd_pit_mastered_neu_scen,
                    CASE WHEN pd_pit_up_scen < 0.01 THEN 0.00 WHEN pd_pit_up_scen <= 0.04 THEN 0.03 WHEN pd_pit_up_scen <= 0.09 THEN 0.07 WHEN pd_pit_up_scen <= 0.15 THEN 0.13 WHEN pd_pit_up_scen <= 0.23 THEN 0.19 WHEN pd_pit_up_scen <= 0.38 THEN 0.30 WHEN pd_pit_up_scen <= 0.73 THEN 0.49 WHEN pd_pit_up_scen <= 1.42 THEN 1.10 WHEN pd_pit_up_scen <= 2.25 THEN 1.85 WHEN pd_pit_up_scen <= 3.21 THEN 2.75 WHEN pd_pit_up_scen <= 4.29 THEN 3.75 WHEN pd_pit_up_scen <= 5.61 THEN 4.91 WHEN pd_pit_up_scen <= 11.18 THEN 6.43 WHEN pd_pit_up_scen <= 27.89 THEN 19.45 WHEN pd_pit_up_scen <= 46.00 THEN 40 WHEN pd_pit_up_scen <= 60.86 THEN 52.92 WHEN pd_pit_up_scen > 60.86 AND pd_pit <= 99.99 THEN 70 WHEN account_status = 3 AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 100 WHEN annualised_ff = 100 THEN 100 END AS pd_pit_mastered_up_scen,
                    CASE WHEN pd_pit_down_scen < 0.01 THEN 0.00 WHEN pd_pit_down_scen <= 0.04 THEN 0.03 WHEN pd_pit_down_scen <= 0.09 THEN 0.07 WHEN pd_pit_down_scen <= 0.15 THEN 0.13 WHEN pd_pit_down_scen <= 0.23 THEN 0.19 WHEN pd_pit_down_scen <= 0.38 THEN 0.30 WHEN pd_pit_down_scen <= 0.73 THEN 0.49 WHEN pd_pit_down_scen <= 1.42 THEN 1.10 WHEN pd_pit_down_scen <= 2.25 THEN 1.85 WHEN pd_pit_down_scen <= 3.21 THEN 2.75 WHEN pd_pit_down_scen <= 4.29 THEN 3.75 WHEN pd_pit_down_scen <= 5.61 THEN 4.91 WHEN pd_pit_down_scen <= 11.18 THEN 6.43 WHEN pd_pit_down_scen <= 27.89 THEN 19.45 WHEN pd_pit_down_scen <= 46.00 THEN 40 WHEN pd_pit_down_scen <= 60.86 THEN 52.92 WHEN pd_pit_down_scen > 60.86 AND pd_pit <= 99.99 THEN 70 WHEN account_status = 3 AND consecutive_months_in_arrears >= 3 AND ROUND(arrears_balance_ratio, 1) >= 1 THEN 100 WHEN annualised_ff = 100 THEN 100 END AS pd_pit_mastered_down_scen
                FROM pd_pit_scenario_weighted AS temp_hold1
            ) AS temp_hold2
        ),
        collateral_adjustement AS (
            SELECT temp13.*, indexed_vlaue_scalar_down_scen * collateral_value AS adjusted_collateral_value_down_scen, indexed_vlaue_scalar_neu_scen * collateral_value AS adjusted_collateral_value_neu_scen, indexed_vlaue_scalar_up_scen * collateral_value AS adjusted_collateral_value_up_scen
            FROM pd_pit_to_master_scale AS temp13
        ),
        foreclosure_value_calculation AS (
            SELECT temp14.*, GREATEST(guaranteed_foreclosure_value, adjusted_collateral_value_down_scen * (1 - forcl_discount - forcl_variable_cost)) AS fitch_foreclosure_value_down_scen, GREATEST(guaranteed_foreclosure_value, adjusted_collateral_value_neu_scen * (1 - forcl_discount - forcl_variable_cost)) AS fitch_foreclosure_value_neu_scen, GREATEST(guaranteed_foreclosure_value, adjusted_collateral_value_up_scen * (1 - forcl_discount - forcl_variable_cost)) AS fitch_foreclosure_value_up_scen
            FROM collateral_adjustement AS temp14
        ),
        lgd_calculation AS (
            SELECT temp16.*, CASE WHEN loan_term <= 24 THEN 0 ELSE GREATEST(5, lgd_pit_down_scen) END AS lgd_down_scen, CASE WHEN loan_term <= 24 THEN 0 ELSE GREATEST(5, lgd_pit_neu_scen) END AS lgd_neu_scen, CASE WHEN loan_term <= 24 THEN 0 ELSE GREATEST(5, lgd_pit_up_scen) END AS lgd_up_scen	 
            FROM (
                SELECT temp15.*, CASE WHEN loan_term <= 24 OR current_balance = 0 THEN 0 ELSE 100 * (1 - cure_rate) * GREATEST(0, 1 - (fitch_foreclosure_value_down_scen / current_balance)) END AS lgd_pit_down_scen, CASE WHEN loan_term <= 24 OR current_balance = 0 THEN 0 ELSE 100 * (1 - cure_rate) * GREATEST(0, 1 - (fitch_foreclosure_value_neu_scen / current_balance)) END AS lgd_pit_neu_scen, CASE WHEN loan_term <= 24 OR current_balance = 0 THEN 0 ELSE 100 * (1 - cure_rate) * GREATEST(0, 1 - (fitch_foreclosure_value_up_scen / current_balance)) END AS lgd_pit_up_scen
                FROM foreclosure_value_calculation AS temp15
            ) temp16
        ),
        ecl_calculation AS (
            SELECT *, lgd_down_scen * 0.0225 + lgd_neu_scen * 0.955 + lgd_up_scen * 0.0225 AS lgd_weighted, pd_pit_mastered_down_scen / 100 * (lgd_down_scen / 100) * current_balance AS ecl_down_scen, pd_pit_mastered_neu_scen / 100 * (lgd_neu_scen / 100) * current_balance AS ecl_neu_scen, pd_pit_mastered_up_scen / 100 * (lgd_up_scen / 100) * current_balance AS ecl_up_scen
            FROM lgd_calculation
        ),
        final_tbl AS (
            SELECT temp17.*, ecl_down_scen * 0.0225 + ecl_neu_scen * 0.955 + ecl_up_scen * 0.0225 AS expected_loss_1year, (1 - POWER((1 - 0.003), LEAST(7, GREATEST(0, remaining_term)))) * original_balance AS performance_trigger, province_code, pronvince_name, GETDATE() AS etl_updated, 'alfred_teye'::text AS created_by
            FROM ecl_calculation AS temp17 LEFT JOIN credit_risk_playground.stg_mrt_fitch_location_codes AS geo_code ON temp17.geographic_region_list = geo_code.geographical_code 
        )
        SELECT pool_cut_off_date::DATE, borrower_identifier::VARCHAR, loan_identifier::VARCHAR, segment::VARCHAR, primary_income::FLOAT, secondary_income::FLOAT, total_income::FLOAT, borrowers_employment_status::INTEGER, loan_origination_date::DATE, date_of_loan_maturity::DATE, interest_revision_date::DATE, loan_term::INTEGER, purpose::VARCHAR, original_balance::FLOAT, current_balance::FLOAT, undrawn_loanpart::FLOAT, payment_type::SMALLINT, payment_due::FLOAT, debt_to_income::FLOAT, current_interest_rate::FLOAT, interest_rate_reset_interval::INTEGER, geographic_region_list::VARCHAR, original_loan_to_value::FLOAT, valuation_amount::FLOAT, current_loan_to_value::FLOAT, current_valuation_amount::FLOAT, property_value_at_time_of_latest_loan_advance::FLOAT, indexed_foreclosure_value::FLOAT, guaranteed_foreclosure_value::FLOAT, fitch_dti::FLOAT debt_service_coverage_ratio, account_status::SMALLINT, number_months_in_arrears::INTEGER, first_date_in_arrears::DATE, last_arrears_info_date::DATE, arrears_balance::FLOAT, consecutive_months_in_arrears::INT, customer_level_original_balance::FLOAT, customer_level_current_balance::FLOAT, total_collateral_value::FLOAT, current_bridge_exposure::FLOAT, arrears_balance_ratio::FLOAT, current_debt_to_income_ratio::FLOAT, current_debt_to_income_ratio_excl_bridge::FLOAT, rwa::FLOAT, months_to_revision_date::INT, remaining_term::INT, next_month_interest_payment::FLOAT, one_month_paydown::FLOAT, prev_month_interest_payment::FLOAT, monthly_redemption::FLOAT, total_monthly_income::FLOAT, cust_level_prev_month_interest_payment::FLOAT, cust_level_monthly_redemption::FLOAT, total_monthly_debt_service::FLOAT, fitch_dti::FLOAT, fitch_ltv_class::INTEGER, fitch_term_class::INTEGER, fitch_dti_class::INTEGER, fitch_oltv_class::VARCHAR, fitch_cltv_class::VARCHAR, fitch_remaining_term_class::VARCHAR, fitch_oltv_term_class::VARCHAR, fitch_cltv_term_class::VARCHAR, baseline_ff::FLOAT, io_adjustment::FLOAT, income_adjustment::FLOAT, ff_floor::FLOAT, final_baseline_ff::FLOAT, final_io_adjustment::FLOAT, final_ff_floor::FLOAT, adjusted_fitch_base_ff::FLOAT, annualised_ff::FLOAT, forcl_discount::FLOAT, forcl_variable_cost::FLOAT, cure_rate::FLOAT, pd_factor_down_scen::FLOAT, pd_factor_neu_scen::FLOAT, pd_factor_up_scen::FLOAT, indexed_vlaue_scalar_down_scen::FLOAT, indexed_vlaue_scalar_neu_scen::FLOAT, indexed_vlaue_scalar_up_scen::FLOAT, pd_ttc::FLOAT, pd_pit_down_scen::FLOAT, pd_pit_neu_scen::FLOAT, pd_pit_up_scen::FLOAT, pd_pit::FLOAT, pd_pit_mastered::FLOAT, pd_pit_mastered_neu_scen::FLOAT, pd_pit_mastered_up_scen::FLOAT, pd_pit_mastered_down_scen::FLOAT, pd_pit_class::INTEGER, collateral_value::FLOAT, adjusted_collateral_value_down_scen::FLOAT, adjusted_collateral_value_neu_scen::FLOAT, adjusted_collateral_value_up_scen::FLOAT, fitch_foreclosure_value_down_scen::FLOAT, fitch_foreclosure_value_neu_scen::FLOAT, fitch_foreclosure_value_up_scen::FLOAT, lgd_pit_down_scen::FLOAT, lgd_pit_neu_scen::FLOAT, lgd_pit_up_scen::FLOAT, lgd_down_scen::FLOAT, lgd_neu_scen::FLOAT, lgd_up_scen::FLOAT, lgd_weighted::FLOAT, ecl_down_scen::FLOAT, ecl_neu_scen::FLOAT, ecl_up_scen::FLOAT, expected_loss_1year::FLOAT, performance_trigger::FLOAT, province_code::INT, pronvince_name::VARCHAR, GETDATE() AS etl_updated, 'alfred_teye'::text AS created_by
        FROM final_tbl;
    """,

    "17_Neo_Portfolio_Stats": """
        INSERT INTO credit_risk_playground.stg_mrt_neo_portfolio_stats
        WITH reporting_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        credit_risk_calculations AS (
            SELECT risk_cal.*, 0.28 / 100 AS servicing_cost_pct
            FROM credit_risk_playground.stg_mrt_neo_credit_risk_calculations risk_cal
            INNER JOIN reporting_date rep_date
            ON risk_cal.pool_cut_off_date = rep_date.reporting_date
        ),
        exposure_overview AS (
            SELECT 
                pool_cut_off_date,
                COUNT(DISTINCT borrower_identifier) AS customers,
                ROUND(SUM(current_balance), 0) AS total_principal,
                ROUND(SUM(undrawn_loanpart), 0) AS undrawn_amounts,
                ROUND(SUM(CASE WHEN purpose = 'Brigding' THEN current_balance END)) AS bridge_loans,
                ROUND(MAX(customer_level_current_balance)) AS largest_exposure,
                ROUND(SUM(DATEDIFF('year', pool_cut_off_date, LAST_DAY(interest_revision_date)) * current_balance) / SUM(current_balance), 2) AS wa_reset_period_years,
                ROUND(SUM(rwa), 0) AS rwa,
                ROUND(100 * SUM(rwa) / SUM(current_balance), 2) rwa_pct,
                ROUND(SUM(current_balance * current_loan_to_value) / SUM(current_balance), 2) AS wa_cltv_pct,
                ROUND(SUM(CASE WHEN debt_service_coverage_ratio IS NULL THEN 0 ELSE debt_service_coverage_ratio END * current_balance) / SUM(current_balance), 2) AS wa_dscr,
                ROUND(SUM(current_interest_rate * current_balance) / SUM(current_balance) - 100 * servicing_cost_pct, 2) AS wa_coupon_net_servicing_cost_pct,
                ROUND(SUM(current_interest_rate * current_balance) / SUM(current_balance), 2) AS gross_coupon_pct,
                ROUND(SUM(expected_loss_1year), 0) AS ecl_one_year,
                ROUND(100 * ROUND(SUM(expected_loss_1year), 2) / SUM(current_balance), 2) AS ecl_one_year_pct,
                ROUND(SUM(lgd * current_balance) / SUM(current_balance), 2) AS wa_lgd_pct
            FROM credit_risk_calculations, credit_risk_playground.stg_mrt_portfolio_parameters 
            WHERE parameter_name = 'servicing_cost'
            GROUP BY pool_cut_off_date, servicing_cost_pct
            ORDER BY pool_cut_off_date
        ),
        sub_portfolio_overview AS (
            SELECT 
                pool_cut_off_date, segment,
                COUNT(DISTINCT borrower_identifier) AS customers,
                ROUND(SUM(current_balance)) AS total_principal,
                ROUND(SUM(undrawn_loanpart)) AS undrawn_amounts,
                ROUND(SUM(CASE WHEN purpose = 'Brigding' THEN current_balance END)) AS bridge_loans,
                ROUND(MAX(customer_level_current_balance)) AS largest_exposure,
                ROUND(SUM(DATEDIFF('year', pool_cut_off_date, LAST_DAY(interest_revision_date)) * current_balance) / SUM(current_balance), 2) AS wa_reset_period_years,
                ROUND(SUM(rwa), 0) AS rwa,
                ROUND(100 * SUM(rwa) / SUM(current_balance), 2) rwa_pct,
                ROUND(SUM(current_balance * current_loan_to_value) / SUM(current_balance), 2) AS wa_cltv_pct,
                ROUND(SUM(CASE WHEN debt_service_coverage_ratio IS NULL THEN 0 ELSE debt_service_coverage_ratio END * current_balance) / SUM(current_balance), 2) AS wa_dscr,
                ROUND(SUM(current_interest_rate * current_balance) / SUM(current_balance) - 100 * servicing_cost_pct, 2) AS wa_coupon_net_servicing_cost_pct,
                ROUND(SUM(current_interest_rate * current_balance) / SUM(current_balance), 2) AS gross_coupon_pct,
                ROUND(SUM(expected_loss_1year), 0) AS ecl_one_year,
                ROUND(100 * ROUND(SUM(expected_loss_1year), 2) / SUM(current_balance), 2) AS ecl_one_year_pct,
                ROUND(SUM(lgd * current_balance) / SUM(current_balance), 2) AS wa_lgd_pct
            FROM credit_risk_calculations, credit_risk_playground.stg_mrt_portfolio_parameters 
            WHERE parameter_name = 'servicing_cost'
            GROUP BY pool_cut_off_date, segment, servicing_cost_pct
            ORDER BY pool_cut_off_date
        ),
        pd_overview AS (
            SELECT pool_cut_off_date, ROUND(SUM(pd_pit_mastered * current_balance) / SUM(current_balance), 2) wa_pd_pct
            FROM credit_risk_calculations WHERE pd_pit_mastered < 100
            GROUP BY pool_cut_off_date ORDER BY pool_cut_off_date
        ),
        pd_sub_portfolio_overview AS (
            SELECT pool_cut_off_date, segment, ROUND(SUM(pd_pit_mastered * current_balance) / SUM(current_balance), 2) wa_pd_pct
            FROM credit_risk_calculations WHERE pd_pit_mastered < 100
            GROUP BY pool_cut_off_date, segment ORDER BY pool_cut_off_date
        ),
        aggregate_stats AS (
            (SELECT a.pool_cut_off_date, 'total' AS segment, a.customers, a.total_principal, a.undrawn_amounts, a.bridge_loans, a.largest_exposure, a.wa_reset_period_years, a.rwa, a.rwa_pct, a.wa_cltv_pct, a.wa_dscr, a.wa_coupon_net_servicing_cost_pct, gross_coupon_pct, a.ecl_one_year, a.ecl_one_year_pct, a.wa_lgd_pct, b.wa_pd_pct
             FROM exposure_overview a INNER JOIN pd_overview b ON a.pool_cut_off_date = b.pool_cut_off_date ORDER BY pool_cut_off_date)
            UNION 
            (SELECT c.*, d.wa_pd_pct FROM sub_portfolio_overview c INNER JOIN pd_sub_portfolio_overview d ON c.pool_cut_off_date = d.pool_cut_off_date AND c.segment = d.segment ORDER BY pool_cut_off_date)
            ORDER BY pool_cut_off_date, segment
        ),
        capitalisation AS (
            SELECT stat.*, 0.28 / 100 AS servicing_cost_pct, ROUND(rwa * value) AS required_capital, ROUND(rwa_pct * value, 2) AS required_capital_pct
            FROM aggregate_stats AS stat, credit_risk_playground.stg_mrt_portfolio_parameters 
            WHERE parameter_name = 'required_equity'
        )
        SELECT cap.*, ROUND(100 * (gross_coupon_pct - 100 * servicing_cost_pct) / required_capital_pct, 2) AS roe_pct, ROUND(100 * (total_principal * (gross_coupon_pct / 100 - servicing_cost_pct) - ecl_one_year) / required_capital, 2) AS roe_pct1, GETDATE() AS etl_updated, 'alfred_teye_reader' AS created_by
        FROM capitalisation AS cap
        ORDER BY pool_cut_off_date ASC, segment DESC;
    """,

    "18_Neo_IFRS9_ECL_Reporting_Scenarios": """
        INSERT INTO credit_risk_playground.stg_mrt_neo_ifrs9_ecl_reporting_with_scenarios
        WITH RECURSIVE horizon(next_period) AS (
            SELECT 0
            UNION ALL
            SELECT next_period + 1 FROM horizon WHERE next_period < 36
        ),
        report_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        base_calculations AS (
            SELECT 
                risk_cal.pool_cut_off_date AS reporting_date, risk_cal.borrower_identifier, risk_cal.loan_identifier, risk_cal.segment, risk_cal.purpose, risk_cal.current_balance, risk_cal.customer_level_current_balance, risk_cal.undrawn_loanpart, risk_cal.loan_origination_date, risk_cal.date_of_loan_maturity, risk_cal.loan_term, risk_cal.remaining_term, COALESCE(risk_cal.current_interest_rate, non_null_rate) AS current_interest_rate, risk_cal.payment_type, risk_cal.payment_due, risk_cal.arrears_balance, risk_cal.number_months_in_arrears, risk_cal.account_status, risk_cal.consecutive_months_in_arrears AS mpd, risk_cal.pd_pit_mastered AS one_year_pd, risk_cal.pd_pit_class AS rating_class, risk_cal.pd_pit_mastered_down_scen, risk_cal.pd_pit_mastered_neu_scen, risk_cal.pd_pit_mastered_up_scen, risk_cal.lgd_down_scen, risk_cal.lgd_neu_scen, risk_cal.lgd_up_scen, risk_cal.lgd_weighted, risk_cal.expected_loss_1year
            FROM credit_risk_playground.stg_mrt_neo_credit_risk_calculations_with_scenarios AS risk_cal
            LEFT JOIN (
                SELECT borrower_identifier, loan_identifier, current_interest_rate AS non_null_rate, ROW_NUMBER() OVER(PARTITION BY borrower_identifier, loan_identifier ORDER BY pool_cut_off_date) AS rankings
                FROM credit_risk_playground.stg_mrt_neo_credit_risk_calculations_with_scenarios WHERE current_interest_rate IS NOT NULL
            ) AS init_rates
            ON risk_cal.borrower_identifier = init_rates.borrower_identifier AND risk_cal.loan_identifier = init_rates.loan_identifier
            WHERE rankings = 1
        ),
        initial_recognition_values AS (
            SELECT a.*, b.initial_pd
            FROM (SELECT MIN(pool_cut_off_date) AS first_date_in_portfolio, loan_identifier FROM credit_risk_playground.stg_mrt_neo_credit_risk_calculations GROUP BY loan_identifier ORDER BY loan_identifier) a
            INNER JOIN (SELECT pool_cut_off_date AS reporting_date, borrower_identifier, loan_identifier, pd_pit_mastered AS initial_pd FROM credit_risk_playground.stg_mrt_neo_credit_risk_calculations) b 
            ON a.first_date_in_portfolio = b.reporting_date AND a.loan_identifier = b.loan_identifier
            ORDER BY loan_identifier
        ),
        merged_data AS (
            SELECT init_values.first_date_in_portfolio, base.*,
                CASE WHEN base.reporting_date::date > init_values.first_date_in_portfolio::date THEN init_values.initial_pd ELSE one_year_pd END initial_pd,
                ROW_NUMBER() OVER(PARTITION BY base.borrower_identifier, base.loan_identifier, base.reporting_date ORDER BY base.borrower_identifier, base.loan_identifier, base.reporting_date) AS duplicates
            FROM base_calculations AS base LEFT JOIN initial_recognition_values AS init_values USING(loan_identifier)
        ),
        main_data AS (
            SELECT md.reporting_date, borrower_identifier, loan_identifier, segment, purpose, current_balance, undrawn_loanpart, customer_level_current_balance, loan_origination_date, first_date_in_portfolio, date_of_loan_maturity, loan_term, payment_type, payment_due, remaining_term, current_interest_rate, arrears_balance, number_months_in_arrears, account_status, mpd, initial_pd, one_year_pd, rating_class, pd_pit_mastered_down_scen, pd_pit_mastered_neu_scen, pd_pit_mastered_up_scen, lgd_down_scen, lgd_neu_scen, lgd_up_scen, lgd_weighted, expected_loss_1year,
                CASE WHEN initial_pd = 0 THEN 0 ELSE 100 * (1 - POWER(1 - initial_pd / 100, remaining_term)) END AS initial_ltpd,
                CASE WHEN one_year_pd = 0 THEN 0 ELSE 100 * (1 - POWER(1 - one_year_pd / 100, remaining_term)) END AS current_ltpd
            FROM merged_data AS md INNER JOIN report_date AS r_date USING(reporting_date) WHERE duplicates = 1
        ),
        ifrs_staging1 AS ( 
            SELECT *,
                CASE WHEN one_year_pd = 100 THEN 3 WHEN mpd >= 2 AND one_year_pd < 100 AND arrears_balance > 0 THEN 2 WHEN current_ltpd > 3 * initial_ltpd THEN 2 WHEN rating_class IN ('15','16','17') THEN 2 ELSE 1 END AS stage,
                current_balance AS ead
            FROM main_data
        ),
        ifrs_staging AS (SELECT *, MAX(stage) OVER(PARTITION BY reporting_date, borrower_identifier) AS ifrs_stage FROM ifrs_staging1),
        ecl_calculation AS (
            SELECT *,
                CASE WHEN ifrs_stage = 1 THEN (one_year_pd / 100) * (lgd_down_scen / 100) * ead WHEN ifrs_stage = 3 THEN (lgd_down_scen / 100) * ead END AS ecl_down_scen,
                CASE WHEN ifrs_stage = 1 THEN (one_year_pd / 100) * (lgd_neu_scen / 100) * ead WHEN ifrs_stage = 3 THEN (lgd_neu_scen / 100) * ead END AS ecl_neu_scen,
                CASE WHEN ifrs_stage = 1 THEN (one_year_pd / 100) * (lgd_up_scen / 100) * ead WHEN ifrs_stage = 3 THEN (lgd_up_scen / 100) * ead END AS ecl_up_scen
            FROM ifrs_staging 
        ),
        cashflow_projection1 AS (
            SELECT reporting_date, borrower_identifier, loan_identifier, remaining_term, payment_type, payment_due, current_interest_rate, ifrs_stage, one_year_pd, lgd_down_scen, lgd_neu_scen, lgd_up_scen, ead, next_period AS future_period
            FROM ecl_calculation CROSS JOIN horizon WHERE ifrs_stage = 2
        ),
        cashflow_projection2 AS (
            SELECT *,
                100 * (POWER(1 - one_year_pd / 100, future_period) - POWER(1 - one_year_pd / 100, future_period + 1)) AS pd_future_period,
                CASE 
                    WHEN payment_type = 1 THEN (CASE WHEN future_period = 0 THEN ead ELSE ROUND(ead * (POWER(1 + current_interest_rate / 1200, remaining_term) - POWER(1 + current_interest_rate / 1200, LEAST(future_period, remaining_term))) / (POWER(1 + current_interest_rate / 1200, remaining_term) - 1), 2) END)
                    WHEN payment_type = 2 THEN ROUND(GREATEST(ead - (future_period * ead / remaining_term), 0), 2)
                    WHEN payment_type = 6 THEN (CASE WHEN future_period <= remaining_term THEN ead ELSE 0 END)
                END ead_future_period,
                POWER(1 + current_interest_rate / 100, -future_period) AS discount_factor
            FROM cashflow_projection1
            ORDER BY borrower_identifier, loan_identifier, future_period
        ),
        temp_lifetime_ecl AS (
            SELECT *,
                (pd_future_period / 100) * (lgd_neu_scen / 100) * ead_future_period AS ecl_future_period_neu_scen,
                (pd_future_period / 100) * (lgd_down_scen / 100) * ead_future_period AS ecl_future_period_down_scen,
                (pd_future_period / 100) * (lgd_up_scen / 100) * ead_future_period AS ecl_future_period_up_scen,
                (pd_future_period / 100) * (lgd_neu_scen / 100) * ead_future_period * discount_factor AS ecl_future_period_discounted_neu_scen,
                (pd_future_period / 100) * (lgd_down_scen / 100) * ead_future_period * discount_factor AS ecl_future_period_discounted_down_scen,
                (pd_future_period / 100) * (lgd_up_scen / 100) * ead_future_period * discount_factor AS ecl_future_period_discounted_up_scen
            FROM cashflow_projection2
        ),
        lifetime_ecl AS (
            SELECT reporting_date, borrower_identifier, loan_identifier, ROUND(SUM(ecl_future_period_discounted_neu_scen * 0.955 + ecl_future_period_discounted_down_scen * 0.0225 + ecl_future_period_discounted_up_scen * 0.0225), 2) AS lifetime_ecl
            FROM temp_lifetime_ecl GROUP BY reporting_date, borrower_identifier, loan_identifier, ifrs_stage
        ),
        final_ecl_calculation AS (
            SELECT a.*, ROUND(COALESCE(a.ecl_down_scen * 0.0225 + a.ecl_neu_scen * 0.955 + a.ecl_up_scen * 0.0225, b.lifetime_ecl), 2) AS expected_credit_loss
            FROM ecl_calculation a LEFT JOIN lifetime_ecl b USING(reporting_date, borrower_identifier, loan_identifier)
        )
        SELECT *, GETDATE() AS etl_updated, 'alfred_teye'::text AS created_by
        FROM final_ecl_calculation
        ORDER BY borrower_identifier, loan_identifier, ifrs_stage;
    """
}

print(f" Registered {len(QUERIES)} Neo queries in execution order:")
for step_num, name in enumerate(QUERIES.keys(), 1):
    print(f"  Step {step_num}: {name}")

 Registered 7 Neo queries in execution order:
  Step 1: 12_Neo_ESME_Raw
  Step 2: 13_Neo_ESMA_To_ECB_Raw
  Step 3: 14_Neo_Arrears_List
  Step 4: 15_Neo_Credit_Risk_Calculations
  Step 5: 16_Neo_Credit_Risk_Calculations_Scenarios
  Step 6: 17_Neo_Portfolio_Stats
  Step 7: 18_Neo_IFRS9_ECL_Reporting_Scenarios


In [8]:
# Cell 3: Embedded SQL Pipeline Registry (Folders 4 & 5: Finance & Regulatory Reporting)

QUERIES = {
    "19_Finance_Data_Prep": """
        INSERT INTO credit_risk_playground.stg_mrt_ifrs9_ecl_reporting_with_scenarios
        WITH report_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        rabo_mortgages AS (
            SELECT 
                temp_b.reporting_date::date AS ReportDate,
                temp_a.reporting_date::date AS CalcDate,
                loan_identifier::varchar AS Instrument_ID, 
                borrower_identifier::varchar AS User_ID,
                'EU'::varchar AS LegalEntity,
                'NLD'::varchar AS Country,
                ead AS OnBalance_EAD,
                undrawn_loanpart::decimal(10,2) AS OffBalance_EAD,
                date_of_loan_maturity AS MaturityDate,
                initial_pd / 100::float AS InitialPD,
                rating_class AS CurrentRating,
                one_year_pd / 100::float AS PD,
                CASE
                    WHEN OnBalance_EAD < 0 THEN 0.05
                    ELSE lgd_weighted / 100::float 
                END AS LGD,
                NULL AS limit,
                ifrs_stage AS Stage,
                expected_credit_loss::float AS ECL_On,
                (PD * LGD * OffBalance_EAD)::decimal(10,2) AS ECL_Off,
                ROUND(mpd / 0.0329)::float AS dpd,
                NULL AS CCF,
                'yes' AS IS_IFRS, 
                'mortgage' AS Instrument_Type,
                'credit' AS Instrument
            FROM credit_risk_playground.stg_mrt_rabobank_ifrs9_ecl_reporting_with_scenarios AS temp_a
            INNER JOIN report_date AS temp_b
            ON temp_a.reporting_date = LAST_DAY(DATE_ADD('month', -1, temp_b.reporting_date))::date
        ),
        neo_mortgages AS (
            SELECT 
                temp_b.reporting_date::date AS ReportDate,
                temp_a.reporting_date::date AS CalcDate,
                loan_identifier::varchar AS Instrument_ID, 
                borrower_identifier::varchar AS User_ID,
                'Neo hypotheken'::varchar AS LegalEntity,
                'NLD'::varchar AS Country,
                ead AS OnBalance_EAD,
                undrawn_loanpart::decimal(10,2) AS OffBalance_EAD,
                date_of_loan_maturity AS MaturityDate,
                initial_pd / 100::float AS InitialPD,
                rating_class AS CurrentRating,
                one_year_pd / 100::float AS PD,
                CASE
                    WHEN OnBalance_EAD < 0 THEN 0.05
                    ELSE lgd_weighted / 100::float 
                END AS LGD,
                NULL AS limit,
                ifrs_stage AS Stage,
                expected_credit_loss::float AS ECL_On,
                (PD * LGD * OffBalance_EAD)::decimal(10,2) AS ECL_Off,
                ROUND(mpd / 0.0329)::float AS dpd,
                NULL AS CCF,
                'yes' AS IS_IFRS, 
                'mortgage' AS Instrument_Type,
                'credit' AS Instrument
            FROM credit_risk_playground.stg_mrt_neo_ifrs9_ecl_reporting_with_scenarios AS temp_a
            INNER JOIN report_date AS temp_b
            ON temp_a.reporting_date = LAST_DAY(DATE_ADD('month', -1, temp_b.reporting_date))::date
        ),
        temp_tbl AS (
            SELECT * FROM rabo_mortgages 
            UNION ALL
            SELECT * FROM neo_mortgages
        )
        SELECT 
            ReportDate,
            CalcDate,
            Instrument_ID, 
            User_ID,
            LegalEntity,
            Country,
            OnBalance_EAD,
            MaturityDate,
            InitialPD,
            CurrentRating,
            PD,
            LGD,
            "limit",
            Stage,
            ECL_On,
            dpd,
            CCF,
            IS_IFRS, 
            Instrument_Type,
            Instrument,
            GETDATE() AS etl_update,
            'alfred_teye_reader' AS created_by,
            OffBalance_EAD,
            ECL_Off
        FROM temp_tbl;
    """,

    "20_Reg_Reporting_Prep": """
        INSERT INTO credit_risk_playground.stg_mrt_ifrs9_ecl_regReporting_V3 
        WITH reporting_date AS (
            SELECT '{{ end_date }}'::date AS reportdate
        ),
        temp_tbl AS (
            SELECT 
                a.reportdate,
                calcdate,
                instrument_id,
                user_id,
                legalentity,
                country,
                onbalance_ead,
                offbalance_ead,
                maturitydate,
                initialpd,
                currentrating,
                pd,
                lgd,
                "limit",
                stage,
                ecl_on,
                ecl_off,
                dpd,
                ccf,
                is_ifrs,
                instrument_type,
                instrument,
                ROW_NUMBER() OVER(PARTITION BY instrument_id, user_id) AS dups
            FROM credit_risk_playground.stg_mrt_ifrs9_ecl_reporting_with_scenarios AS a 
            INNER JOIN reporting_date AS b
            ON a.reportdate = b.reportdate
        ),
        temp_tbl1 AS (
            SELECT * FROM temp_tbl WHERE dups = 1
        ),
        add_label AS (
            SELECT 
                a.*,
                CASE WHEN b.pool_identifier = '310' AND a.legalentity = 'Neo hypotheken' THEN 'Hypotheken B.V.' ELSE 'N26 Bank' END AS label
            FROM temp_tbl1 AS a
            LEFT JOIN (
                SELECT DISTINCT loan_identifier, pool_identifier
                FROM credit_risk_playground.stg_mrt_neo_esma_to_ecb_raw AS p
                INNER JOIN reporting_date AS q
                ON p.pool_cut_off_date = LAST_DAY(DATE_ADD('month', -1, q.reportdate))::date 
            ) AS b
            ON a.instrument_id = b.loan_identifier
        ),
        arrears_data AS (
            SELECT *, pool_cut_off_date AS reportdate FROM credit_risk_playground.stg_mrt_rabobank_arrears_list
            UNION ALL 
            SELECT *, pool_cut_off_date AS reportdate FROM credit_risk_playground.stg_mrt_neo_arrears_list
        ),
        default_date AS (
            SELECT 
                borrower_identifier,
                MIN(reportdate) AS first_time_default,
                MAX(reportdate) AS last_time_default
            FROM arrears_data
            WHERE months_in_arrears = 'default'
            GROUP BY borrower_identifier
        ),
        extended_arrears_data AS (
            SELECT * FROM (
                SELECT arr.*, df_dt.first_time_default, df_dt.last_time_default
                FROM arrears_data AS arr
                LEFT JOIN default_date AS df_dt USING(borrower_identifier)
            ) AS a 
            INNER JOIN reporting_date AS b
            ON a.reportdate = LAST_DAY(DATE_ADD('month', -1, b.reportdate::date)::date)
        ),
        ecl_data AS (
            SELECT 
                main.*,
                def.arrears_balance_customer_level,
                def.arrears_balance_ratio,
                def.months_in_arrears,
                def.first_date_in_arrears,
                def.last_arrears_info_date,
                def.first_time_default,
                def.last_time_default
            FROM add_label AS main
            LEFT JOIN extended_arrears_data AS def
            ON main.User_ID = def.borrower_identifier AND main.Instrument_ID = def.loan_identifier
        ),
        reg_reporting_appendix AS (
            SELECT 
                reportdate,
                calcdate,
                user_id AS entityno,
                instrument_id,
                instrument_type,
                instrument,
                user_id AS itemno,
                is_ifrs,
                CASE WHEN label = 'N26 Bank' THEN 'EU' ELSE 'NLD' END AS country,
                label AS legalentity,
                instrument_type AS product_flag,
                onbalance_ead,
                offbalance_ead,
                onbalance_ead + offbalance_ead AS ead,
                0 AS isoffbalance,  
                CASE WHEN stage = 3 AND pd < 1 THEN 18 ELSE currentrating END AS currentrating,
                CASE WHEN stage = 3 AND pd < 1 THEN 1 ELSE pd END AS pd,
                CASE 
                    WHEN 100 * initialpd < 0.01 THEN 1
                    WHEN 100 * initialpd <= 0.04 THEN 2
                    WHEN 100 * initialpd <= 0.09 THEN 3
                    WHEN 100 * initialpd <= 0.15 THEN 4
                    WHEN 100 * initialpd <= 0.23 THEN 5
                    WHEN 100 * initialpd <= 0.38 THEN 6
                    WHEN 100 * initialpd <= 0.73 THEN 7
                    WHEN 100 * initialpd <= 1.42 THEN 8
                    WHEN 100 * initialpd <= 2.25 THEN 9
                    WHEN 100 * initialpd <= 3.21 THEN 10
                    WHEN 100 * initialpd <= 4.29 THEN 11
                    WHEN 100 * initialpd <= 5.61 THEN 12
                    WHEN 100 * initialpd <= 11.18 THEN 13
                    WHEN 100 * initialpd <= 27.89 THEN 14
                    WHEN 100 * initialpd <= 46.00 THEN 15
                    WHEN 100 * initialpd <= 60.86 THEN 16
                    WHEN 100 * initialpd > 60.86 AND initialpd <= 99.99 THEN 17
                    ELSE 18
                END AS initialrating,
                initialpd,
                dpd,
                ccf,
                lgd,
                maturitydate,
                stage,
                ecl_on + ecl_off AS ecl,
                ecl_on,
                ecl_off,
                CASE WHEN stage = 3 THEN 'J' ELSE 'N' END AS AUSFL,
                CASE WHEN stage = 3 THEN COALESCE(first_time_default, LAST_DAY(DATEADD('day', (dpd*-1)::int, calcdate))) ELSE NULL END AS ADXAUD,
                CASE WHEN stage = 3 THEN 'C' WHEN stage = 2 THEN 'B' ELSE 'A' END AS RISGR,
                CASE WHEN stage = 3 THEN first_time_default ELSE NULL END AS DXNPE,
                NULL AS DXFBE,
                NULL AS FBSFI,
                CASE WHEN stage = 3 THEN '1801' ELSE '1808' END AS IMSFI,
                CASE WHEN stage = 3 THEN '3292' WHEN months_in_arrears = 'probation' THEN '3294' ELSE '3293' END AS PFSFI,
                CASE 
                    WHEN dpd IS NULL OR dpd <= 30 THEN '3295'
                    WHEN dpd <= 90 THEN '5032'
                    WHEN dpd <= 180 THEN '2792'
                    WHEN dpd <= 365 THEN '2785'
                    WHEN dpd <= 730 THEN '3280'
                    ELSE '3282'
                END AS TPDFI,
                NULL AS scra_cluster,
                NULL AS due_dilligence
            FROM ecl_data
        )
        SELECT 
            'mortgage' AS portfolio,
            reportdate::DATE,
            calcdate::DATE AS calc_date,
            NULL AS entityno,
            instrument_id::VARCHAR,
            instrument_type,
            instrument,
            itemno::VARCHAR,
            is_ifrs,
            country,
            legalentity,
            product_flag,
            COALESCE(onbalance_ead::float, 0) AS on_balance_ead,
            COALESCE(offbalance_ead::float, 0) AS off_balance_eod,
            COALESCE(onbalance_ead::float, 0) + COALESCE(offbalance_ead::float, 0) AS ead,
            0 AS isoffbalance,
            currentrating::VARCHAR,
            pd,
            initialrating::VARCHAR,
            initialpd AS initial_pd,
            dpd,
            TRY_CAST(ccf AS float) AS ccf,
            lgd,
            maturitydate,
            stage,
            ecl_on::float + COALESCE(ecl_off::float, 0) AS ecl,
            ecl_off::float AS ecl_off,
            ecl_on::float AS ecl_on,
            ausfl,
            adxaud AS dxaud,
            risgr,
            CAST(dxnpe AS date) AS dxnpe,
            CAST(dxfbe AS date) AS dxfbe,
            fbsfi::varchar AS fbsfi,
            imsfi::varchar AS imsfi,
            pfsfi::varchar AS pfsfi,
            tpdfi::varchar AS tpdfi,
            NULL AS scra_cluster,
            NULL AS due_dilligence
        FROM reg_reporting_appendix;
    """,

    "21_Risk_and_KRI_Reporting": """
        WITH temp_tbl AS (
            SELECT pool_cut_off_date AS calc_date, loan_identifier AS instrument_id, segment, purpose, current_loan_to_value, 'neo' AS business_line FROM credit_risk_playground.stg_mrt_neo_credit_risk_calculations
            UNION ALL 
            SELECT pool_cut_off_date AS calc_date, loan_identifier AS instrument_id, segment, purpose, current_loan_to_value, 'rabo' AS business_line FROM credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations
        ),
        temp_tbl2 AS (
            SELECT 
                *,
                CASE 
                    WHEN segment = 'NHG' THEN 0
                    WHEN current_loan_to_value < 60 THEN 1
                    WHEN current_loan_to_value < 70 THEN 2
                    WHEN current_loan_to_value < 80 THEN 3
                    WHEN current_loan_to_value < 90 THEN 4
                    WHEN current_loan_to_value < 100 THEN 5
                    ELSE 6
                END risk_class,
                CASE 
                    WHEN segment = 'NHG' THEN 'NHG'
                    WHEN current_loan_to_value < 60 THEN 'ltv<60'
                    WHEN current_loan_to_value < 70 THEN 'ltv<70'
                    WHEN current_loan_to_value < 80 THEN 'ltv<80'
                    WHEN current_loan_to_value < 90 THEN 'ltv<90'
                    WHEN current_loan_to_value < 100 THEN 'ltv<100'
                    ELSE 'ltv>=100'
                END risk_class_description
            FROM credit_risk_playground.stg_mrt_ifrs9_ecl_regReporting_V3 AS v3
            LEFT JOIN temp_tbl AS tbl USING(instrument_id, calc_date)
        ) 
        SELECT 
            reportdate,
            SUM(ead) AS exposure,
            SUM(ecl) AS ecl_vol,
            SUM(CASE WHEN stage = 3 THEN ead ELSE 0 END) AS npl_exposure,
            SUM(CASE WHEN segment = 'NHG' THEN ead ELSE 0 END) AS nhg_exposure,
            SUM(CASE WHEN current_loan_to_value > 80 AND segment <> 'NHG' THEN ead ELSE 0 END) AS ltv_above_80_expsoure,
            100 * ecl_vol / exposure AS ecl_pct,
            100 * npl_exposure / exposure AS npl_pct,
            100 * nhg_exposure / exposure AS nhg_pct,
            100 * ltv_above_80_expsoure / exposure AS ltv_above_80_pct
        FROM temp_tbl2
        WHERE reportdate = '{{ end_date }}'::date
        GROUP BY reportdate;
    """,

    "22_Arrears_and_Default_Stats": """
        WITH report_date AS (
            SELECT '{{ end_date }}'::date AS reporting_date
        ),
        arrears AS (
            SELECT arrList.* 
            FROM credit_risk_playground.stg_mrt_rabobank_arrears_list AS arrList
            INNER JOIN report_date AS rdate ON arrList.pool_cut_off_date::date = rdate.reporting_date 
        ),
        temp_tbl AS (
            (SELECT pool_cut_off_date, months_in_arrears, 'total' AS segment, COUNT(DISTINCT borrower_identifier) AS customers, ROUND(SUM(current_balance)) AS principal_balance, ROUND(SUM(arrears_balance_loan_level)) AS arrears_balance
             FROM arrears GROUP BY 1, 2)
            UNION ALL
            (SELECT pool_cut_off_date, months_in_arrears, segment, COUNT(DISTINCT borrower_identifier) AS customers, ROUND(SUM(current_balance)) AS principal_balance, ROUND(SUM(arrears_balance_loan_level)) AS arrears_balance
             FROM arrears GROUP BY 1, 2, 3)
        ),
        temp_tbl1 AS (
            SELECT a.*, b.pd_pit_mastered, b.lgd
            FROM arrears AS a
            LEFT JOIN (
                SELECT borrower_identifier, loan_identifier, pd_pit_mastered, lgd
                FROM credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations 
                WHERE pool_cut_off_date = (SELECT MAX(pool_cut_off_date) FROM credit_risk_playground.stg_mrt_rabobank_credit_risk_calculations)
            ) AS b ON a.borrower_identifier = b.borrower_identifier AND a.loan_identifier = b.loan_identifier
        ),
        temp_tbl2 AS (
            (SELECT months_in_arrears, segment, ROUND(SUM(current_balance * pd_pit_mastered) / SUM(current_balance), 2) AS avg_pd, ROUND(SUM(current_balance * lgd) / SUM(current_balance), 2) AS avg_lgd FROM temp_tbl1 GROUP BY segment, months_in_arrears) 
            UNION ALL
            (SELECT months_in_arrears, 'total' AS segment, ROUND(SUM(current_balance * pd_pit_mastered) / SUM(current_balance), 2) AS avg_pd, ROUND(SUM(current_balance * lgd) / SUM(current_balance), 2) AS avg_lgd FROM temp_tbl1 GROUP BY months_in_arrears)
        )
        SELECT tbl.*, tbl2.avg_pd, tbl2.avg_lgd
        FROM temp_tbl tbl
        LEFT JOIN temp_tbl2 AS tbl2 USING(months_in_arrears, segment)
        ORDER BY segment, months_in_arrears;
    """
}

print(f" Total pipeline queries registered: {len(QUERIES)}")

 Total pipeline queries registered: 4


In [ ]:
# Cell 4: Fully Automated Database Pipeline Execution Engine

import os
import time
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import psycopg2

# --- 1. DYNAMIC DATE LOGIC ---
# Automatically computes the last day of the previous month (YYYY-MM-DD)
today = datetime.today()
first_of_this_month = today.replace(day=1)
last_day_last_month = first_of_this_month - timedelta(days=1)
TARGET_END_DATE = last_day_last_month.strftime("%Y-%m-%d")

# --- 2. CREDENTIALS FROM ENVIRONMENT SECRETS ---
# Reads environment variables for GitHub Actions / production, falls back to defaults for local notebook testing
DB_CONFIG = {
    "host": os.getenv("REDSHIFT_HOST", "my-cluster.xxxxxx.us-east-1.redshift.amazonaws.com"),
    "port": int(os.getenv("REDSHIFT_PORT", 5439)),
    "dbname": os.getenv("REDSHIFT_DB", "dev"),
    "user": os.getenv("REDSHIFT_USER", "your_username"),
    "password": os.getenv("REDSHIFT_PASSWORD", "your_password")
}


# --- 3. EXECUTION ENGINE ---
def execute_pipeline(queries_dict, db_config, end_date):
    """
    Renders {{ end_date }} placeholders in registered SQL queries
    and executes them sequentially against Redshift.
    """
    print(f"🚀 Starting Monthly Credit Risk ETL Pipeline")
    print(f"📅 Target Cutoff Date: {end_date}")
    print("=" * 70)

    conn = None
    try:
        conn = psycopg2.connect(**db_config)
        conn.autocommit = True  # Ensures transactions commit after each step
        cursor = conn.cursor()

        pipeline_start = time.time()
        total_steps = len(queries_dict)

        for step_num, (step_name, raw_sql) in enumerate(queries_dict.items(), 1):
            # Dynamic date placeholder substitution
            rendered_sql = raw_sql.replace("{{ end_date }}", end_date)

            print(f"[{step_num}/{total_steps}] Executing: {step_name}...")
            step_start = time.time()

            cursor.execute(rendered_sql)

            duration = time.time() - step_start
            print(f"    ✔ Completed in {duration:.2f}s")

        total_duration = time.time() - pipeline_start
        print("=" * 70)
        print(f"✨ Pipeline completed successfully in {total_duration:.2f} seconds!")

    except Exception as err:
        print("\n❌ Pipeline Execution Failed!")
        print(f"Error details: {err}")
        raise err  # Triggers GitHub Action failure notification if step fails

    finally:
        if conn:
            cursor.close()
            conn.close()
            print("Database connection closed.")


# --- 4. RUN PIPELINE ---
execute_pipeline(QUERIES, DB_CONFIG, TARGET_END_DATE)